# immgenT-CD8 dataset — analysis notebook


This notebook documents data processing, annotation, and figure generation for the immgenT-CD8 T-cell CITE-seq atlas.

**CITE-seq** (cellular indexing of transcriptomes and epitopes by sequencing) jointly profiles transcriptome-wide gene expression and surface protein (ADT) abundance in single cells ([Stoeckius et al., 2017](https://www.nature.com/articles/nmeth.4380)).

**Dataset:** immgenT-CD8 T cells (multi-organ, multi-condition CITE-seq).

**Notebook structure (paper figure numbering):**
- **Figure 1** — immgenT resolves CD8+ T cell heterogeneity into 21 discrete clusters and a reference
embedding.
- **Figure 2** — Shared CD8+ T cell states across immune conditions.
- **Figure 3** — Effector CD8+ T cells comprise three distinct recurrent states.
- **Figure 4** — Circulating memory CD8+ T cells map to three main clusters that are present broadly beyond
acute infections.
- **Figure 5** — Tissue-resident memory CD8+ T cells converge on a single dominant transcriptional state while
maintaining tissue-specific differences.
- **Figure 6** — Exhaustion and residency-associated states form a continuum.
- **Extended Data Figure 1** — Comprehensive integration of the CD8αβ immgenT framework.
- **Extended Data Figure 2** — Acute effector states (clusters CD8.I, .J, and .K).
- **Extended Data Figure 3** — Phenotypic and transcriptional signatures of circulating memory states
(clusters CD8.E, .F, .G, and .H).
- **Extended Data Figure 4** — Surface-marker and transcriptional heterogeneity of tissue-resident memory
CD8+ T cells in cluster CD8.Q.
- **Extended Data Figure 5** — Detailed mapping of the exhaustion/residency continuum and
progenitor-exhausted cells.
- **Extended Data Figure 7** — immgenT-CD8 flow cytometry panel and validation.

For figures not covered by this notebook, including T-RBI analyses, refer to immgenT-Cosmology (Magill et al., bioRxiv 2026).

Paths and intermediate filenames reflect the analysis environment used to generate the paper figures; adjust them as needed when re-running.

# How to use this object

This MuData contains the curated ImmGenT CD8 CITE-seq dataset with sparse raw counts, full metadata, and the main embedding (MDE_INCREMENTAL).

After loading the object, a typical workflow is:

- Start from the raw counts (layers["counts"] or .X)
- Normalize, log-transform and scale the modalities (sc.pp.normalize_total → sc.pp.log1p → sc.pp.scale)
- Optionally apply CLR normalization to the ADT modality (muon.prot.pp.clr)
- Use the pre-computed embedding available in obsm["MDE_INCREMENTAL"] for visualization and downstream analysis

All key metadata columns are already present in both modalities.

# Libraries and environment


Import required libraries and configure plotting defaults.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import gseapy as gp
import matplotlib.pyplot as plt
import scanpy as sc
import muon as mu
from muon import prot as pt
import random

random.seed(0)
np.random.seed(0)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
sc.set_figure_params(frameon=True, figsize=(4, 4), dpi_save=300, transparent=True, format='eps',)
%config InlineBackend.print_figure_kwargs={'facecolor' : "w"}
%config InlineBackend.figure_format='retina'


Create color dictionary.


In [ ]:

custom_colors = {
    'CD8.A': '#00008B',      # blue2 (darkblue)
    'CD8.B': '#FF7F00',      # darkorange2
    'CD8.C': '#EE0000',      # red2
    'CD8.D': '#556B2F',      # darkolivegreen
    'CD8.E': '#6B8E23',      # darkolivegreen2 (olive drab)
    'CD8.F': '#9400D3',      # darkorchid2
    'CD8.G': '#CD3333',      # brown2
    'CD8.H': '#7AC5CD',      # cadetblue2
    'CD8.I': '#9BCD9B',      # darkseagreen2
    'CD8.J': '#D02090',      # violetred2
    'CD8.K': '#9932CC',      # darkorchid
    'CD8.P': '#000000',      # black
    'CD8.Q': '#76EE00',      # chartreuse2
    'CD8.R': '#FF69B4',      # pink2 (hotpink)
    'CD8.S': '#CD950C',      # darkgoldenrod2
    'CD8.T': '#FF8C00',      # darkorange
    'CD8.wM': '#808080',     # grey
    'CD8.wU': '#00EEEE',     # cyan2
    'CD8.wV': '#0000FF',     # blue
    'CD8.wX': '#EEC591',     # burlywood2
    'CD8.wY': '#00B7EB'      # deepskyblue2
}

Setup object.

In [ ]:
# ------------------------------------------------------------
# Load the public object
# ------------------------------------------------------------
mdata = mu.read_h5mu("data/immgenT-CD8.h5mu")  # ← update path if needed
rna = mdata["RNA"]
adt = mdata["ADT"]
print(mdata)

# ------------------------------------------------------------
# Standard normalization + scaling (RNA)
# ------------------------------------------------------------
# Keep raw counts in a layer
rna.layers["counts"] = rna.X.copy()

# Normalize → log1p
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)
rna.layers["log_norm"] = rna.X.copy()

# Scale
sc.pp.scale(rna, max_value=10)
rna.layers["scaled"] = rna.X.copy()

# ------------------------------------------------------------
# Standard normalization + scaling (ADT)
# ------------------------------------------------------------
adt.layers["counts"] = adt.X.copy()

# Option A: total-count normalization + log1p
sc.pp.normalize_total(adt, target_sum=1e4)
sc.pp.log1p(adt)
adt.layers["log_norm"] = adt.X.copy()

# Reset again to raw counts
adt.X = adt.layers["counts"].copy()

# Option B: Centered Log-Ratio (CLR) – often better for protein data
from muon import prot as pt
pt.pp.clr(adt)
adt.layers["clr"] = adt.X.copy()

# Scale (applied on the current .X, which is CLR at this point)
sc.pp.scale(adt, max_value=10)
adt.layers["scaled"] = adt.X.copy()

print("\nPreprocessing done.")
print(mdata)

# Fig. 1

In [ ]:
# Fig. 1d — global MDE
with plt.rc_context():
    fig, ax = plt.subplots(figsize=(6, 4.5))
    mu.pl.embedding(
        mdata['RNA'],
        basis="MDE_INCREMENTAL",
        color="cluster_annotation",
        palette=custom_colors,
        legend_loc="right margin",
        ax=ax,
        size=1,
        show=False,
    )
    plt.tight_layout(w_pad=0.55)
    plt.savefig("Fig_1d.pdf", bbox_inches="tight")
    plt.show()
    plt.close()
    print("Plot saved: Fig_1d.pdf")

In [ ]:
# Fig. 1e — atlas balloon plot (cluster proportions by sample)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from matplotlib.gridspec import GridSpec

# ==============================================
# Vector-export font settings (Illustrator-compatible)
# ==============================================
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

# ==============================================
# 1. Load and clean sample-level proportion table
# ==============================================
file_path = "data/immgenT-CD8.xlsx"
m3_full = pd.read_excel(file_path)

# Label standardization
m3_full['target_cells'] = m3_full['target_cells'].astype(str).str.strip()
m3_full['target_cells'] = m3_full['target_cells'].replace(['all T', 'allT', 'All T'], 'all CD8', regex=True)
m3_full['condition_broad'] = m3_full['condition_broad'].str.replace('cancer', 'tumor', regex=False, case=False)
m3_full['condition_broad'] = m3_full['condition_broad'].str.replace('autoimmune', 'autoimmunity', regex=False, case=False)

def add_cd8_prefix(label):
    if pd.isna(label):
        return label
    s = str(label).strip()
    if s.startswith('CD8') or s == 'all CD8':
        return s
    if s.endswith(' T') or s.endswith('T'):
        return f"CD8 {s.rstrip(' T').rstrip('T').strip()}"
    return s

m3_full['target_cells'] = m3_full['target_cells'].apply(add_cd8_prefix)
m3_full['target_cells'] = m3_full['target_cells'].str.replace('splenocytes', 'CD45+', regex=False)
m3_full['target_cells'] = m3_full['target_cells'].str.replace('CD8\+', 'CD8', regex=True)
m3_full['target_cells'] = m3_full['target_cells'].str.replace('CD8 T', 'CD8', regex=True)
m3_full['target_cells'] = m3_full['target_cells'].str.replace('OT1', 'OT-I', regex=True)
m3_full['target_cells'] = m3_full['target_cells'].str.replace('all CD8', 'CD8', regex=False)
m3_full['target_cells'] = m3_full['target_cells'].str.replace('CD44+ T', 'CD44+', regex=False)

special_markers = ['CD44+', 'VP1', 'PLZF+', 'Ly49', 'NK1.1', 'KLRG1+', 'CD69+', 'CD103+',
                   'CD62L-', 'CD27-', 'CD127-', 'Tbet+', 'Eomes+', 'Blimp1+', 'Granzyme+',
                   'iNKT', 'MAIT', 'NKT', 'γδ', 'Vd6b', 'Vg4', 'Vg6', 'Vg7']

pattern = r'CD8\s+(?=' + '|'.join([f'.*{m}' for m in special_markers]) + r')'
m3_full['target_cells'] = m3_full['target_cells'].str.replace(pattern, '', regex=True)

for m in special_markers:
    m3_full['target_cells'] = m3_full['target_cells'].str.replace(rf'CD8\s+{m}(\s|$)', m + ' ', regex=True)

# === Remove unwanted conditions from y-axis ===
unwanted_conditions = ['KbDbKO', 'KbDbQa1KO', 'NP_OVA', 'KbDbQa1KO_MCMV']
mask = m3_full['condition_detailed_simplified'].isin(unwanted_conditions)
n_removed = mask.sum()
m3_full = m3_full[~mask].copy()
print(f"Removed {n_removed} rows belonging to unwanted conditions: {unwanted_conditions}")

print("Data loaded, standardized, and filtered.")

# Canonical order
order_df = (m3_full[['condition_broad', 'condition_detailed_simplified']]
            .drop_duplicates()
            .sort_values(['condition_broad', 'condition_detailed_simplified'])
            .reset_index(drop=True))

CANONICAL_CONDITION_ORDER = order_df['condition_detailed_simplified'].unique().tolist()
print(f"Canonical order defined: {len(CANONICAL_CONDITION_ORDER)} unique detailed conditions")

# Color maps
unique_broad = sorted(order_df['condition_broad'].dropna().unique())
broad_palette = sns.color_palette("husl", len(unique_broad))
broad_color_map = dict(zip(unique_broad, broad_palette))
cmap_all = sns.color_palette("Greens", as_cmap=True)

# ==============================================
# DOTPLOT FUNCTION
# ==============================================
def make_comp_dotplot_level2_only(
    data,
    condition_col="condition_detailed_simplified",
    cmap=None,
    vmin=0,
    vmax=0.25,
    colorbar_ticks=[0, 0.05, 0.10, 0.15, 0.20, 0.25],
    figsize=(18, 22),
    outfile=None,
    exclude_clusters=None,
    min_cells_per_sample=10,
):
    if cmap is None:
        cmap = sns.color_palette("Greens", as_cmap=True)
    if exclude_clusters is None:
        exclude_clusters = []

    value_cols = [c for c in data.columns if c.endswith(('.prop', '.ncells'))]
    if not value_cols:
        print("No Level-2 columns found!")
        return pd.DataFrame()

    # Melt
    long = pd.melt(data, id_vars=[condition_col, 'sample_code'],
                   value_vars=value_cols, var_name='col', value_name='value')
    long[['cluster', 'metric']] = long['col'].str.rsplit('.', n=1, expand=True)
    long = long[~long['cluster'].isin(exclude_clusters)]

    # Sample quality filtering
    wide = long.pivot_table(index=[condition_col, 'sample_code', 'cluster'],
                            columns='metric', values='value', aggfunc='first').reset_index()
    
    sample_quality = wide.groupby(['sample_code', condition_col])['ncells'].max().reset_index()
    sample_quality['keep_sample'] = sample_quality['ncells'] >= min_cells_per_sample
    selected_samples = sample_quality[sample_quality['keep_sample']].copy()
    
    print(f"\nKept {len(selected_samples)} / {len(sample_quality)} samples "
          f"({len(selected_samples)/len(sample_quality)*100:.1f}%) based on min_cells_per_sample >= {min_cells_per_sample}")

    # Summary per condition
    total_per_cond = sample_quality.groupby(condition_col).size()
    good_per_cond = selected_samples.groupby(condition_col).size()
    summary = pd.DataFrame({'total_samples': total_per_cond})
    summary['selected_samples'] = good_per_cond
    summary['removed_samples'] = summary['total_samples'] - summary['selected_samples'].fillna(0)
    summary['kept_pct'] = (summary['selected_samples'].fillna(0) / summary['total_samples'] * 100).round(1)
    summary = summary.fillna({'selected_samples': 0, 'removed_samples': 0}).astype({
        'selected_samples': int, 'removed_samples': int
    })
    
    print("\n=== SAMPLES PER CONDITION (after cell count filtering) ===")
    print(summary.sort_values('kept_pct').to_string())

    # === Save good samples list (NEW) ===
    if outfile:
        selected_samples_df = selected_samples.copy()
        broad_map = order_df.set_index('condition_detailed_simplified')['condition_broad'].to_dict()
        selected_samples_df['condition_broad'] = selected_samples_df[condition_col].map(broad_map)
        
        csv_path = outfile.replace('.pdf', '_selected_samples.csv')
        selected_samples_df = selected_samples_df[[condition_col, 'condition_broad', 'sample_code', 'ncells']]
        selected_samples_df = selected_samples_df.sort_values([condition_col, 'sample_code'])
        selected_samples_df.to_csv(csv_path, index=False)
        print(f"\nSaved good samples list → {csv_path} ({len(selected_samples_df)} samples)")

    # Filter + aggregate
    wide_filtered = wide[wide['sample_code'].isin(selected_samples['sample_code'])].copy()

    agg = wide_filtered.groupby([condition_col, 'cluster']).agg(
        mean_prop=('prop', 'mean'),
        total_ncells=('ncells', 'sum'),
        n_samples=('sample_code', 'nunique')
    ).reset_index()

    valid_conditions = [c for c in CANONICAL_CONDITION_ORDER if c in agg[condition_col].values]
    agg = agg[agg[condition_col].isin(valid_conditions)].copy()
    agg[condition_col] = pd.Categorical(agg[condition_col], categories=valid_conditions, ordered=True)

    # Plotting
    cluster_order = []
    seen = set()
    for col in data.columns:
        if col.endswith('.prop'):
            cl = col.rsplit('.', 1)[0]
            if cl not in seen and cl not in exclude_clusters:
                cluster_order.append(cl)
                seen.add(cl)
    cluster_order = sorted(cluster_order)

    agg['cluster'] = pd.Categorical(agg['cluster'], categories=cluster_order, ordered=True)
    agg = agg.sort_values([condition_col, 'cluster']).copy()

    row_colors_filtered = [
        broad_color_map[order_df[order_df['condition_detailed_simplified'] == cond]['condition_broad'].iloc[0]]
        for cond in valid_conditions
    ]

    agg['size_log10'] = np.log10(agg['total_ncells'] + 1)

    fig = plt.figure(figsize=figsize)
    gs = GridSpec(1, 2, width_ratios=[0.07, 1], wspace=0.02)
    ax_cb = fig.add_subplot(gs[0])
    ax_cb.set_ylim(0, len(valid_conditions))
    ax_cb.invert_yaxis()
    ax_cb.axis('off')
    for i, color in enumerate(row_colors_filtered):
        ax_cb.add_patch(plt.Rectangle((0, i), 1, 1, facecolor=color, edgecolor='white', linewidth=0.5))

    ax = fig.add_subplot(gs[1])
    sns.scatterplot(data=agg, x='cluster', y=condition_col, size='size_log10',
                    sizes=(40, 800), hue='mean_prop', palette=cmap, hue_norm=(vmin, vmax),
                    edgecolor='black', linewidth=0.4, ax=ax, legend=False)

    ax.margins(y=0.015)
    plt.xticks(rotation=90, ha='center', fontsize=10.5)
    plt.yticks(fontsize=11)
    plt.xlabel(None)
    plt.ylabel(None)
    ax.grid(False)

    if outfile:
        title = os.path.basename(outfile).replace('.pdf', '').replace('_', ' ')
        plt.suptitle(title, fontsize=16, fontweight='bold', y=0.98)

    norm = plt.Normalize(vmin, vmax)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.8, aspect=30, pad=0.02)
    cbar.set_label('Mean proportion of CD8 T cells', fontsize=12, labelpad=12)
    cbar.set_ticks(colorbar_ticks)
    cbar.set_ticklabels([f"{t:.2f}" for t in colorbar_ticks])

    max_cells_val = agg['total_ncells'].max()
    handles, labels = [], []
    for size in [1, 10, 100, 1_000, 10_000, 50_000]:
        if size <= max_cells_val:
            s_scaled = np.log10(size + 1) * 800 / np.log10(max_cells_val + 1)
            handles.append(plt.scatter([], [], s=s_scaled, c='gray', edgecolor='black', linewidth=0.4))
            labels.append(f"{size:,}")
    ax.legend(handles, labels, title="Total cells", loc='center left',
              bbox_to_anchor=(1.08, 0.5), frameon=False, fontsize=11, title_fontsize=12)

    plt.tight_layout(rect=[0.06, 0.02, 0.88, 0.98])

    if outfile:
        plt.savefig(outfile, bbox_inches='tight', dpi=600, facecolor='white')
        print(f"Saved: {outfile}")
    else:
        plt.show()

    return agg


# ==============================================
# 5. Generate final atlas balloon plot
# ==============================================
out_dir = "./"
os.makedirs(out_dir, exist_ok=True)

make_comp_dotplot_level2_only(
    data=m3_full,
    cmap=cmap_all,
    vmin=0,
    vmax=0.25,
    colorbar_ticks=[0, 0.05, 0.10, 0.15, 0.20, 0.25],
    figsize=(15, 23),
    exclude_clusters=[],
    outfile=f"{out_dir}/Fig_1e.pdf",
    min_cells_per_sample=10
)

print("\nDONE!")


In [ ]:
# Fig. 1e — atlas dot plot (transgenic cells)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ================== 1. Load observation metadata ==================
df = mdata['RNA'].obs.copy()

# Ensure key columns are strings
for col in ['Ag_spe_v1', 'condition_broad', 'organ_simplified', 'condition_detailed_simplified', 'sample_code']:
    df[col] = df[col].astype(str)

# Label standardization (match atlas heatmap)
df['Ag_spe_v1'] = df['Ag_spe_v1'].str.strip()
df['Ag_spe_v1'] = df['Ag_spe_v1'].replace(['all T', 'allT', 'All T'], 'all CD8', regex=True)

def add_cd8_prefix(label):
    if pd.isna(label):
        return label
    s = str(label).strip()
    if s.startswith('CD8') or s == 'all CD8':
        return s
    if s.endswith(' T') or s.endswith('T'):
        return f"CD8 {s.rstrip(' T').rstrip('T').strip()}"
    return s

df['Ag_spe_v1'] = df['Ag_spe_v1'].apply(add_cd8_prefix)
df['Ag_spe_v1'] = df['Ag_spe_v1'].str.replace('splenocytes', 'CD45+', regex=False)
df['Ag_spe_v1'] = df['Ag_spe_v1'].str.replace('CD8\+', 'CD8', regex=True)
df['Ag_spe_v1'] = df['Ag_spe_v1'].str.replace('CD8 T', 'CD8', regex=True)
df['Ag_spe_v1'] = df['Ag_spe_v1'].str.replace('OT1', 'OT-I', regex=True)
df['Ag_spe_v1'] = df['Ag_spe_v1'].str.replace('all CD8', 'CD8', regex=False)
df['Ag_spe_v1'] = df['Ag_spe_v1'].str.replace('CD44+ T', 'CD44+', regex=False)

# Clean broad names
df['condition_broad'] = df['condition_broad'].str.replace('cancer', 'tumor', regex=False, case=False)
df['condition_broad'] = df['condition_broad'].str.replace('autoimmune', 'autoimmunity', regex=False, case=False)

special_markers = ['CD44+', 'VP1', 'PLZF+', 'Ly49', 'NK1.1', 'KLRG1+', 'CD69+', 'CD103+',
                   'CD62L-', 'CD27-', 'CD127-', 'Tbet+', 'Eomes+', 'Blimp1+', 'Granzyme+',
                   'iNKT', 'MAIT', 'NKT', 'γδ', 'Vd6b', 'Vg4', 'Vg6', 'Vg7']

pattern = r'CD8\s+(?=' + '|'.join([f'.*{m}' for m in special_markers]) + r')'
df['Ag_spe_v1'] = df['Ag_spe_v1'].str.replace(pattern, '', regex=True)

for m in special_markers:
    df['Ag_spe_v1'] = df['Ag_spe_v1'].str.replace(rf'CD8\s+{m}(\s|$)', m + ' ', regex=True)

print("Light label standardization complete (matching atlas1).")

# === Filter to good samples from CSV ===
selected_samples_path = './Fig_1e_selected_samples.csv'
selected_df = pd.read_csv(selected_samples_path)
selected_sample_codes = selected_df['sample_code'].astype(str).unique()

n_before = len(df)
df = df[df['sample_code'].isin(selected_sample_codes)].copy()
print(f"Filtered to good samples: kept {len(df)} / {n_before} cells "
      f"({len(selected_sample_codes)} samples)")

# Optional: also remove the specific unwanted conditions (like in atlas1)
unwanted_conditions = ['KbDbKO', 'KbDbQa1KO', 'NP_OVA', 'KbDbQa1KO_MCMV']
df = df[~df['condition_detailed_simplified'].isin(unwanted_conditions)].copy()
print(f"Additionally removed conditions: {unwanted_conditions}")

print(f"Final rows after all filtering: {len(df)}")

# ================== DATA PREP ==================
dfu = df[['condition_detailed_simplified', 'condition_broad', 'organ_simplified', 'Ag_spe_v1']].drop_duplicates()

# Alphabetical ordering
dfu = dfu.sort_values(['condition_broad', 'condition_detailed_simplified']).reset_index(drop=True)
conditions = dfu['condition_detailed_simplified'].unique()
cond_to_idx = {cond: i for i, cond in enumerate(conditions)}

present_pairs = dfu.groupby(['condition_detailed_simplified', 'organ_simplified']).size()
present_pairs = present_pairs[present_pairs > 0].index.tolist()

tissue_counts = dfu.groupby('organ_simplified').size()
tissues = tissue_counts.sort_values(ascending=False).index.tolist()
tissue_to_idx = {t: j for j, t in enumerate(tissues)}

print(f"Showing {len(present_pairs)} profiled combinations")

presence_dict = dfu.groupby(['condition_detailed_simplified', 'organ_simplified'])['Ag_spe_v1']\
    .apply(lambda x: sorted(set(x))).to_dict()

# Colors
colors = sns.color_palette("tab20") + sns.color_palette("tab20b") + sns.color_palette("tab20c")
all_targets = sorted(dfu['Ag_spe_v1'].unique())
target_color = dict(zip(all_targets, colors[:len(all_targets)]))

unique_broad = dfu['condition_broad'].dropna().unique()
broad_palette = sns.color_palette("husl", len(unique_broad))
broad_color_map = dict(zip(unique_broad, broad_palette))

broad_colors_for_rows = [
    broad_color_map[dfu[dfu['condition_detailed_simplified'] == cond]['condition_broad'].iloc[0]]
    for cond in conditions
]

# ================== PLOT ==================
fig = plt.figure(figsize=(17, len(conditions) * 0.6 + 3))
gs = fig.add_gridspec(1, 2, width_ratios=[0.08, 1], wspace=0.02)

# Left color bar
ax_colorbar = fig.add_subplot(gs[0])
ax_colorbar.set_xlim(0, 1)
ax_colorbar.set_ylim(0, len(conditions))
ax_colorbar.invert_yaxis()
ax_colorbar.axis('off')
for i, color in enumerate(broad_colors_for_rows):
    ax_colorbar.add_patch(plt.Rectangle((0, i), 1, 1, facecolor=color, edgecolor='white', linewidth=0.5))

# Main plot
ax = fig.add_subplot(gs[1])
cell_size = 0.9
border = 1.0

for cond, tissue in present_pairs:
    i = cond_to_idx[cond]
    j = tissue_to_idx[tissue]
    cx = j + 0.5
    cy = i + 0.5
    targets = presence_dict[(cond, tissue)]
    cols = [target_color[t] for t in targets]
    ax.pie([1] * len(targets),
           center=(cx, cy),
           radius=cell_size / 2,
           colors=cols,
           wedgeprops=dict(width=cell_size/2, edgecolor='black', linewidth=border))

ax.set_xticks(np.arange(len(tissues)) + 0.5)
ax.set_yticks(np.arange(len(conditions)) + 0.5)
ax.set_xticklabels(tissues, rotation=90, ha='center', fontsize=11, fontweight='bold')
ax.set_yticklabels(conditions, fontsize=10)
ax.set_xlim(0, len(tissues))
ax.set_ylim(0, len(conditions))
ax.set_aspect('equal')
ax.invert_yaxis()
ax.grid(False)

# Legend
handles = [plt.Rectangle((0,0),1,1, facecolor=target_color[t], edgecolor='black', lw=1) for t in all_targets]
ax.legend(handles, all_targets, title='Target Cell Type',
          bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9.5, title_fontsize=11, frameon=False)

ax.set_title('ImmGenT CD8 Atlas: Condition × Tissue Coverage\n'
             'Y-axis grouped & colored by condition_broad (alphabetical)',
             fontsize=16, fontweight='bold', pad=30)

plt.tight_layout()
with plt.rc_context({'pdf.fonttype': 42, 'pdf.compression': 0}):
    plt.savefig('Fig_1e_v2.pdf', dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')

plt.show()
print("Done!")
print(f"{len(present_pairs)} combinations | {len(conditions)} detailed conditions")


# Fig. 2

In [ ]:
# Fig. 2a — Genes dot plot
selected_genes = ["Tcf7", "Lef1", "Bach2", "Id3", "Zeb1", "Sell", "Il7r", "Fas", "Tbx21", "Cd44", "Zeb2", "Id2", "Klrg1", "Klrd1", "Itgax", "Cx3cr1", "Cd160", "Pecam1", "Gzma", "Gzmk", "Gzmb", "Ifng", "Prf1", "Entpd1", "Nt5e", "Runx3", "Hic1", "Itgae", "Cd69", "P2rx7", "Itga1", "Xcl1", "Cxcr6", "Ccr9", "Prdm1", "Tox", "Pdcd1", "Havcr2", "Tnfrsf18", "Mki67",]

sc.pl.dotplot(rna, var_names=list(selected_genes), groupby='cluster_annotation', 
              use_raw=False,  # Set to True if .X is raw and you have .raw
              cmap='Reds',  # Color map for expression
              title='Differentially Expressed Genes (RNA)',
              standard_scale='var',  # Scale by variable (gene)
              figsize=(13, 6),  # Try passing figsize directly
              show=False)
plt.savefig('Fig_2a.pdf', format='pdf', bbox_inches='tight')
plt.show()
plt.close()

print('Plot saved: Fig_2a.pdf')


In [ ]:
# Fig. 2a — Proteins dot plot
selected_proteins = ['CD62L', 'IL7RA.CD127', 'CD27', 'CD28', 'CD55.DAF', 'CD44', 'KLRG1', 'CD94', 'ITAX.CD11C', 'CX3CR1', 'CD160', 'B220', 'CD31', 'CD39', 'CD73.5NTD', 'CD103', 'CD69', 'CD49A', 'CXCR6.CD186', 'CD29', 'ITB7', 'PDCD1.PD1.CD279', 'HAVCR2', 'GITR.CD357']

sc.pl.dotplot(adt, 
              var_names=selected_proteins, 
              groupby='cluster_annotation',
              use_raw=False,
              cmap='Blues', 
              title='Differentially Expressed Proteins (ADT)',
              standard_scale='var', 
              figsize=(9, 6),
              expression_cutoff=0.75,  # This sets the positivity threshold
              size_title='Fraction positive',
              show=False)

plt.savefig('Fig_2a_v2.pdf', format='pdf', bbox_inches='tight')
plt.show()
plt.close()

print('Plot saved: Fig_2a_v2.pdf')

In [ ]:
# Fig. 2b — multi-condition MDE gallery
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd

# ------------------------------------------------------------------
# Automatically find up to TOP 3 condition_detailed_organ with most ENDOGENOUS cells
# for EACH condition_broad — preferring different IGT values — excluding spleen_NP_OVA and spleen_KbDbQa1KO
# ------------------------------------------------------------------
endo_obs = rna.obs[rna.obs['Ag_spe_v1'].str.lower() == 'endogenous'].copy()
# Exclude unwanted detailed conditions (case-insensitive)
exclude_patterns = ['spleen_NP_OVA', 'spleen_KbDbQa1KO']
mask_exclude = ~endo_obs['condition_detailed_organ'].str.lower().str.contains('|'.join(exclude_patterns))
endo_obs_filtered = endo_obs[mask_exclude]
# Add IGT for diversity selection
# (assuming 'IGT' is string or can be converted to str; NaN → 'unknown')
endo_obs_filtered['IGT_str'] = endo_obs_filtered['IGT'].astype(str).replace('nan', 'unknown')
# Group and count endogenous cells per (broad, detailed)
counts = (
    endo_obs_filtered
    .groupby(['condition_broad', 'condition_detailed_organ', 'IGT_str'])
    .size()
    .reset_index(name='n_endo')
)
# For each broad, sort by n_endo descending
counts = counts.sort_values(['condition_broad', 'n_endo'], ascending=[True, False])

# Select up to 3 per broad, preferring different IGT
selected = []
for broad, group in counts.groupby('condition_broad'):
    if group.empty:
        continue
    # Take the absolute top one
    top1 = group.iloc[0]
    selected.append(top1)
    
    # Remaining after top1
    remaining = group.iloc[1:]
    
    # Try to find a second one with different IGT
    different_igt = remaining[remaining['IGT_str'] != top1['IGT_str']]
    if not different_igt.empty:
        top2 = different_igt.iloc[0]
        selected.append(top2)
        # Update remaining after adding top2
        remaining = remaining[remaining.index != top2.name]
    
    # Try to find a third one with yet another different IGT (if possible)
    different_igt_3 = remaining[~remaining['IGT_str'].isin([top1['IGT_str']] + ([top2['IGT_str']] if 'top2' in locals() else []))]
    if not different_igt_3.empty:
        top3 = different_igt_3.iloc[0]
        selected.append(top3)

# Create final lists from automatic selection
selected_df = pd.DataFrame(selected)
broad_list = selected_df['condition_broad'].tolist()
detailed_list = selected_df['condition_detailed_organ'].tolist()

# ────────────────────────────────────────────────
# Force-add two extra specific conditions at the end (if not already present)
extra_detailed = ['colonLP_Foxp3mutant_D28', 'LNmediastinal_flu']

for extra_cond in extra_detailed:
    if extra_cond not in detailed_list:
        # Find the broad category for this condition
        matching_broad = rna.obs.loc[
            rna.obs['condition_detailed_organ'] == extra_cond,
            'condition_broad'
        ].unique()
        
        if len(matching_broad) == 1:
            broad_list.append(matching_broad[0])
            detailed_list.append(extra_cond)
        elif len(matching_broad) > 1:
            # Rare edge case: ambiguous broad → pick first or skip/log
            broad_list.append(matching_broad[0])
            detailed_list.append(extra_cond)
        # if no match → silently skip (or you can raise warning)

# ────────────────────────────────────────────────
n_broad = len(broad_list)
n_cols = 4
n_rows = int(np.ceil(n_broad / n_cols))

# --------------------------------------------------------------
# Figure size control
# --------------------------------------------------------------
width_per_plot = 3.4
height_per_plot = 3.8
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(width_per_plot * n_cols, height_per_plot * n_rows),
                         constrained_layout=True)
axes = axes.flatten()
default_color = "#BAB0AC"

# Pre-compute IGTHT info
igtht_info_list = []
for broad, condition in zip(broad_list, detailed_list):
    mask = (
        (rna.obs['condition_detailed_organ'] == condition) &
        (rna.obs['Ag_spe_v1'].str.lower() == 'endogenous')
    )
    igtht_values = rna.obs.loc[mask, 'IGTHT'].dropna().unique()
    if len(igtht_values) == 1:
        igtht_str = str(igtht_values[0])
    elif len(igtht_values) > 1:
        igtht_str = "multiple"
    else:
        igtht_str = "unknown"
    igtht_info_list.append(igtht_str)

# Plotting loop
for idx, ((broad, condition), ax, igtht_str) in enumerate(zip(zip(broad_list, detailed_list), axes, igtht_info_list)):
    # Highlight ALL endogenous cells from the selected detailed condition
    rna.obs['highlight_color'] = np.where(
        (rna.obs['condition_detailed_organ'] == condition) &
        (rna.obs['Ag_spe_v1'].str.lower() == 'endogenous'),
        rna.obs['cluster_annotation'],
        np.nan
    )
    sizes = np.where(rna.obs['highlight_color'].notnull(), 20, 1)
    
    # Only label clusters that have >10 highlighted cells
    highlighted_mask = rna.obs['highlight_color'].notnull()
    cluster_counts = rna.obs.loc[highlighted_mask, 'cluster_annotation'].value_counts()
    large_clusters = cluster_counts[cluster_counts > 10].index
    
    # Label only large clusters; small ones get NaN → no label
    rna.obs['highlight_color_clean'] = np.where(
        highlighted_mask & rna.obs['cluster_annotation'].isin(large_clusters),
        rna.obs['cluster_annotation'],
        np.nan
    )
    
    # ────────────────────────────────────────────────
    # Plot: color by the filtered labels (small clusters colored but unlabeled)
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color_clean",          # this controls both color AND which get labels
        palette=custom_colors,
        legend_loc='on data',
        ax=ax,
        size=sizes,
        show=False,
        title="",
        frameon=False
    )
    
    # Main title (broad condition)
    ax.set_title(broad, fontsize=13, fontweight='bold', pad=15)
    
    # Subtitle: detailed organ + IGTHT info
    subtitle = condition.replace('_', ' ')
    if igtht_str != "unknown":
        subtitle += f" ({igtht_str})"
    ax.text(0.5, 1.01, subtitle,
            transform=ax.transAxes, fontsize=10, ha='center', color='gray')
    
    # Cell count
    n_highlighted = sizes[sizes == 20].sum()
    ax.text(0.02, 0.98, f'n={n_highlighted}', transform=ax.transAxes,
            fontsize=8, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Hide unused subplots
for idx in range(n_broad, len(axes)):
    axes[idx].axis('off')

# ------------------------------------------------------------------
plt.savefig("Fig_2b.pdf", bbox_inches="tight")
plt.show()
plt.close()

# Clean up
rna.obs.drop(columns=['highlight_color', 'highlight_color_clean'], errors='ignore', inplace=True)

print("Combined plot saved: Fig_2b.pdf")
print("Plotted conditions (up to 3 per broad, preferring different IGT + 2 extras):")
for b, d, i in zip(broad_list, detailed_list, igtht_info_list):
    igtht_display = i if i != "unknown" else "(no IGTHT info)"
    print(f" • {b} → {d.replace('_', ' ')} | IGTHT: {igtht_display}")


In [ ]:
# Fig. 2c — melanoma B16 tumor on MDE
# Antigen-specificity groups to highlight
ag_spe_values = ['p14', 'pmel', 'endogenous']

# Gray background for non-highlighted cells
default_color = '#BAB0AC'  # Gray for cells not meeting the condition

# Three-panel layout
fig, axes = plt.subplots(1, 3, figsize=(10.5, 4))  # 3 subplots horizontally, adjusted size

# Iterate over Ag_spe values and corresponding axes
for idx, (ag_spe, ax) in enumerate(zip(ag_spe_values, axes)):
    # Create a new column for coloring based on conditions
    rna.obs['highlight_color'] = np.where(
        (rna.obs['IGT'] == 'IGT35') & 
        (rna.obs['sample_name'].str.lower().str.endswith('tumor')) & 
        (~rna.obs['condition_detailed_organ'].str.lower().str.contains('apd1a41bb')) &
        (rna.obs['Ag_spe_v1'].str.lower() == ag_spe),
        rna.obs['cluster_annotation'],  # Use cluster_annotation for highlighted cells
        None  # None for cells that don't meet the condition
    )
    
    # Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 20, 1)
    
    # Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None,  # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=f"Ag_spe: {ag_spe}"  # Add Ag_spe value as subplot title
    )

# Adjust layout to prevent overlap
plt.tight_layout()

# Save and show the combined plot
plt.savefig("Fig_2c.pdf", bbox_inches="tight")
plt.show()
plt.close()

print("Combined plot saved: Fig_2c.pdf")

In [ ]:
# Fig. 2c — lung KP tumor on MDE
# Antigen-specificity groups to highlight
ag_spe_values = ['ot1', 'endogenous']

# Gray background for non-highlighted cells
default_color = '#BAB0AC'  # Gray for cells not meeting the condition

# Two-panel layout
fig, axes = plt.subplots(1, 2, figsize=(7, 4))  # 2 subplots horizontally, adjusted size

# Iterate over Ag_spe values and corresponding axes
for idx, (ag_spe, ax) in enumerate(zip(ag_spe_values, axes)):
    # Create a new column for coloring based on conditions
    rna.obs['highlight_color'] = np.where(
        (rna.obs['IGT'].isin(['IGT95', 'IGT96'])) & 
        (rna.obs['condition_detailed_organ'] == 'lung_KP') & 
        (rna.obs['Ag_spe_v1'].str.lower() == ag_spe),
        rna.obs['cluster_annotation'],  # Use cluster_annotation for highlighted cells
        None  # None for cells that don't meet the condition
    )
    
    # Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 20, 1)
    
    # Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None,  # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=f"Ag_spe: {ag_spe}"  # Add Ag_spe value as subplot title
    )

# Adjust layout to prevent overlap
plt.tight_layout()

# Save and show the combined plot
plt.savefig("Fig_2c_v2.pdf", bbox_inches="tight")
plt.show()
plt.close()

print("Combined plot saved: Fig_2c_v2.pdf")

In [ ]:
# Fig. 2c — pancreatic PDAC tumor on MDE
# Antigen-specificity group to highlight
ag_spe = 'endogenous'

# Gray background for non-highlighted cells
default_color = '#BAB0AC'  # Gray for cells not meeting the condition

# Set up the figure with one subplot
fig, ax = plt.subplots(1, 1, figsize=(3.5, 4))

# Create a new column for coloring based on conditions
rna.obs['highlight_color'] = np.where(
    (rna.obs['IGT'].isin(['IGT64', 'IGT65'])) &
    (rna.obs['condition_detailed_organ'] == 'pancreas_PDAC') &
    (rna.obs['Ag_spe_v1'].str.lower() == ag_spe),
    rna.obs['cluster_annotation'],  # Use cluster_annotation for highlighted cells
    None  # None for cells that don't meet the condition
)

# Set dot sizes: larger for highlighted cells
sizes = np.where(rna.obs['highlight_color'].notnull(), 20, 1)

# Plot MDE
sc.pl.embedding(
    rna,
    basis="MDE_INCREMENTAL",
    color="highlight_color",
    palette=custom_colors,  # Ensure custom_colors is defined
    legend_loc=None,  # No legend
    ax=ax,
    size=sizes,
    show=False,
    title=f"Ag_spe: {ag_spe}"  # Add Ag_spe value as plot title
)

# Adjust layout
plt.tight_layout()

# Save and show the plot
plt.savefig("Fig_2c_v3.pdf", bbox_inches="tight")
plt.show()
plt.close()

print("Plot saved: Fig_2c_v3.pdf")

In [ ]:
# Fig. 2d-e — GP analysis on CD8.A and CD8.B
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load matrices (keep as before)
F_df = pd.read_csv("data/gene_factor_matrix.txt", sep="\t", index_col=0)
L_df = pd.read_csv("data/cell_factor_matrix.txt", sep="\t", index_col=0)

F_np = F_df.to_numpy()
L_np = L_df.to_numpy()

factor_names = L_df.columns

# MuData handling
mdata_obs = mdata['RNA'].obs
cell_id_col = 'cellID' if 'cellID' in mdata_obs.columns else 'IGT_cellID'

# Define the two clusters only
target_clusters = ['CD8.A', 'CD8.B']
unique_clusters = ['CD8.A', 'CD8.B']  # Only these two

print("Comparing only:", unique_clusters)

# Function to compute log2FC: CD8.A vs CD8.B
def compute_factor_log2fc(L_np, group1_cells, group2_cells, factor_names):
    group1_idx = [L_df.index.get_loc(g) for g in group1_cells if g in L_df.index]
    group2_idx = [L_df.index.get_loc(g) for g in group2_cells if g in L_df.index]
    if not group1_idx or not group2_idx:
        return None
    loadings_group1 = L_np[group1_idx].mean(axis=0)
    loadings_group2 = L_np[group2_idx].mean(axis=0)
    fc_loadings = loadings_group1 - loadings_group2
    return pd.DataFrame({'SYMBOL': factor_names, 'log2FC': fc_loadings / np.log(2)})

# Get cells
CD8_A_cells = mdata_obs[mdata_obs['cluster_annotation'] == 'CD8.A'][cell_id_col].tolist()
CD8_B_cells = mdata_obs[mdata_obs['cluster_annotation'] == 'CD8.B'][cell_id_col].tolist()

print(f"CD8.A cells: {len(CD8_A_cells)}")
print(f"CD8.B cells: {len(CD8_B_cells)}")

log2fc_df = compute_factor_log2fc(L_np, CD8_A_cells, CD8_B_cells, factor_names)

# Remove F1 and select strongest in each direction
log2fc_df = log2fc_df[log2fc_df['SYMBOL'] != 'F1']

strongest_CD8_A = log2fc_df.loc[log2fc_df['log2FC'].idxmax()]  # Highest positive = most enriched in CD8_A
strongest_CD8_B = log2fc_df.loc[log2fc_df['log2FC'].idxmin()]  # Lowest negative = most enriched in CD8_B

top_factors = [strongest_CD8_A['SYMBOL'], strongest_CD8_B['SYMBOL']]

print(f"Strongest factor for CD8.A: {strongest_CD8_A['SYMBOL']} (log2FC = {strongest_CD8_A['log2FC']:.3f})")
print(f"Strongest factor for CD8.B: {strongest_CD8_B['SYMBOL']} (log2FC = {strongest_CD8_B['log2FC']:.3f})")

# Compute mean loadings for these two factors in the two clusters
cluster_loadings = []
for cluster in unique_clusters:
    cells = mdata_obs[mdata_obs['cluster_annotation'] == cluster][cell_id_col].tolist()
    idx = [L_df.index.get_loc(g) for g in cells if g in L_df.index]
    if idx:
        mean_loadings = L_np[idx].mean(axis=0)
        for factor in top_factors:
            factor_idx = list(factor_names).index(factor)
            cluster_loadings.append({
                'cluster': cluster,
                'factor': factor,
                'mean_loading': mean_loadings[factor_idx]
            })

loadings_df = pd.DataFrame(cluster_loadings)
pivot_df = loadings_df.pivot(index='cluster', columns='factor', values='mean_loading')

# Ensure order
pivot_df = pivot_df.reindex(unique_clusters)

# Print the exact mean loadings for the strongest factors in each cluster
print("\nExact mean factor loadings for the top gene programs:")
print(pivot_df.round(4))  # Round to 4 decimals for readability, adjust as needed

# Alternatively, for a more verbose readout:
print("\nDetailed values:")
for cluster in unique_clusters:
    print(f"{cluster}:")
    for factor in top_factors:
        value = pivot_df.loc[cluster, factor]
        print(f"  {factor}: {value:.4f}")

# === Plotting (adapted from your code) ===
fig, axes = plt.subplots(nrows=len(top_factors), ncols=1, figsize=(1, 4 * len(top_factors)))
if len(top_factors) == 1:
    axes = [axes]

# Colors: one for each factor
colors = ["#b3c52d", "#24b2e1"]

for i, (factor, ax) in enumerate(zip(top_factors, axes)):
    x = np.arange(len(unique_clusters))
    # Bar plot
    bars = ax.bar(x, pivot_df[factor], color=colors[i], label=factor)
    
    # Highlight both clusters with black edge (since we only have two, both are "target")
    for j in x:
        ax.bar(j, pivot_df[factor].iloc[j], color=colors[i], edgecolor='black', linewidth=0.5)
    
    # Customize
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Mean Factor Loading')
    ax.set_title(f'Gene Program Activation: {factor}')
    ax.set_xticks(x)
    ax.set_ylim(0, 1)  # Set consistent y-axis limit
    ax.set_xticklabels(['CD8.A', 'CD8.B'], fontsize=12)
    ax.grid(False)
    ax.set_xlim(-0.6, len(unique_clusters) - 0.4)

plt.tight_layout()
plt.savefig('Fig_2d-e.pdf', dpi=300, bbox_inches='tight', format='pdf')
plt.show()


In [ ]:
# Fig. 2d-e — GP analysis on CD8.A and CD8.B
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import scipy.sparse as sparse

# Move scaled to X
mdata['RNA'].X = mdata['RNA'].layers['scaled']

gene_names = F_df.index.tolist()

# === Extract FULL ranked driver genes from loadings ===
factor_genes_full = {}
for factor in top_factors:
    factor_idx = list(factor_names).index(factor)
    gene_loadings = pd.Series(F_np[:, factor_idx], index=gene_names)
    sorted_genes = gene_loadings.abs().sort_values(ascending=False)
    factor_genes_full[factor] = sorted_genes.index.tolist()

# Optional: print top 30 drivers
for factor in top_factors:
    print(f"Top 30 driver genes for {factor}:")
    print(factor_genes_full[factor][:30])
    print()

# === Compute mean expression per cluster ===
unique_clusters = ['CD8.A', 'CD8.B']
cluster_mask = mdata_obs['cluster_annotation'].isin(unique_clusters)
cluster_labels = mdata_obs[cluster_mask]['cluster_annotation'].values
expr_data = mdata['RNA'].X[cluster_mask]

unique_labels, inverse = np.unique(cluster_labels, return_inverse=True)
n_genes = expr_data.shape[1]
mean_expr_matrix = np.empty((len(unique_labels), n_genes))

for i, label in enumerate(unique_labels):
    mask_i = (inverse == i)
    if sparse.issparse(expr_data):
        mean_expr_matrix[i] = np.ravel(expr_data[mask_i].mean(axis=0))
    else:
        mean_expr_matrix[i] = expr_data[mask_i].mean(axis=0)

mean_expr_pivot = pd.DataFrame(
    mean_expr_matrix,
    index=unique_labels,
    columns=mdata['RNA'].var_names
).T

mean_expr_pivot = mean_expr_pivot[unique_clusters]
print("Mean expression shape:", mean_expr_pivot.shape)

# === Target clusters per factor ===
factor_to_cluster = {'F9': 'CD8.A', 'F11': 'CD8.B'}

# === Manual genes to force-include (only if higher MEAN in target cluster and in factor) ===
manual_genes = {
    'F9': [],   # add your manual genes here if any
    'F11': []
}

allowed_genes = {factor: set(factor_genes_full[factor]) for factor in top_factors}
forced_genes = {}

for factor in top_factors:
    target = factor_to_cluster[factor]
    other = 'CD8.B' if target == 'CD8.A' else 'CD8.A'
    
    valid = []
    if factor in manual_genes:
        for g in manual_genes[factor]:
            if g in allowed_genes[factor] and mean_expr_pivot.loc[g, target] > mean_expr_pivot.loc[g, other]:
                valid.append(g)
    forced_genes[factor] = valid
    if valid:
        print(f"Forcing inclusion for {factor}: {valid}")
    else:
        print(f"No forced genes for {factor}")

# === Select top 30 cluster-specific genes by MEAN EXPRESSION DIFFERENCE ===
n_top_genes = 30
min_expr_threshold = 0.1          # lowered as requested

factor_genes_final = {}

for factor in top_factors:
    target = factor_to_cluster[factor]
    other = 'CD8.B' if target == 'CD8.A' else 'CD8.A'
    
    # Compute difference: positive = higher in target
    if target == 'CD8.A':
        delta = mean_expr_pivot['CD8.A'] - mean_expr_pivot['CD8.B']
        sorted_delta = delta.sort_values(ascending=False)
    else:
        delta = mean_expr_pivot['CD8.B'] - mean_expr_pivot['CD8.A']
        sorted_delta = delta.sort_values(ascending=False)  # still positive = better for CD8.A
    
    # Exclude ribosomal genes
    filtered = sorted_delta[~sorted_delta.index.str.startswith(('Rpl', 'Rps'))]
    
    # Require reasonable expression in at least one cluster + higher in target
    cluster_specific = filtered[
        (mean_expr_pivot.loc[filtered.index, target] > min_expr_threshold) &
        (mean_expr_pivot.loc[filtered.index, target] > mean_expr_pivot.loc[filtered.index, other])
    ]
    
    # Only keep genes from the factor's driver list
    cluster_specific = cluster_specific[cluster_specific.index.isin(allowed_genes[factor])]
    
    # Take top 30 (or all available if fewer)
    top_genes = cluster_specific.head(n_top_genes).index.tolist()
    
    # Add forced genes at the beginning (if any)
    selected_genes = list(set(forced_genes[factor])) + [g for g in top_genes if g not in forced_genes[factor]]
    
    # If after adding forced we still have < 30, we keep only what we have
    factor_genes_final[factor] = selected_genes[:n_top_genes]
    
    print(f"\n=== {factor}: Top {len(factor_genes_final[factor])} genes with highest mean expression difference ===")
    print(f"(higher in {target}, from factor drivers, min expr > {min_expr_threshold})")
    print(factor_genes_final[factor])
    print()

# === Build expression DataFrame ===
expr_data = []
for factor in top_factors:
    for gene in factor_genes_final[factor]:
        if gene in mean_expr_pivot.index:
            for cluster in unique_clusters:
                expr_data.append({
                    'factor': factor,
                    'gene': gene,
                    'cluster': cluster,
                    'mean_expr': mean_expr_pivot.loc[gene, cluster]
                })

expr_df = pd.DataFrame(expr_data)

# === Colormaps ===
colors_f9 = ["lightgray", "white", "#b3c52d", '#b3c52d']
colors_f11 = ["lightgray", "white", "#24b2e1", '#24b2e1']

cmap_f9 = LinearSegmentedColormap.from_list('cmap_f9', colors_f9)
cmap_f11 = LinearSegmentedColormap.from_list('cmap_f11', colors_f11)

colormaps = {'F9': cmap_f9, 'F11': cmap_f11}

# === Plotting: Heatmaps showing top genes by mean expression difference ===
for factor in top_factors:
    factor_df = expr_df[expr_df['factor'] == factor]
    pivot = factor_df.pivot(index='gene', columns='cluster', values='mean_expr')
    pivot = pivot[unique_clusters]
    
    # Sort by mean expression difference (highest in target at top)
    target_col = factor_to_cluster[factor]
    other_col = 'CD8.B' if target_col == 'CD8.A' else 'CD8.A'
    pivot['diff'] = pivot[target_col] - pivot[other_col]
    pivot = pivot.sort_values(by='diff', ascending=False).drop(columns='diff')
    
    fig, ax = plt.subplots(figsize=(1.2, 8))  # slightly wider and taller for clarity
    
    im = ax.imshow(
        pivot.values,
        cmap=colormaps[factor],
        aspect='auto',
        interpolation='nearest'
    )
    
    # You can keep or adjust these limits according to your actual data range
    im.set_clim(vmin=-0.5, vmax=1.2)
    
    ax.set_xticks(np.arange(len(unique_clusters)))
    ax.set_yticks(np.arange(len(pivot)))
    ax.set_xticklabels(['CD8.A', 'CD8.A'], fontsize=12)
    ax.set_yticklabels(pivot.index, fontsize=9)
    
    ax.set_xlabel('Cluster', fontsize=12)
    ax.set_ylabel('Genes', fontsize=12)
    
    ax.set_title(f'{factor}\nTop genes by mean expr difference\n(higher in {"CD8.A" if factor=="F9" else "CD8.A"})',
                 fontsize=14, pad=20)
    
    ax.grid(False)
    
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Mean Normalized Expression', fontsize=11)
    
    plt.tight_layout()
    plt.savefig(f'Fig_2d-e_v2_{factor}.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    print(f"Saved: Fig_2d-e_v2_{factor}.pdf\n")


In [ ]:
# Fig. 2f — NLT vs SLO proportions within CD8.A/B
import pandas as pd
import matplotlib.pyplot as plt

# Load the Excel file
file_path = 'data/immgenT-CD8.xlsx'
df = pd.read_excel(file_path)

# ====================== TISSUE CLASSIFICATION ======================
def assign_tissue_category(row):
    if pd.notna(row['condition_detailed_organ']):
        organ = str(row['condition_detailed_organ']).lower()
        
        tumor_lns = ['lninguinal_b16_act', 'lninguinal_b16_actapd1a41bb', 
                     'lnmediastinal_kp', 'lninguinal_kp']
        if any(tumor_ln in organ for tumor_ln in tumor_lns):
            return 'Tumor'
        
        lymphoid_keywords = ['spleen', 'thymus', 'ln', 'bonemarrow', 'bone marrow', 'blood']
        if any(kw in organ for kw in lymphoid_keywords):
            return 'Lymphoid organs and blood'
    
    if pd.notna(row.get('condition_broad')) and ('tumor' in str(row['condition_broad']).lower() or 
                                                  'cancer' in str(row['condition_broad']).lower()):
        return 'Tumor'
    
    return 'NLT'


df['tissue_category'] = df.apply(assign_tissue_category, axis=1)

# ====================== FILTERS ======================
if 'target_cells' in df.columns:
    df = df[~df['target_cells'].str.contains('CD4\+ CD44\+', case=False, na=False)].copy()

unwanted = ['Treg', 'CD4+ T', 'DN', 'DO11.10 KJ1-26', 'OT2', 'TCRBV-TCRGD', 'TFH',
            'Vd6b F4.22', 'Vg4 49.2', 'Vg6 1C10', 'Vg7 F2.67', 'thymocytes']
mask = df['target_cells'].isin(unwanted) | df['target_cells'].str.contains('|'.join(unwanted), case=False, na=False)
df = df[~mask].copy()

exclude_conditions = ['KbDbKO', 'KbDbQa1KO', 'KbDbQa1KO_MCMV']
df = df[~df['condition_detailed_organ'].str.contains('|'.join(exclude_conditions), case=False, na=False)].copy()

if 'condition_detailed_simplified' in df.columns:
    df = df[df['condition_detailed_simplified'] != 'baseline'].copy()

# ====================== AGGREGATE REPLICATES ======================
group_cols = ['condition_detailed_organ', 'tissue_category']
agg_dict = {
    'CD8.A.prop': 'mean',
    'CD8.B.prop': 'mean',
    'CD8.A.ncells': 'mean',
    'CD8.B.ncells': 'mean'
}
if 'target_cells' in df.columns:
    agg_dict['target_cells'] = 'first'

df_agg = df.groupby(group_cols, as_index=False).agg(agg_dict)

# Quality filter
min_cells = 10
valid_mask = (df_agg['CD8.A.ncells'] >= min_cells) | (df_agg['CD8.B.ncells'] >= min_cells)
df_filtered = df_agg[valid_mask].copy()

df_filtered = df_filtered[df_filtered['tissue_category'] != 'Tumor'].copy()

print(f"Final number of unique conditions used: {df_filtered.shape[0]}")

# ====================== CALCULATE PROPORTIONS USING COUNTS ======================
lymph_A = df_filtered[df_filtered['tissue_category'] == 'Lymphoid organs and blood']['CD8.A.ncells'].sum()
nlt_A   = df_filtered[df_filtered['tissue_category'] == 'NLT']['CD8.A.ncells'].sum()
lymph_B = df_filtered[df_filtered['tissue_category'] == 'Lymphoid organs and blood']['CD8.B.ncells'].sum()
nlt_B   = df_filtered[df_filtered['tissue_category'] == 'NLT']['CD8.B.ncells'].sum()

total_A = lymph_A + nlt_A
total_B = lymph_B + nlt_B

summary = pd.DataFrame({
    'CD8.A': {
        'NLT': nlt_A / total_A,
        'Lymphoid organs and blood': lymph_A / total_A
        
    },
    'CD8.B': {
        'Lymphoid organs and blood': lymph_B / total_B,
        'NLT': nlt_B / total_B
    }
}).T

# ====================== PLOT ======================
fig, ax = plt.subplots(figsize=(5, 4))

summary.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    color=['#952391', '#FFA811'],
    edgecolor='black'
)

ax.set_title('Proportion of tissue origin in CD8 subclusters\n(Lymphoid organs and blood vs NLT)', fontsize=14)
ax.set_xlabel('CD8 Subcluster', fontsize=12)
ax.set_ylabel('Proportion within CD8+ T cells', fontsize=12)

ax.legend(title='Tissue Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(False)
plt.xticks(rotation=0, ha='center', fontsize=12)
plt.yticks(fontsize=12)
plt.ylim(0, 1.05)
plt.tight_layout()

plt.savefig('Fig_2f.pdf', bbox_inches='tight')
plt.show()
plt.close()

print("Plot saved: Fig_2f.pdf")
print(f"\nTotal cells used → CD8.A: {total_A:,.0f} | CD8.B: {total_B:,.0f}")
print("\nFinal proportions:")
print(summary.round(3))


# Fig. 3

In [ ]:
# Fig. 3a — Effector memory clusters on MDE plot
# Memory clusters: early (D7) vs late (D30+D60) SLO embeddings
titles = ['D7', 'D30 + D60']

# Define the three organs to merge
organs = ['LNmediastinal', 'spleen', 'LNmesenteric']

# Gray background for non-highlighted cells
default_color = '#BAB0AC' # Gray for cells not meeting the condition

# Set up the figure with TWO subplots
fig, axes = plt.subplots(1, 2, figsize=(7, 4))

# Iterate over the two plots
for idx, (title, ax) in enumerate(zip(titles, axes)):
    
    if idx == 0:  # First plot: D7 only
        conditions = [f"{organ}_LCMVarm_D7" for organ in organs]
    else:         # Second plot: D30 + D60
        conditions = [f"{organ}_LCMVarm_D30" for organ in organs] + \
                     [f"{organ}_LCMVarm_D60" for organ in organs]
    
    # Create mask for the selected conditions
    mask = rna.obs['condition_detailed_organ'].isin(conditions)
    
    # Create a new column for coloring based on conditions
    rna.obs['highlight_color'] = np.where(
        mask & (rna.obs['Ag_spe_v2'].str.lower() == 'p14'),
        rna.obs['cluster_annotation'], # Use cluster_annotation for highlighted cells
        None # None for cells that don't meet the condition
    )
    
    # Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 100, 1)
    
    # Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None, # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=title # Use nice title
    )

# Adjust layout to prevent overlap
plt.tight_layout()

# Save and show the combined plot
plt.savefig("Fig_3a.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Combined plot saved: Fig_3a.pdf")


In [ ]:
# Fig. 3a — Effector memory clusters on MDE plot
# SLO organs merged per time point (endogenous highlight)
timepoints = ['D7', 'D30', 'D60']

# Define the three conditions (organs) to merge for each time point
organs = ['LNmediastinal', 'spleen', 'LNmesenteric']

# Gray background for non-highlighted cells
default_color = '#BAB0AC'  # Gray for cells not meeting the condition

# Set up the figure with three subplots (one per time point)
fig, axes = plt.subplots(1, 3, figsize=(10.5, 4))

# Iterate over time points and corresponding axes
for idx, (tp, ax) in enumerate(zip(timepoints, axes)):
    
    # Create condition list for this time point
    conditions = [f"{organ}_LCMVarm_{tp}" for organ in organs]
    
    # Create a new column for coloring: highlight endogenous cells from ANY of the three organs at this time point
    rna.obs['highlight_color'] = np.where(
        (rna.obs['condition_detailed_organ'].isin(conditions)) & 
        (rna.obs['Ag_spe_v2'].str.lower() == 'endogenous'),
        rna.obs['cluster_annotation'],   # Use cluster_annotation for highlighted cells
        None                            # None for all other cells
    )
    
    # Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 20, 1)
    
    # Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None,          # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=f"LCMVarm {tp}"     # Title shows the time point
    )

# Adjust layout to prevent overlap
plt.tight_layout()

# Save and show the combined plot
plt.savefig("Fig_3a_v2.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Combined plot saved: Fig_3a_v2.pdf")

In [ ]:
# Fig. 3f — Dot plot
import pandas as pd
import numpy as np

def select_top_genes(signature_file, clusters, pval_threshold=0.05, log2fc_threshold=0.5, highlight_gene_column='SYMBOL'):
    """
    Select the top 10 up-regulated and top 10 down-regulated genes for each cluster from a signature file.
    Parameters:
    - signature_file (str): Path to the signature file with DE results.
    - clusters (list): List of cluster names (e.g., ['CD8.E', 'CD8.F', ...]).
    - pval_threshold (float): Adjusted p-value threshold for significance.
    - log2fc_threshold (float): |log2FC| threshold for significance.
    - highlight_gene_column (str): Column name for genes in signature_file.
    Returns:
    - dict: Dictionary with cluster names as keys and tuples of (top_10_up, top_10_down) gene lists as values.
    """
    # Load the signature DataFrame
    signature_df = pd.read_csv(signature_file, sep='\t', encoding='utf-8')
    
    top_genes = {}
    for cluster in clusters:
        # Prepare data for the cluster
        log2fc_col = f'log2FC_{cluster}_vs_All'
        pval_col = f'adj.P.Val_{cluster}_vs_All'
        plot_df = signature_df[[highlight_gene_column, log2fc_col, pval_col]].copy()
        plot_df['gene'] = plot_df[highlight_gene_column]
        plot_df['log2FC'] = plot_df[log2fc_col]
        plot_df['minus_log10_padj'] = -np.log10(plot_df[pval_col].clip(lower=1e-300))
        
        # Highlight significant genes
        significant = plot_df[
            (plot_df[pval_col] < pval_threshold) & 
            (abs(plot_df['log2FC']) > log2fc_threshold)
        ]
        
        # Select top 10 up-regulated (highest positive log2FC)
        top_10_up = significant[significant['log2FC'] > 0][['gene', 'log2FC']].sort_values(
            'log2FC', ascending=False
        ).head(10)['gene'].tolist()
        
        # Select top 10 down-regulated (lowest negative log2FC)
        top_10_down = significant[significant['log2FC'] < 0][['gene', 'log2FC']].sort_values(
            'log2FC', ascending=True
        ).head(10)['gene'].tolist()
        
        top_genes[cluster] = (top_10_up, top_10_down)
    
    return top_genes

# Define file path and clusters
main_signature_file = 'data/ttlist_OneVsAll.txt'
clusters = ['CD8.I', 'CD8.J', 'CD8.K']

# Select top genes
top_genes_dict = select_top_genes(
    signature_file=main_signature_file,
    clusters=clusters,
    pval_threshold=0.05,
    log2fc_threshold=0.5,
    highlight_gene_column='SYMBOL'
)

# Print results
for cluster in clusters:
    up_genes, down_genes = top_genes_dict[cluster]
    print(f"Cluster {cluster}:")
    print(f"  Top 10 Up-regulated Genes: {up_genes}")
    print(f"  Top 10 Down-regulated Genes: {down_genes}")


In [ ]:
# Fig. 3f — Dot plot
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================================================
# 0. Clusters of interest
# ==============================================================
clusters_of_interest = ['CD8.I', 'CD8.J', 'CD8.K']

# ==============================================================
# 1. Subset AnnData to only these 3 clusters
# ==============================================================
adata = mdata['RNA']
subset_adata = adata[adata.obs['cluster_annotation'].isin(clusters_of_interest)]
print(f"Subset contains {subset_adata.n_obs} cells from {len(clusters_of_interest)} clusters")

# ==============================================================
# 2. Build ordered gene list
# ==============================================================
ordered_genes = []

# ---- BLOCK 0: All UP-regulated genes (CD8.I → CD8.J → CD8.K) ----
for cl in clusters_of_interest:
    for g in top_genes_dict[cl][0]:  # UP genes
        if g not in ordered_genes and g in subset_adata.var_names:
            ordered_genes.append(g)

# ---- BLOCK 1: Extra genes ----
extra_genes = ['Il2', 'Tnf', 'Ifng', 'Prf1', 'Lamp1']
for g in extra_genes:
    if g in subset_adata.var_names and g not in ordered_genes:
        ordered_genes.append(g)

# ---- BLOCK 2: All DOWN-regulated genes (CD8.I → CD8.J → CD8.K) ----
for cl in clusters_of_interest:
    for g in top_genes_dict[cl][1]:  # DOWN genes
        if g not in ordered_genes and g in subset_adata.var_names:
            ordered_genes.append(g)

# Warn about any missing genes
missing = [g for g in extra_genes + 
           [g for cl in clusters_of_interest for g in top_genes_dict[cl][0] + top_genes_dict[cl][1]] 
           if g not in subset_adata.var_names]
if missing:
    print(f"Warning: {len(missing)} requested genes not found in dataset: {missing}")

print(f"Final gene order ({len(ordered_genes)} genes): "
      f"{len(extra_genes)} extra @ top ↑ | UP block | DOWN block ↓")

# ==============================================================
# 3. Subset to these genes
# ==============================================================
adata_plot = subset_adata[:, ordered_genes].copy()

# Ensure correct category order for clusters
adata_plot.obs['cluster_annotation'] = adata_plot.obs['cluster_annotation'].astype('category')
adata_plot.obs['cluster_annotation'] = adata_plot.obs['cluster_annotation'].cat.reorder_categories(clusters_of_interest)

# ==============================================================
# 4. FINAL DOTPLOT — extra genes at the TOP, genes on y-axis
# ==============================================================
sc.pl.dotplot(
    adata_plot,
    var_names=ordered_genes,           # respects our custom order
    groupby='cluster_annotation',
    cmap='Reds',
    standard_scale='var',
    colorbar_title='Mean expression\n(z-scored per gene)',
    size_title='Fraction expressing',
    figsize=(2.75, 14),                   # tall figure for many genes
    title='Top Unique Markers in CD8.I/J/K\n'
          '(Il2, Tnf, Ifng, Prf1, Lamp1 at the top → UP → DOWN)',
    save='Fig_3f.pdf',
    dendrogram=False,
    swap_axes=True,                    # genes on y, clusters on x
    show=True
)


In [ ]:
# Fig. 3g — Effector cluster frequency across samples
import pandas as pd
import matplotlib.pyplot as plt

# Load the Excel file
file_path = 'data/immgenT-CD8.xlsx'
df = pd.read_excel(file_path)

print(f"Loaded Excel file with {df.shape[0]} rows and {df.shape[1]} columns.")

# Define columns
prop_cols = ['CD8.I.prop', 'CD8.J.prop', 'CD8.K.prop']
ncells_cols = ['CD8.I.ncells', 'CD8.J.ncells', 'CD8.K.ncells']

# === FILTERS (applied on original rows before averaging) ===
if 'target_cells' in df.columns:
    df = df[~df['target_cells'].str.contains('CD4\+ CD44\+', case=False, na=False)].copy()
    unwanted = ['Treg', 'CD4+ T', 'DN', 'DO11.10 KJ1-26', 'OT2', 'TCRBV-TCRGD', 'TFH',
                'Vd6b F4.22', 'Vg4 49.2', 'Vg6 1C10', 'Vg7 F2.67', 'thymocytes']
    mask = df['target_cells'].isin(unwanted) | df['target_cells'].str.contains('|'.join(unwanted), case=False, na=False)
    df = df[~mask].copy()
    print(f"After target_cells filters: {df.shape[0]} rows remain.")

# Exclude KO conditions
exclude_conditions = ['KbDbKO', 'KbDbQa1KO', 'KbDbQa1KO_MCMV']
exclude_mask = df['condition_detailed_organ'].str.contains('|'.join(exclude_conditions), case=False, na=False)
df = df[~exclude_mask].copy()
print(f"After excluding KbDb conditions: {df.shape[0]} rows remain.")

# Exclude baseline
if 'condition_detailed_simplified' in df.columns:
    df = df[df['condition_detailed_simplified'] != 'baseline'].copy()
    print(f"After excluding baseline: {df.shape[0]} rows remain.")

# === AGGREGATE REPLICATES (Group by condition_detailed_organ) ===
group_cols = ['condition_detailed_organ']

agg_dict = {col: 'mean' for col in prop_cols + ncells_cols}
if 'target_cells' in df.columns:
    agg_dict['target_cells'] = 'first'   # keep one value for reference

df_agg = df.groupby(group_cols, as_index=False).agg(agg_dict)

print(f"After aggregating replicates: {df_agg.shape[0]} unique conditions.")

# === FILTER on averaged ncells: ≥10 in at least one cluster ===
min_cells = 10
valid_mask = (df_agg['CD8.I.ncells'] >= min_cells) | \
             (df_agg['CD8.J.ncells'] >= min_cells) | \
             (df_agg['CD8.K.ncells'] >= min_cells)

df_filtered = df_agg[valid_mask].copy()
print(f"After ≥ {min_cells} cells (averaged) in at least one cluster: {df_filtered.shape[0]} conditions remain.")

# Calculate total proportion for ranking
df_filtered['total_prop'] = df_filtered[prop_cols].sum(axis=1)
df_filtered = df_filtered[df_filtered['total_prop'] > 0].copy()

# Select top 50 unique conditions
df_top50 = df_filtered.nlargest(50, 'total_prop')

print(f"Selected top 50 unique conditions by average combined proportion.")

# Prepare for plotting
freq_table = df_top50[prop_cols].copy()
freq_table.index = df_top50['condition_detailed_organ']
freq_table.columns = ['CD8.I', 'CD8.J', 'CD8.K']

# Plot using your pre-existing custom_colors
fig, ax = plt.subplots(figsize=(14, 8))
freq_table.plot(kind='bar', stacked=True, ax=ax, 
                color=[custom_colors[col] for col in freq_table.columns])

ax.set_xlabel('Condition (Detailed Organ)')
ax.set_ylabel('Percentage within each sample (%)')
ax.set_title('Top 50 Unique Conditions by CD8.I + CD8.J + CD8.K Proportion\n(Averaged across replicates • After all filters)')
ax.grid(False)
ax.legend(title='Cluster', bbox_to_anchor=(1, 1), loc='upper left')

plt.xticks(rotation=90, ha='center')
plt.tight_layout()

# Save
plt.savefig('Fig_3g.pdf', bbox_inches='tight')
plt.show()
plt.close()

print("Stacked bar plot saved: Fig_3g.pdf")

# Preview
print("\nTop 10 unique conditions:")
print(df_top50[['condition_detailed_organ', 'total_prop'] + ncells_cols + prop_cols].head(10))


In [ ]:
# Fig. 3h — Representative condition MDEs
conditions = ['spleen_Foxp3mutant_D19', 'lung_MTB_3w', 'uterus_CtrachomatisD_primaryD5']

# Gray background for non-highlighted cells
default_color = '#BAB0AC'  # Gray for cells not meeting the condition

# Three-panel layout
fig, axes = plt.subplots(1, 3, figsize=(10.5, 4))  # 3 subplots horizontally, adjusted size

# Iterate over conditions and corresponding axes
for idx, (condition, ax) in enumerate(zip(conditions, axes)):
    # Create a new column for coloring based on conditions
    rna.obs['highlight_color'] = np.where(
        rna.obs['condition_detailed_organ'] == condition,
        rna.obs['cluster_annotation'],  # Use cluster_annotation for highlighted cells
        None  # None for cells that don't meet the condition
    )
    
    # Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 20, 1)
    
    # Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None,  # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=condition  # Add condition as subplot title
    )

# Adjust layout to prevent overlap
plt.tight_layout()

# Save and show the combined plot
plt.savefig("Fig_3h.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Combined plot saved: Fig_3h.pdf")


# Fig. 4


In [ ]:
# Fig. 4a — Circulating memory clusters on MDE plot
# Memory clusters: early (D7) vs late (D30+D60) SLO embeddings
titles = ['D7', 'D30 + D60']

# Define the three organs to merge
organs = ['LNmediastinal', 'spleen', 'LNmesenteric']

# Gray background for non-highlighted cells
default_color = '#BAB0AC' # Gray for cells not meeting the condition

# Set up the figure with TWO subplots
fig, axes = plt.subplots(1, 2, figsize=(7, 4))

# Iterate over the two plots
for idx, (title, ax) in enumerate(zip(titles, axes)):
    
    if idx == 0:  # First plot: D7 only
        conditions = [f"{organ}_LCMVarm_D7" for organ in organs]
    else:         # Second plot: D30 + D60
        conditions = [f"{organ}_LCMVarm_D30" for organ in organs] + \
                     [f"{organ}_LCMVarm_D60" for organ in organs]
    
    # Create mask for the selected conditions
    mask = rna.obs['condition_detailed_organ'].isin(conditions)
    
    # Create a new column for coloring based on conditions
    rna.obs['highlight_color'] = np.where(
        mask & (rna.obs['Ag_spe_v2'].str.lower() == 'p14'),
        rna.obs['cluster_annotation'], # Use cluster_annotation for highlighted cells
        None # None for cells that don't meet the condition
    )
    
    # Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 100, 1)
    
    # Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None, # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=title # Use nice title
    )

# Adjust layout to prevent overlap
plt.tight_layout()

# Save and show the combined plot
plt.savefig("Fig_4a.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Combined plot saved: Fig_4a.pdf")


In [ ]:
# Fig. 4a — spleen LCMV tetramer⁺ populations on MDE
# Conditions to highlight
conditions = ['spleen_dirty_2w', 'spleen_dirty_2m',
'spleen_dirty_2m_LCMVarm_D7', 'spleen_dirty_2m_LCMVarm_D30',
'spleen_LCMVarm_D30']
# Gray background for non-highlighted cells
default_color = '#BAB0AC' # Gray for cells not meeting the condition
# Multi-panel layout
fig, axes = plt.subplots(1, len(conditions), figsize=(3.5 * len(conditions), 4)) # 3 subplots horizontally, adjusted size
# Iterate over conditions and corresponding axes
for idx, (condition, ax) in enumerate(zip(conditions, axes)):
# Create a new column for coloring based on conditions
    if condition in ['spleen_dirty_2w', 'spleen_dirty_2m']:
        rna.obs['highlight_color'] = np.where(
            (rna.obs['condition_detailed_organ'] == condition) & (rna.obs['Ag_spe_v1'].str.lower() == 'endogenous'),
            rna.obs['cluster_annotation'], # Use cluster_annotation for highlighted cells
None # None for cells that don't meet the condition
        )
    else:
        rna.obs['highlight_color'] = np.where(
            (rna.obs['condition_detailed_organ'] == condition) & (rna.obs['Ag_spe_v1'].str.lower() == 'gp33gp276np396tetp'),
            rna.obs['cluster_annotation'], # Use cluster_annotation for highlighted cells
None # None for cells that don't meet the condition
        )
# Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 20, 1)
# Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None, # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=condition # Add condition as subplot title
    )
# Adjust layout to prevent overlap
plt.tight_layout()
# Save and show the combined plot
plt.savefig("Fig_4a_v2.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Combined plot saved: Fig_4a_v2.pdf")


In [ ]:
# Fig. 4d — Dot plot of DEGs in memory clusters
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc  # Explicitly import scanpy for clarity

# Define selected genes
selected_genes = ["Tcf7", "Lef1", "Bach2", "Id3", "Zeb1", "Sell", "Il7r", "Tbx21", "Cd44", "Zeb2",
                  "Id2", "Klrg1", "Klrd1", "Itgax", "Cx3cr1", "Cd160", "Pecam1", "Gzma", "Gzmk",
                  "Gzmb", "Ifng", "Prf1", "Entpd1", "Nt5e", "Runx3", "Hic1", "Itgae", "Cd69",
                  "P2rx7", "Itga1", "Xcl1", "Cxcr6", "Ccr9", "Prdm1", "Tox", "Pdcd1", "Havcr2",
                  "Tnfrsf18", "Mki67"]

# Subset mdata['RNA'] to include only clusters CD8.E, CD8.F, CD8.G, and CD8.H
subset_adata = mdata['RNA'][mdata['RNA'].obs['cluster_annotation'].isin(['CD8.E', 'CD8.F', 'CD8.G', 'CD8.H']), :]

# Create dot plot with the subsetted data
sc.pl.dotplot(subset_adata, var_names=list(selected_genes), groupby='cluster_annotation',
              use_raw=False,  # Set to True if .X is raw and you have .raw
              cmap='Reds',    # Color map for expression
              title='Differentially Expressed Genes (RNA)',
              standard_scale='var',  # Scale by variable (gene)
              figsize=(11.5, 1.25),      # Try passing figsize directly
              show=False)
plt.savefig('Fig_4d.pdf', format='pdf', bbox_inches='tight')
plt.show()
plt.close()
print('Plot saved: Fig_4d.pdf')


In [ ]:
# Fig. 4e — Stacked bar plot of memory cluster frequencies
import pandas as pd
import matplotlib.pyplot as plt

# Load the Excel file
file_path = 'data/immgenT-CD8.xlsx'
df = pd.read_excel(file_path)

print(f"Loaded Excel file with {df.shape[0]} rows and {df.shape[1]} columns.")

# Define columns
prop_cols = ['CD8.E.prop', 'CD8.F.prop', 'CD8.G.prop', 'CD8.H.prop']
ncells_cols = ['CD8.E.ncells', 'CD8.F.ncells', 'CD8.G.ncells', 'CD8.H.ncells']

# === FILTERS (applied on original rows before averaging) ===
if 'target_cells' in df.columns:
    df = df[~df['target_cells'].str.contains('CD4\+ CD44\+', case=False, na=False)].copy()
    unwanted = ['Treg', 'CD4+ T', 'DN', 'DO11.10 KJ1-26', 'OT2', 'TCRBV-TCRGD', 'TFH',
                'Vd6b F4.22', 'Vg4 49.2', 'Vg6 1C10', 'Vg7 F2.67', 'thymocytes']
    mask = df['target_cells'].isin(unwanted) | df['target_cells'].str.contains('|'.join(unwanted), case=False, na=False)
    df = df[~mask].copy()
    print(f"After target_cells filters: {df.shape[0]} rows remain.")

# Exclude KO conditions
exclude_conditions = ['KbDbKO', 'KbDbQa1KO', 'KbDbQa1KO_MCMV']
exclude_mask = df['condition_detailed_organ'].str.contains('|'.join(exclude_conditions), case=False, na=False)
df = df[~exclude_mask].copy()
print(f"After excluding KbDb conditions: {df.shape[0]} rows remain.")

# Exclude baseline
if 'condition_detailed_simplified' in df.columns:
    df = df[df['condition_detailed_simplified'] != 'baseline'].copy()
    print(f"After excluding baseline: {df.shape[0]} rows remain.")

# === AGGREGATE REPLICATES (Group by condition_detailed_organ) ===
group_cols = ['condition_detailed_organ']

agg_dict = {col: 'mean' for col in prop_cols + ncells_cols}
if 'target_cells' in df.columns:
    agg_dict['target_cells'] = 'first'   # keep one value for reference

df_agg = df.groupby(group_cols, as_index=False).agg(agg_dict)

print(f"After aggregating replicates: {df_agg.shape[0]} unique conditions.")

# === FILTER on averaged ncells: ≥10 in at least one cluster ===
min_cells = 10
valid_mask = (df_agg['CD8.E.ncells'] >= min_cells) | \
             (df_agg['CD8.F.ncells'] >= min_cells) | \
             (df_agg['CD8.G.ncells'] >= min_cells) | \
             (df_agg['CD8.H.ncells'] >= min_cells)

df_filtered = df_agg[valid_mask].copy()
print(f"After ≥ {min_cells} cells (averaged) in at least one cluster: {df_filtered.shape[0]} conditions remain.")

# Calculate total proportion for ranking
df_filtered['total_prop'] = df_filtered[prop_cols].sum(axis=1)
df_filtered = df_filtered[df_filtered['total_prop'] > 0].copy()

# Select top 50 unique conditions
df_top50 = df_filtered.nlargest(50, 'total_prop')

print(f"Selected top 50 unique conditions by average combined proportion.")

# Prepare for plotting
freq_table = df_top50[prop_cols].copy()
freq_table.index = df_top50['condition_detailed_organ']
freq_table.columns = ['CD8.E', 'CD8.F', 'CD8.G', 'CD8.H']

# Plot using your pre-existing custom_colors
fig, ax = plt.subplots(figsize=(14, 8))
freq_table.plot(kind='bar', stacked=True, ax=ax, 
                color=[custom_colors[col] for col in freq_table.columns])

ax.set_xlabel('Condition (Detailed Organ)')
ax.set_ylabel('Percentage within each sample (%)')
ax.set_title('Top 50 Unique Conditions by CD8.E + CD8.F + CD8.G + CD8.H Proportion\n(Averaged across replicates • After all filters)')
ax.grid(False)
ax.legend(title='Cluster', bbox_to_anchor=(1, 1), loc='upper left')

plt.xticks(rotation=90, ha='center')
plt.tight_layout()

# Save
plt.savefig('Fig_4e.pdf', bbox_inches='tight')
plt.show()
plt.close()

print("Stacked bar plot saved: Fig_4e.pdf")

# Preview
print("\nTop 10 unique conditions:")
print(df_top50[['condition_detailed_organ', 'total_prop'] + ncells_cols + prop_cols].head(10))


In [ ]:
# Fig. 4f — Representative conditions enriched for memory clusters
conditions = ['B16_ACT','spleen_2mo', 'spleen_18mo', 'spleen_Foxp3mutant_D19', 'lung_MTB_6w']

# Gray background for non-highlighted cells
default_color = '#BAB0AC'  # Gray for cells not meeting the condition

# Multi-panel layout
fig, axes = plt.subplots(1, 5, figsize=(17.5, 4))  # 4 subplots horizontally, adjusted size

# Iterate over conditions and corresponding axes
for idx, (condition, ax) in enumerate(zip(conditions, axes)):
    # Create a new column for coloring based on conditions
    rna.obs['highlight_color'] = np.where(
        rna.obs['condition_detailed_organ'] == condition,
        rna.obs['cluster_annotation'],  # Use cluster_annotation for highlighted cells
        None  # None for cells that don't meet the condition
    )
    
    # Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 20, 1)
    
    # Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None,  # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=condition  # Add condition as subplot title
    )

# Adjust layout to prevent overlap
plt.tight_layout()

# Save and show the combined plot
plt.savefig("Fig_4f.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Combined plot saved: Fig_4f.pdf")


# Fig. 5


In [ ]:
# Fig. 5a — MDE plot with CD8.Q highlighted
# Gray background for non-highlighted cells
default_color = '#d3d3d3'  # Gray for cells not meeting the condition

# Create a new column for coloring based on conditions
rna.obs['highlight_color'] = np.where(
    rna.obs['cluster_annotation'] == 'CD8.Q',
    'CD8.Q',  # Use category name for highlighted cells
    'other'  # Use category name for non-highlighted cells
)

# Define palette for the categories in highlight_color
palette = {
    'CD8.Q': custom_colors.get('CD8.Q'),  # Use color from custom_colors or default to red
    'other': default_color  # Gray for non-highlighted cells
}

# Set up the plot
plt.figure(figsize=(5.5, 5))  # Single plot, moderate size
ax = plt.gca()

# Set dot parameters: larger for highlighted cells
sizes = np.where(rna.obs['highlight_color'] == 'CD8.Q', 5, 1)  # Optional: larger size for highlighted cells
alphas = np.where(rna.obs['highlight_color'] == 'CD8.Q', 0.5, 1)  # Optional: set opacity for highlighted cells

# Plot MDE
sc.pl.embedding(
    rna,
    basis="MDE_INCREMENTAL",
    color="highlight_color",
    palette=palette,  # Use the defined palette
    legend_loc="right margin",
    ax=ax,
    alpha=alphas,
    size=sizes,
    show=False,
)

# Save and show the plot
plt.tight_layout()
plt.savefig("Fig_5a.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Plot saved: Fig_5a.pdf")


In [ ]:
# Fig. 5b — MDE plot with LCMVarm memory cells from non-lymphoid tissues highlighted
# List of condition_detailed_organ to check
conditions = [
    'prostate_LCMVarm_D30', 'salivarygland_LCMVarm_D30', 'smallintestineIEL_LCMVarm_D30',
    'smallintestineLP_LCMVarm_D30', 'lung_LCMVarm_D30', 'prostate_LCMVarm_D60',
    'salivarygland_LCMVarm_D60', 'smallintestineIEL_LCMVarm_D60', 'smallintestineLP_LCMVarm_D60',
    'lung_LCMVarm_D60'
]

# Define custom palette for conditions
custom_colors = {
    'prostate_LCMVarm_D30': '#1f77b4',  # Blue
    'salivarygland_LCMVarm_D30': '#ff7f0e',  # Orange
    'smallintestineIEL_LCMVarm_D30': '#2ca02c',  # Green
    'smallintestineLP_LCMVarm_D30': '#d62728',  # Red
    'lung_LCMVarm_D30': '#9467bd',  # Purple
    'prostate_LCMVarm_D60': '#8c564b',  # Brown
    'salivarygland_LCMVarm_D60': '#e377c2',  # Pink
    'smallintestineIEL_LCMVarm_D60': '#7f7f7f',  # Gray
    'smallintestineLP_LCMVarm_D60': '#bcbd22',  # Olive
    'lung_LCMVarm_D60': '#17becf'  # Cyan
}

# Filter for specified conditions
subset_conditions = rna[
    (rna.obs['condition_detailed_organ'].isin(conditions)) & 
    (rna.obs['Ag_spe_v2'] == 'P14')
].copy()

# Create figure and axis
fig, ax = plt.subplots(figsize=(6.5, 4))

# Plot all cells in gray as background
sc.pl.embedding(
    rna,
    basis='MDE_INCREMENTAL',
    color='condition_detailed_organ',
    palette=['#d3d3d3'],  # Gray color for all cells
    size=1,  # Smaller point size for background
    show=False,
    ax=ax,
    legend_loc=None  # Remove legend
)

# Overlay cells from specified conditions, colored by condition_detailed_organ
sc.pl.embedding(
    subset_conditions,
    basis='MDE_INCREMENTAL',
    color='condition_detailed_organ',
    groups=conditions,  # Ensure only specified conditions are colored
    palette=custom_colors,  # Apply custom palette
    title='Highlighted Conditions Colored by Condition_Detailed_Organ',
    legend_loc='right margin',
    size=25,  # Larger point size for foreground
    show=False,
    ax=ax
)

# Save and show the plot
plt.tight_layout()
plt.savefig("Fig_5b.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Plot saved: Fig_5b.pdf")


In [ ]:
# Fig. 5c — Stacked bar plot of cell counts by condition
# List of condition_detailed_organ to check
conditions = [
'prostate_LCMVarm_D30', 'salivarygland_LCMVarm_D30', 'smallintestineIEL_LCMVarm_D30',
'smallintestineLP_LCMVarm_D30', 'lung_LCMVarm_D30', 'prostate_LCMVarm_D60',
'salivarygland_LCMVarm_D60', 'smallintestineIEL_LCMVarm_D60', 'smallintestineLP_LCMVarm_D60',
'lung_LCMVarm_D60'
]

# Filter for specified conditions
subset_conditions = rna[
    (rna.obs['condition_detailed_organ'].isin(conditions))
].copy()

# Get the FULL list of cluster_annotation that exist in your dataset (not just the filtered subset)
all_annotations = rna.obs['cluster_annotation'].cat.categories.tolist()

# Calculate cell counts and reindex to force ALL cluster_annotation (with 0s where missing)
counts = (subset_conditions.obs
          .groupby(['cluster_annotation', 'condition_detailed_organ'])
          .size()
          .unstack(fill_value=0)
          .reindex(index=all_annotations, fill_value=0))   # <-- this line forces all levels

# Define custom colors for conditions
custom_colors = {
'prostate_LCMVarm_D30': '#1f77b4',
'salivarygland_LCMVarm_D30': '#ff7f0e',
'smallintestineIEL_LCMVarm_D30': '#2ca02c',
'smallintestineLP_LCMVarm_D30': '#d62728',
'lung_LCMVarm_D30': '#9467bd',
'prostate_LCMVarm_D60': '#8c564b',
'salivarygland_LCMVarm_D60': '#e377c2',
'smallintestineIEL_LCMVarm_D60': '#7f7f7f',
'smallintestineLP_LCMVarm_D60': '#bcbd22',
'lung_LCMVarm_D60': '#17becf'
}

# Create figure and axis
fig, ax = plt.subplots(figsize=(10, 5))

# Plot stacked bar plot
counts.plot(kind='bar', stacked=True, ax=ax, 
            color=[custom_colors[cond] for cond in counts.columns])

# Customize plot
ax.set_xlabel('Annotation Level 2 Cluster')
ax.set_ylabel('Number of Cells')
ax.set_title('Cell Counts by Condition (Detailed Organ) Across Clusters')
ax.legend(title='Condition (Detailed Organ)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)

# Save and show the plot
plt.tight_layout()
plt.savefig("Fig_5c.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Plot saved: Fig_5c.pdf")


In [ ]:
# Fig. 5d — MDE plot with IGT95 and IGT96 cells from lung_flu highlighted
color_palette = {
    'OT1': "#8f0f0f",  # Deep red
    'Endogenous': "#65d8d8",    # Cyan
    # Add other categories in condition_detailed if necessary
}

# Filter for IGT == IGT95, IGT96 and condition_detailed_organ == 'lung_flu'
subset_conditions = rna[
    (rna.obs['IGT'].isin(['IGT95', 'IGT96'])) & 
    (rna.obs['condition_detailed_organ'].str.contains('lung_flu', na=False))
].copy()

# Create figure and axis
fig, ax = plt.subplots(figsize=(5.5, 4.5))

# Plot all cells in gray as background
sc.pl.embedding(
    rna,
    basis='MDE_INCREMENTAL',
    color='condition_broad',
    palette=['#d3d3d3'],  # Gray color for all cells
    size=1,  # Smaller point size for background
    show=False,
    ax=ax,
    legend_loc=None  # Remove legend
)

# Overlay cells from specified conditions
sc.pl.embedding(
    subset_conditions,
    basis='MDE_INCREMENTAL',
    color='Ag_spe_v1',  # Color by condition_detailed
    groups=['Endogenous', 'OT1'],  # Ensure only specified TG cells are colored
    palette=color_palette,  # Use defined color palette
    title='Highlighted Conditions (lung_flu)',
    legend_loc='right margin',
    size=10,  # Larger point size for foreground
    show=False,
    ax=ax
)

# Save and show the plot
plt.tight_layout()
plt.savefig("Fig_5d.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Plot saved: Fig_5d.pdf")


In [ ]:
# Fig. 5h — Balloon plot of CD8.Q across tissues
import os
import scanpy as sc
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

# Custom color for CD8.Q
custom_colors = {
    'CD8.Q': '#76EE00',
}

def generate_balloon_plot_CD8_Q_IGT38_by_organ(
    main_signature_file,
    mdata,
    cluster='CD8.Q',
    pval_threshold=0.05,
    log2fc_threshold=0.5,
    top_n_genes=15,
    extra_genes=None,
    min_cells_per_organ=10,
    figures_dir='figures'
):
    """
    Balloon plot for CD8.Q:
    - Non-spleen organs: CD8.Q cells from IGT38/IGT40 only, with >10 cells
    - Spleen (healthy): ONLY healthy spleen CD8.Q cells
    - Spleen (non-healthy): CD8.Q cells from spleen in IGT38/IGT40 (if >10 cells)
    - X-axis gene order:
        - Top 15 upregulated (highest → lowest log2FC)
        - Top 15 downregulated (15th → 1st most significant)
        - Extra genes are fully integrated and sorted together with the top genes by log2FC
    """
    adata_all = mdata['RNA']
    os.makedirs(figures_dir, exist_ok=True)

    # ==============================================================
    # 1. Non-spleen organs: CD8.Q + IGT38/IGT40
    # ==============================================================
    cluster_mask = adata_all.obs['cluster_annotation'] == cluster
    igt_mask = adata_all.obs['IGT'].isin(['IGT38', 'IGT40'])
    non_spleen_mask = adata_all.obs['organ'] != 'spleen'

    adata_nonspleen = adata_all[cluster_mask & igt_mask & non_spleen_mask].copy()

    # ==============================================================
    # 2. Spleen non-healthy (IGT38/IGT40)
    # ==============================================================
    spleen_mask = adata_all.obs['organ'] == 'spleen'
    adata_spleen_nonhealthy = adata_all[cluster_mask & igt_mask & spleen_mask].copy()

    # ==============================================================
    # 3. Spleen healthy
    # ==============================================================
    healthy_mask = adata_all.obs['condition_broad'] == 'healthy'
    adata_spleen_healthy = adata_all[cluster_mask & spleen_mask & healthy_mask].copy()

    # ==============================================================
    # 4. Combine all parts
    # ==============================================================
    adata_list = []
    if adata_nonspleen.n_obs > 0:
        adata_list.append(adata_nonspleen)
    if adata_spleen_nonhealthy.n_obs > 0:
        adata_list.append(adata_spleen_nonhealthy)
    if adata_spleen_healthy.n_obs > 0:
        adata_list.append(adata_spleen_healthy)

    if not adata_list:
        print("No CD8.Q cells found matching the criteria.")
        return

    adata_plot = sc.concat(adata_list, label="source", keys=["nonspleen", "spleen_nonhealthy", "spleen_healthy"])

    # Use normalized scaled data
    if 'scaled' in adata_plot.layers:
        adata_plot.X = adata_plot.layers['scaled']
    elif 'log_norm' in adata_plot.layers:
        adata_plot.X = adata_plot.layers['log_norm']

    # ==============================================================
    # 5. Create display labels for spleen
    # ==============================================================
    adata_plot.obs['organ_plot'] = adata_plot.obs['organ'].astype(str)

    spleen_healthy_idx = (adata_plot.obs['organ'] == 'spleen') & (adata_plot.obs['source'] == 'spleen_healthy')
    spleen_nonhealthy_idx = (adata_plot.obs['organ'] == 'spleen') & (adata_plot.obs['source'] == 'spleen_nonhealthy')

    adata_plot.obs.loc[spleen_healthy_idx, 'organ_plot'] = 'spleen (healthy)'
    adata_plot.obs.loc[spleen_nonhealthy_idx, 'organ_plot'] = 'spleen (IGT38/40)'

    # ==============================================================
    # 6. Count and filter organs
    # ==============================================================
    organ_counts = adata_plot.obs['organ_plot'].value_counts()

    valid_organs = organ_counts[
        (organ_counts > min_cells_per_organ) |
        (organ_counts.index.str.contains('spleen'))
    ].index.tolist()

    if not valid_organs:
        print("No organs passed filtering.")
        return

    adata_plot = adata_plot[adata_plot.obs['organ_plot'].isin(valid_organs)].copy()

    ordered_organs = organ_counts.loc[valid_organs].sort_values(ascending=False).index.tolist()

    print(f"Plotting {len(ordered_organs)} organ entries:")
    for org in ordered_organs:
        count = organ_counts[org]
        source = "healthy" if 'healthy' in org else "IGT38/40"
        marker = " (forced)" if 'spleen' in org and count <= min_cells_per_organ else ""
        print(f"  - {org}: {count} cells ({source}{marker})")

    # ==============================================================
    # 7. Load DE results and build gene list
    # ==============================================================
    signature_df = pd.read_csv(main_signature_file, sep='\t', encoding='utf-8')
    
    log2fc_col = f'log2FC_{cluster}_vs_All'
    pval_col = f'adj.P.Val_{cluster}_vs_All'
    
    if log2fc_col not in signature_df.columns or pval_col not in signature_df.columns:
        raise ValueError(f"Columns for {cluster} not found in signature file.")

    plot_df = signature_df[['SYMBOL', log2fc_col, pval_col]].copy()
    plot_df['gene'] = plot_df['SYMBOL']
    plot_df['log2FC'] = plot_df[log2fc_col]

    significant = plot_df[
        (plot_df[pval_col] < pval_threshold) &
        (abs(plot_df['log2FC']) > log2fc_threshold)
    ].copy()

    if significant.empty:
        print(f"No significant genes for {cluster} with current thresholds.")
        return

    # Top 15 upregulated: highest → lowest
    top_up = significant[significant['log2FC'] > 0].sort_values('log2FC', ascending=False).head(top_n_genes)

    # Top 15 downregulated: select most significant (lowest log2FC), then reverse for plotting
    top_down_significant = significant[significant['log2FC'] < 0].sort_values('log2FC', ascending=True).head(top_n_genes)
    top_down_plot = top_down_significant.sort_values('log2FC', ascending=False)  # 15th → 1st

    # Start with the ordered top genes
    gene_order_df = pd.concat([top_up, top_down_plot])

    # Add extra genes (with their real log2FC if available)
    if extra_genes is not None:
        extra_df = plot_df[plot_df['gene'].isin(extra_genes)].copy()
        if not extra_df.empty:
            gene_order_df = pd.concat([gene_order_df, extra_df])

    # Remove duplicates and sort ALL together by log2FC descending
    gene_order_df = gene_order_df.drop_duplicates(subset='gene')
    gene_order_df = gene_order_df.sort_values('log2FC', ascending=False)

    gene_order = gene_order_df['gene'].tolist()

    # Keep only present genes
    gene_order = [g for g in gene_order if g in adata_plot.var_names]

    if not gene_order:
        print("None of the selected genes are present.")
        return

    print(f"Plotting {len(gene_order)} genes (top {top_n_genes} up/down + extras, ALL sorted by log2FC descending).")

    # Custom colormap
    cluster_color = custom_colors.get(cluster, '#76EE00')
    custom_cmap = LinearSegmentedColormap.from_list(
        f'white_to_{cluster}', ['#FFFFFF', cluster_color], N=256
    )

    # ==============================================================
    # 8. Balloon plot
    # ==============================================================
    sc.pl.dotplot(
        adata_plot,
        var_names=gene_order,
        groupby='organ_plot',
        categories_order=ordered_organs,
        cmap='Reds',
        standard_scale='var',
        dot_min=0.05,
        dot_max=0.8,
        figsize=(12, 3),
        save=f'Fig_5h.pdf'
    )

    print(f"Balloon plot saved to: {figures_dir}/Fig_5h.pdf")


# ==========================
# Usage
# ==========================

main_signature_file = 'data/ttlist_OneVsAll.txt'

generate_balloon_plot_CD8_Q_IGT38_by_organ(
    main_signature_file=main_signature_file,
    mdata=mdata,
    top_n_genes=15,
    extra_genes=['Itgae', 'Itga1'],
    min_cells_per_organ=10,
    figures_dir='figures'
)

In [ ]:
# Fig. 5j - GP analysis across tissues within CD8.Q
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Assuming mdata is your MuData object containing the AnnData
# Load matrices (adjust paths as needed)
F_df = pd.read_csv("data/gene_factor_matrix.txt", sep="\t", index_col=0)
L_df = pd.read_csv("data/cell_factor_matrix.txt", sep="\t", index_col=0)

# Convert to NumPy arrays for efficient computation
F_np = F_df.to_numpy()
L_np = L_df.to_numpy()

# Preserve names
factor_names = L_df.columns  # e.g., F1–F200
gene_names = F_df.index

# Verify shapes
print("F_np shape:", F_np.shape)  # Expected: (19805, 200)
print("L_np shape:", L_np.shape)  # Expected: (682953, 200)

# Access observations from the AnnData
mdata_obs = mdata['RNA'].obs

# Check for cell ID column
if 'cellID' in mdata_obs.columns:
    print("Using 'cellID' from mdata_obs")
    cell_id_col = 'cellID'
else:
    print("Using 'IGT_cellID' from mdata_obs")
    cell_id_col = 'IGT_cellID'

# Subset to IGT == 'IGT38' and exclude condition_broad == 'healthy'
igt38_obs = mdata_obs[(mdata_obs['IGT'] == 'IGT38') & (mdata_obs['condition_broad'] != 'healthy')]

# Get unique detailed organs and filter by cell count (>= 50 cells)
organ_cell_counts = igt38_obs['condition_detailed_organ'].value_counts()
unique_organs = organ_cell_counts[organ_cell_counts >= 50].index.tolist()
print(f"Found {len(unique_organs)} unique detailed organs in IGT38 (non-healthy) with >= 50 cells: {unique_organs}")

# Optional: Sort organs if needed (e.g., alphabetically)
unique_organs = sorted(unique_organs)

# Function to compute log2FC for a group vs others
def compute_factor_log2fc(L_np, group1, group2, factor_names):
    group1 = [g for g in group1 if g in L_df.index]
    group2 = [g for g in group2 if g in L_df.index]
    if not group1 or not group2:
        return None
    group1_idx = [L_df.index.get_loc(g) for g in group1]
    group2_idx = [L_df.index.get_loc(g) for g in group2]
    loadings_group1 = L_np[group1_idx].mean(axis=0)
    loadings_group2 = L_np[group2_idx].mean(axis=0)
    fc_loadings = loadings_group1 - loadings_group2
    return pd.DataFrame({
        'SYMBOL': factor_names,
        'log2FC': fc_loadings / np.log(2)
    })

# Identify top factors for each detailed organ within IGT38 (non-healthy)
top_factors_per_organ = {}
n_top = 1  # Top 1 factor per organ; adjust as needed
for organ in unique_organs:
    group1 = igt38_obs[igt38_obs['condition_detailed_organ'] == organ][cell_id_col].tolist()
    group2 = igt38_obs[igt38_obs['condition_detailed_organ'] != organ][cell_id_col].tolist()
    if group1 and group2:
        log2fc_df = compute_factor_log2fc(L_np, group1, group2, factor_names)
        if log2fc_df is not None:
            top = log2fc_df.sort_values(by='log2FC', ascending=False).head(n_top)
            top_factors_per_organ[organ] = top['SYMBOL'].tolist()
            print(f"Top factor(s) for {organ}: {top['SYMBOL'].tolist()}")
    else:
        print(f"Skipping {organ}: group1 has {len(group1)} cells, group2 has {len(group2)} cells")

# Collect all unique top factors across organs
all_top_factors = list(dict.fromkeys([f for factors in top_factors_per_organ.values() for f in factors]))
print(f"All selected top factors: {all_top_factors}")

# Compute mean loadings for each organ and top factor
organ_loadings = []
for organ in unique_organs:
    group = igt38_obs[igt38_obs['condition_detailed_organ'] == organ][cell_id_col].tolist()
    if group:
        group_idx = [L_df.index.get_loc(g) for g in group if g in L_df.index]
        if group_idx:
            loadings = L_np[group_idx].mean(axis=0)
            for i, factor in enumerate(factor_names):
                if factor in all_top_factors:
                    organ_loadings.append({
                        'organ': organ,
                        'factor': factor,
                        'mean_loading': loadings[i],
                        'num_cells': len(group)
                    })
        else:
            print(f"Skipping {organ}: no valid cells")
    else:
        print(f"Skipping {organ}: no valid cells")

# Convert to DataFrame
loadings_df = pd.DataFrame(organ_loadings)

# Pivot for visualization (organs as rows, factors as columns)
pivot_df = loadings_df.pivot(index='organ', columns='factor', values='mean_loading').fillna(0)

# Reindex to sorted organs
pivot_df = pivot_df.reindex(unique_organs)

# Print the pivot table
print(pivot_df)


In [ ]:
# Fig. 5j - GP analysis across tissues within CD8.Q
import matplotlib.pyplot as plt
import numpy as np

# Assuming pivot_df, all_top_factors, and unique_organs are defined from the previous code
# pivot_df: DataFrame with organs as index and factors as columns, filtered to organs with >= 50 cells
# all_top_factors: List of unique top factors across organs
# unique_organs: List of unique condition_detailed_organ values with >= 50 cells
# No target_clusters equivalent; we won't highlight specific organs unless specified

# Create a figure with subplots (one subplot per factor)
fig, axes = plt.subplots(nrows=len(all_top_factors), ncols=1, figsize=(3.5, 5 * len(all_top_factors)))

# Ensure axes is a list even if there's only one factor
if len(all_top_factors) == 1:
    axes = [axes]

# Colors for factors (distinct and theme-compatible)
colors = ["#a8fffb", "#f6402f", "#1f78b4", "#dcda97", "#8f1fb4"][:len(all_top_factors)]  # Up to 5 factors, add more colors if needed

# Plot a bar plot for each factor
for i, (factor, ax) in enumerate(zip(all_top_factors, axes)):
    # X positions for organs
    x = np.arange(len(unique_organs))
    # Plot bars for the current factor
    bars = ax.bar(x, pivot_df[factor], color=colors[i], label=factor)
    # Customize each subplot
    ax.set_xlabel('Detailed Organ')
    ax.set_ylabel('Mean Factor Loading')
    ax.set_title(f'Gene Program Activation for {factor} (IGT38, Non-Healthy Conditions)')
    ax.set_xticks(x)
    ax.set_ylim(0, 2)  # Adjust ylim if needed based on data
    ax.set_xticklabels(unique_organs, rotation=90, ha='center')  # Use filtered organ names, adjust rotation for readability
    ax.grid(False)  # Explicitly disable any grid
    # Reduce gaps before first bar and after last bar
    ax.set_xlim(-0.5, len(unique_organs) - 0.5)

# Adjust layout to prevent overlap
plt.tight_layout()

# Save as PDF
plt.savefig('Fig_5j.pdf', dpi=300, bbox_inches='tight', format='pdf')

# Show the plot
plt.show()


In [ ]:
# Fig. 5j - GP analysis across tissues within CD8.Q
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import scipy.sparse as sparse

# Ensure we use your preferred normalized data
mdata['RNA'].X = mdata['RNA'].layers['scaled']

# Recompute top factors (self-contained block)
def compute_factor_log2fc(L_np, group1, group2, factor_names):
    group1 = [g for g in group1 if g in L_df.index]
    group2 = [g for g in group2 if g in L_df.index]
    if not group1 or not group2:
        return None
    group1_idx = [L_df.index.get_loc(g) for g in group1]
    group2_idx = [L_df.index.get_loc(g) for g in group2]
    loadings_group1 = L_np[group1_idx].mean(axis=0)
    loadings_group2 = L_np[group2_idx].mean(axis=0)
    fc_loadings = loadings_group1 - loadings_group2
    return pd.DataFrame({
        'SYMBOL': factor_names,
        'log2FC': fc_loadings / np.log(2)
    })

# Filter for IGT38 non-healthy (assuming already defined as igt38_obs)
# If not, define it here for safety
if 'igt38_obs' not in globals():
    igt38_obs = mdata_obs[(mdata_obs['IGT'] == 'IGT38') & (mdata_obs['condition_broad'] != 'healthy')]

unique_organs = sorted(igt38_obs['condition_detailed_organ'].unique())
print(f"Organs in IGT38 (non-healthy): {unique_organs}")

# Identify top factors for each detailed organ within IGT38 (non-healthy)
top_factors_per_organ = {}
n_top = 1  # Top 1 factor per organ
for organ in unique_organs:
    group1 = igt38_obs[igt38_obs['condition_detailed_organ'] == organ][cell_id_col].tolist()
    group2 = igt38_obs[igt38_obs['condition_detailed_organ'] != organ][cell_id_col].tolist()
    if group1 and group2:
        log2fc_df = compute_factor_log2fc(L_np, group1, group2, factor_names)
        if log2fc_df is not None:
            top = log2fc_df.sort_values(by='log2FC', ascending=False).head(n_top)
            top_factors_per_organ[organ] = top['SYMBOL'].tolist()
            print(f"Top factor(s) for {organ}: {top['SYMBOL'].tolist()}")
    else:
        print(f"Skipping {organ}: insufficient cells")

# Collect all unique top factors
all_top_factors = list(dict.fromkeys([f for factors in top_factors_per_organ.values() if factors for f in factors]))
print(f"Selected factors: {all_top_factors}")

# Verify we have 5 factors
if len(all_top_factors) != 5:
    print(f"Warning: Found {len(all_top_factors)} factors instead of 5: {all_top_factors}")

# Extract top genes for each selected factor
n_top_genes = 10
factor_genes = {factor: [] for factor in all_top_factors}
for factor in all_top_factors:
    factor_idx = list(factor_names).index(factor)
    gene_loadings = pd.Series(F_np[:, factor_idx], index=gene_names)
    top = gene_loadings.abs().sort_values(ascending=False).head(n_top_genes)
    factor_genes[factor] = top.index.tolist()
    print(f"Top {n_top_genes} genes for {factor}: {top.index.tolist()}")

# === REAL average expression per organ (trusted method) ===
# Create a boolean mask over the FULL mdata['RNA'] object
full_mask = mdata_obs.index.isin(igt38_obs.index) & mdata_obs['condition_detailed_organ'].isin(unique_organs)

# Subset expression matrix using boolean mask (safe for AnnData/MuData)
subset_X = mdata['RNA'].X[full_mask]

# Subset obs accordingly
subset_obs = mdata_obs[full_mask]

# Get organ labels
organ_labels = subset_obs['condition_detailed_organ'].values

unique_labels, inverse = np.unique(organ_labels, return_inverse=True)
n_genes = subset_X.shape[1]

mean_expr_matrix = np.empty((len(unique_labels), n_genes))
for i, label in enumerate(unique_labels):
    mask_i = (inverse == i)
    if sparse.issparse(subset_X):
        mean_expr_matrix[i] = np.ravel(subset_X[mask_i].mean(axis=0))
    else:
        mean_expr_matrix[i] = subset_X[mask_i].mean(axis=0)

mean_expr_pivot = pd.DataFrame(
    mean_expr_matrix,
    index=unique_labels,
    columns=mdata['RNA'].var_names
).T  # genes x organs
mean_expr_pivot = mean_expr_pivot[unique_organs]  # Order correctly

# === Build expression DataFrame using REAL expression ===
gene_expression = []
for organ in unique_organs:
    for factor in all_top_factors:
        for gene in factor_genes[factor]:
            if gene in mean_expr_pivot.index:
                gene_expression.append({
                    'organ': organ,
                    'gene': gene,
                    'mean_expr': mean_expr_pivot.loc[gene, organ],
                    'factor': factor
                })

expr_df = pd.DataFrame(gene_expression)

# Create custom colormaps for 5 factors
colors = [
    ['lightgray', 'white', '#a8fffb'],  # light blue
    ['lightgray', 'white', '#f6402f'],  # red
    ['lightgray', 'white', '#8f1fb4'],  # dark blue
    ['lightgray', 'white', '#1f78b4'],  # yellow
    ['lightgray', 'white', '#dcda97']   # purple
]
cmaps = [LinearSegmentedColormap.from_list(f'custom_{i+1}', colors[i]) for i in range(5)]

# Set z-score range
zscore_min = -2.0
zscore_max = 2.0

# Create heatmaps for each factor
for i, factor in enumerate(all_top_factors):
    factor_expr_df = expr_df[expr_df['factor'] == factor]
    pivot_df = factor_expr_df.pivot(index='gene', columns='organ', values='mean_expr').fillna(0)
    pivot_df = pivot_df[unique_organs]

    # Z-score normalization per gene
    pivot_df = pivot_df.apply(lambda x: (x - x.mean()) / x.std() if x.std() != 0 else x, axis=1)

    # Order genes by max abs z-score
    pivot_df = pivot_df.loc[pivot_df.abs().max(axis=1).sort_values(ascending=False).index]

    cmap = cmaps[i]

    fig, ax = plt.subplots(figsize=(5.25, 6))
    heatmap = ax.imshow(pivot_df.values, cmap=cmap, aspect='auto',
                        vmin=zscore_min, vmax=zscore_max, interpolation='nearest')
    ax.invert_yaxis()

    ax.set_xticks(np.arange(len(unique_organs)))
    ax.set_yticks(np.arange(len(pivot_df.index)))
    ax.set_xticklabels(unique_organs, rotation=45, ha='right')
    ax.set_yticklabels(pivot_df.index)
    ax.set_xlabel('Detailed Organ')
    ax.set_ylabel('Genes')
    ax.set_title(f'Differentially Expressed Genes for {factor} (IGT38, Non-Healthy Conditions)')
    ax.grid(False)

    plt.colorbar(heatmap, label='Z-score Normalized Expression')
    plt.tight_layout()
    plt.savefig(f'Fig_5j_v2_{factor}.pdf', format='pdf', bbox_inches='tight', dpi=300)
    plt.show()
    plt.close()
    print(f'Plot saved: Fig_5j_v2_{factor}.pdf')

# Print results
print("\nZ-score normalized gene expression for all genes:")
pivot_df = expr_df.groupby(['gene', 'organ'])['mean_expr'].first().unstack().fillna(0)[unique_organs]
pivot_df = pivot_df.apply(lambda x: (x - x.mean()) / x.std() if x.std() != 0 else x, axis=1)
print(pivot_df)


# Fig. 6

In [ ]:
# Fig. 6a — MDE plots
cd44_neg = {'CD8.A', 'CD8.B', 'CD8.D', 'CD8.T', 'CD8.wM', 'CD8.wX'}

rna.obs['CD44_status'] = 'CD44+'
rna.obs.loc[rna.obs['cluster_annotation'].isin(cd44_neg), 'CD44_status'] = 'CD44-'

# Quick check
print(rna.obs['CD44_status'].value_counts(dropna=False))


In [ ]:
# Fig. 6a — MDE plots
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

# Gray background for non-highlighted cells
default_color = '#BAB0AC'

# List of the 3 conditions to highlight (one panel each)
conditions = [
    {
        'name': 'autoimmunity',
        'mask': (
            (rna.obs['condition_broad'].str.lower() == 'autoimmunity') &
            (rna.obs['CD44_status'] == 'CD44+')
        ),
        'title': 'Autoimmunity'
    },
    {
        'name': 'tumor',
        'mask': (
            (rna.obs['condition_broad'].str.lower() == 'tumor') &
            (rna.obs['condition_detailed_organ'].isin(['B16_ACT', 'pancreas_PDAC', 'lung_KP'])) &
            (rna.obs['Ag_spe_v1'] == 'Endogenous') &
            (rna.obs['CD44_status'] == 'CD44+')
        ),
        'title': 'Tumor (B16_ACT, pancreas_PDAC, lung_KP)'
    },
    {
        'name': 'virus_specific',
        'mask': (
            (rna.obs['condition_broad'].str.lower() == 'virus') &
            (rna.obs['condition_detailed_simplified'].isin(['LCMVcl13', 'MNVCR6', 'MCMV'])) &
            (rna.obs['CD44_status'] == 'CD44+')
        ),
        'title': 'Virus (LCMVcl13, MNVCR6, MCMV)'
    }
]

# Create a figure with 3 subplots side-by-side
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for idx, cond in enumerate(conditions):
    # Create the highlight mask for this condition
    highlight_mask = cond['mask']
    
    # Color by cluster_annotation only for highlighted cells, else np.nan → will use na_color
    rna.obs['highlight_color'] = np.where(
        highlight_mask,
        rna.obs['cluster_annotation'],
        np.nan  # ← Changed from None to np.nan
    )
    
    # Make highlighted cells visibly larger
    sizes = np.where(highlight_mask, 20, 1)
    
    ax = axes[idx]
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        na_in_legend=False,
        legend_loc="right margin" if idx == 2 else "none",
        ax=ax,
        size=sizes,
        title=cond['title'],
        show=False,
    )

plt.tight_layout()
plt.savefig("Fig_6a.pdf", bbox_inches="tight")
plt.show()
plt.close()

print("Plot saved: Fig_6a.pdf")


In [ ]:
# Fig. 6d — Bar plot showing cluster distribution
import pandas as pd
import matplotlib.pyplot as plt

# ==============================================================
# Settings
# ==============================================================
clusters = ['CD8.Q', 'CD8.R', 'CD8.S', 'CD8.C']

# Exclusions
exclude_conditions = ['KbDbKO', 'KbDbQa1KO', 'KbDbQa1KO_MCMV', 'thymus']

# Load data
file_path = 'data/immgenT-CD8.xlsx'
df = pd.read_excel(file_path)

# === FILTERS ===
if 'target_cells' in df.columns:
    df = df[~df['target_cells'].str.contains('CD4\+ CD44\+', case=False, na=False)].copy()
    unwanted = ['Treg', 'CD4+ T', 'DN', 'DO11.10 KJ1-26', 'OT2', 'TCRBV-TCRGD', 'TFH',
                'Vd6b F4.22', 'Vg4 49.2', 'Vg6 1C10', 'Vg7 F2.67', 'thymocytes']
    mask = df['target_cells'].isin(unwanted) | df['target_cells'].str.contains('|'.join(unwanted), case=False, na=False)
    df = df[~mask].copy()

df = df[~df['condition_detailed_organ'].str.contains('|'.join(exclude_conditions), case=False, na=False)].copy()

if 'condition_detailed_simplified' in df.columns:
    df = df[df['condition_detailed_simplified'] != 'baseline'].copy()

# Average replicates
agg_dict = {f'{cl}.prop': 'mean' for cl in clusters}
agg_dict.update({f'{cl}.ncells': 'mean' for cl in clusters})
if 'condition_broad' in df.columns:
    agg_dict['condition_broad'] = 'first'

df_agg = df.groupby('condition_detailed_organ', as_index=False).agg(agg_dict)

# Cell count filter
min_cells = 10
valid_mask = df_agg[[f'{cl}.ncells' for cl in clusters]].ge(min_cells).any(axis=1)
df_agg = df_agg[valid_mask].copy()

print(f"Final unique conditions: {df_agg.shape[0]}")

# ==============================================================
# 1. Top 15 Tumor/Cancer
# ==============================================================
if 'condition_broad' in df_agg.columns:
    tumor_mask = df_agg['condition_broad'].str.contains('tumor|cancer', case=False, na=False)
else:
    tumor_mask = df_agg['condition_detailed_organ'].str.contains('tumor|cancer', case=False, na=False)

tumor_df = df_agg[tumor_mask].copy()
tumor_df['total_prop'] = tumor_df[[f'{cl}.prop' for cl in clusters]].sum(axis=1)
top15_tumor = tumor_df.nlargest(15, 'total_prop')['condition_detailed_organ'].tolist()

# ==============================================================
# 2. Sequential Top 15 per cluster (no overlap)
# ==============================================================
remaining = df_agg[~df_agg['condition_detailed_organ'].isin(top15_tumor)].copy()
top15_per_cluster = {}

for cl in clusters:
    if len(remaining) == 0:
        top15_per_cluster[cl] = []
        continue
    top15 = remaining.nlargest(15, f'{cl}.prop')['condition_detailed_organ'].tolist()
    top15_per_cluster[cl] = top15
    # Remove assigned conditions from remaining
    remaining = remaining[~remaining['condition_detailed_organ'].isin(top15)]

# ==============================================================
# Final ordered list
# ==============================================================
ordered_conditions = (
    top15_tumor +
    top15_per_cluster['CD8.Q'] +
    top15_per_cluster['CD8.R'] +
    top15_per_cluster['CD8.S'] +
    top15_per_cluster['CD8.C']
)

# Section boundaries
n_tumor = len(top15_tumor)
n_Q = len(top15_per_cluster['CD8.Q'])
n_R = len(top15_per_cluster['CD8.R'])
n_S = len(top15_per_cluster['CD8.S'])
n_C = len(top15_per_cluster['CD8.C'])

tumor_end = n_tumor - 0.5
Q_start = tumor_end
Q_end   = Q_start + n_Q
R_start = Q_end
R_end   = R_start + n_R
S_start = R_end
S_end   = S_start + n_S
C_start = S_end
C_end   = len(ordered_conditions) - 0.5

# ==============================================================
# PLOTTING
# ==============================================================
for cluster in clusters:
    color = custom_colors.get(cluster, '#808080')
    
    prop_dict = dict(zip(df_agg['condition_detailed_organ'], df_agg[f'{cluster}.prop']))
    percentages = [prop_dict.get(cond, 0) * 100 for cond in ordered_conditions]

    plt.figure(figsize=(15, 4.5))
    plt.bar(range(len(ordered_conditions)), percentages,
            color=color, edgecolor='black', linewidth=0.4)

    # Background shading
    plt.axvspan(-0.5, tumor_end, color="#dedede", alpha=0.12)
    plt.axvspan(Q_start, Q_end, color=color, alpha=0.09)
    plt.axvspan(R_start, R_end, color=color, alpha=0.09)
    plt.axvspan(S_start, S_end, color=color, alpha=0.09)
    plt.axvspan(C_start, C_end, color=color, alpha=0.09)

    # Separators
    plt.axvline(tumor_end, color='black', linewidth=1.0, linestyle='--')
    plt.axvline(Q_end, color='black', linewidth=1.0, linestyle='--')
    plt.axvline(R_end, color='black', linewidth=1.0, linestyle='--')
    plt.axvline(S_end, color='black', linewidth=1.0, linestyle='--')

    plt.ylabel('Percentage within sample (%)', fontsize=9)
    plt.title(f'{cluster} Distribution\n(Top 15 Tumor/Cancer + Sequential Top 15 per Cluster)', 
              fontsize=11, pad=20)
    plt.xticks(range(len(ordered_conditions)), ordered_conditions,
               rotation=90, ha='center', fontsize=7)
    plt.margins(x=0.005)   # Very tight padding (recommended to start with)
    plt.ylim(0, 100)
    plt.grid(False, axis='y', linestyle=':')
    plt.tight_layout()
    plt.grid(False)
    
    plt.savefig(f"Fig_6d_{cluster}.pdf", format='pdf', bbox_inches='tight', dpi=300)
    plt.show()

    print(f"Saved: Fig_6d_{cluster}.pdf")
    print(f" → Tumor: {n_tumor} | Q: {n_Q} | R: {n_R} | S: {n_S} | C: {n_C}\n")


In [ ]:
# Fig. 6e-g — FC versus FC comparisons among exhausted-like clusters
import pandas as pd
import matplotlib.pyplot as plt
from adjustText import adjust_text
import numpy as np
from scipy.stats import linregress

# Load the tab-separated text file
de_data = pd.read_csv("data/ttlist_OneVsAll.txt", sep="\t")

def create_fc_plot(x_col, y_col, x_label, y_label,
                   title, pdf_filename, de_data,
                   xlim=(-3,3), ylim=(-3,3), extra_genes=None):
    if extra_genes is None:
        extra_genes = []

# ------------------------------------------------------------------ #
# 1. Extract cluster identifiers for the p-value columns
# ------------------------------------------------------------------ #
    x_cluster = '_'.join(x_col.split('_')[1:3]) # e.g. CD8.R
    y_cluster = '_'.join(y_col.split('_')[1:3])
# ------------------------------------------------------------------ #
# 2. Pull the data we need
# ------------------------------------------------------------------ #
    cols = ['SYMBOL', x_col, y_col,
            f'adj.P.Val_{x_cluster}_All',
            f'adj.P.Val_{y_cluster}_All']
    df = de_data[cols].copy()
# ------------------------------------------------------------------ #
# 3. Keep only genes that are significant in at least one contrast
# ------------------------------------------------------------------ #
    sig = df[(df[cols[3]] < 0.05) | (df[cols[4]] < 0.05)].copy()
# ------------------------------------------------------------------ #
# 4. Top-10 up / down genes (largest positive & largest negative FC)
# ------------------------------------------------------------------ #
    sig['maxFC'] = sig[[x_col, y_col]].max(axis=1)
    sig['minFC'] = sig[[x_col, y_col]].min(axis=1)
    top_pos = sig.nlargest(10, 'maxFC')
    top_neg = sig.nsmallest(10, 'minFC')
    top_genes = pd.concat([top_pos, top_neg]).drop_duplicates()
# ------------------------------------------------------------------ #
# 5. Regression line (using only the significant points)
# ------------------------------------------------------------------ #
    x = sig[x_col].values
    y = sig[y_col].values
# Remove NaNs that would break the fit
    mask = ~np.isnan(x) & ~np.isnan(y)
    x_clean = x[mask]
    y_clean = y[mask]
    slope, intercept, r_val, _, _ = linregress(x_clean, y_clean)
    r2 = r_val**2
# Line coordinates for plotting
    line_x = np.array(xlim)
    line_y = slope * line_x + intercept
# ------------------------------------------------------------------ #
# 6. Plot
# ------------------------------------------------------------------ #
    plt.figure(figsize=(5, 5))
# background dots
    plt.scatter(sig[x_col], sig[y_col],
                c='black', alpha=1, s=5, rasterized=True)
# regression line
    plt.plot(line_x, line_y,
             color='royalblue', lw=1.5,
             label=f'y = {slope:.2f}x + {intercept:.2f}\nR² = {r2:.3f}')
# top-gene labels + extra genes
    texts = []
    # First: top 10 up/down
    for _, row in top_genes.iterrows():
        texts.append(
            plt.text(row[x_col], row[y_col], row['SYMBOL'],
                     fontsize=8, color='red',
                     ha='center', va='center')
        )
    # Second: user-specified extra genes (if present in data)
    extra_df = sig[sig['SYMBOL'].isin(extra_genes)]
    for _, row in extra_df.iterrows():
        texts.append(
            plt.text(row[x_col], row[y_col], row['SYMBOL'],
                     fontsize=8, color='red',
                     ha='center', va='center')
        )

    adjust_text(texts,
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
# cosmetics
    plt.xlabel(f'log₂FC ({x_label} vs All)')
    plt.ylabel(f'log₂FC ({y_label} vs All)')
    plt.title(title)
    plt.grid(True, color='#ddd', ls='--', lw=0.5)
    plt.axhline(0, color='k', lw=0.5, ls='--')
    plt.axvline(0, color='k', lw=0.5, ls='--')
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.legend(loc='upper left', fontsize=8, frameon=False)
# ------------------------------------------------------------------ #
# 7. Save & show
# ------------------------------------------------------------------ #
    plt.tight_layout()
    plt.savefig(pdf_filename, format='pdf', bbox_inches='tight', dpi=300)
    plt.show()
# ------------------------------------------------------------------ #
# 8. Print top genes (optional)
# ------------------------------------------------------------------ #
    print(f"\n=== {x_label} vs {y_label} ===")
    print("Top 10 positive FC genes:")
    print(top_pos[['SYMBOL', x_col, y_col, 'maxFC']])
    print("\nTop 10 negative FC genes:")
    print(top_neg[['SYMBOL', x_col, y_col, 'minFC']])
    print(f"Regression: slope={slope:.3f}, intercept={intercept:.3f}, R²={r2:.3f}\n")

# ================================================================== #
# Example usage with extra genes
# ================================================================== #
comparisons = [
    # 1. CD8.Q (X) vs CD8.R (Y)
    {
        'x_col': 'log2FC_CD8.Q_vs_All',
        'y_col': 'log2FC_CD8.R_vs_All',
        'x_label': 'CD8.Q',
        'y_label': 'CD8.R',
        'title': 'FC vs FC: CD8.Q (X) vs CD8.R (Y)',
        'pdf_filename': 'Fig_6e-g.pdf',
        'extra_genes': ['Tox', 'Pdcd1', 'Itgae', 'Itga1', 'Tigit', 'Lag3']  # <-- ADD YOUR GENES HERE
    },
    # 2. CD8.R vs CD8.S
    {
        'x_col': 'log2FC_CD8.R_vs_All',
        'y_col': 'log2FC_CD8.S_vs_All',
        'x_label': 'CD8.R',
        'y_label': 'CD8.S',
        'title': 'FC vs FC: CD8.R vs CD8.S',
        'pdf_filename': 'Fig_6e-g_v2.pdf',
        'extra_genes': ['Tox','Pdcd1', 'Itgae', 'Itga1', 'Cxcr6']  # or add genes like ['Tox', 'Eomes']
    },
    # 3. CD8.Q vs CD8.S
    {
        'x_col': 'log2FC_CD8.Q_vs_All',
        'y_col': 'log2FC_CD8.S_vs_All',
        'x_label': 'CD8.Q',
        'y_label': 'CD8.S',
        'title': 'FC vs FC: CD8.Q vs CD8.S',
        'pdf_filename': 'Fig_6e-g_v3.pdf',
        'extra_genes': ['Tox','Pdcd1', 'Itgae', 'Itga1', 'Cxcr6']
    }
]

# Run everything
for comp in comparisons:
    create_fc_plot(
        x_col=comp['x_col'],
        y_col=comp['y_col'],
        x_label=comp['x_label'],
        y_label=comp['y_label'],
        title=comp['title'],
        pdf_filename=comp['pdf_filename'],
        de_data=de_data,
        xlim=(-3, 3),
        ylim=(-3, 3),
        extra_genes=comp.get('extra_genes', [])  # safely pass extra genes
    )


In [ ]:
# Fig. 6i - Volcano plot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text

# === CONFIGURATION ===
cluster = 'CD8.C'
color = '#EE0000'  # Bright red
signature_file = 'data/ttlist_OneVsAll.txt'
save_path = 'Fig_6i.pdf'

# Genes you ALWAYS want labeled in bold red
extra_label_genes = ['Tox', 'Cx3cr1']

# Plot settings
pval_threshold = 0.05
log2fc_threshold = 0.5
label_top_n = 10
figsize = (5, 5)
xlim_min, xlim_max = -4, 4
# =====================

# Load data
df = pd.read_csv(signature_file, sep='\t', encoding='utf-8')

# Extract columns
log2fc_col = f'log2FC_{cluster}_vs_All'
pval_col = f'adj.P.Val_{cluster}_vs_All'
plot_df = df[['SYMBOL', log2fc_col, pval_col]].copy()
plot_df = plot_df.rename(columns={'SYMBOL': 'gene'})
plot_df['log2FC'] = plot_df[log2fc_col]
plot_df['minus_log10_padj'] = -np.log10(plot_df[pval_col].clip(lower=1e-300))

# Significance
plot_df['significant'] = (plot_df[pval_col] < pval_threshold) & (abs(plot_df['log2FC']) > log2fc_threshold)

# Top up/down regulated
sig = plot_df[plot_df['significant']]
top_up = sig[sig['log2FC'] > 0].sort_values('log2FC', ascending=False).head(label_top_n)
top_down = sig[sig['log2FC'] < 0].sort_values('log2FC', ascending=True).head(label_top_n)
top_genes = pd.concat([top_up, top_down])

# Extra genes to label (avoid duplicates with top genes)
extra_df = plot_df[plot_df['gene'].isin(extra_label_genes)]
extra_to_label = extra_df[~extra_df['gene'].isin(top_genes['gene'])]

# === Plot ===
fig, ax = plt.subplots(figsize=figsize)

# Non-significant
ax.scatter(plot_df[~plot_df['significant']]['log2FC'],
           plot_df[~plot_df['significant']]['minus_log10_padj'],
           color='grey', s=1, alpha=0.5, rasterized=True)

# Significant
ax.scatter(plot_df[plot_df['significant']]['log2FC'],
           plot_df[plot_df['significant']]['minus_log10_padj'],
           color=color, s=5, alpha=1, rasterized=True)

# === Labels ===
texts = []

# Top genes
for _, row in top_genes.iterrows():
    texts.append(ax.text(row['log2FC'], row['minus_log10_padj'], row['gene'],
                         fontsize=6, color='black',
                         ha='left' if row['log2FC'] > 0 else 'right'))

# Extra genes (in black)
for _, row in extra_to_label.iterrows():
    texts.append(ax.text(row['log2FC'], row['minus_log10_padj'], row['gene'],
                         fontsize=6, color='black',
                         ha='left' if row['log2FC'] > 0 else 'right'))

# Avoid label overlap
adjust_text(texts, arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
            expand_points=(1.3, 1.3), force_text=0.5)

# === Formatting ===
ax.set_xlabel(f'Log2 Fold Change ({cluster} vs Others)')
ax.set_ylabel('-Log10 (Adjusted P-value)')
ax.set_title(f'Volcano Plot: {cluster} vs All Others (Signature Genes)')
ax.set_xlim(xlim_min, xlim_max)

# Threshold lines
ax.axvline(log2fc_threshold, color='black', linestyle='--', lw=0.5)
ax.axvline(-log2fc_threshold, color='black', linestyle='--', lw=0.5)
ax.axhline(-np.log10(pval_threshold), color='black', linestyle='--', lw=0.5)

# **remove the grid**
ax.grid(False)

plt.tight_layout()
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"Volcano plot saved: {save_path}")


# Extended Data Fig. 1

In [ ]:
# Extended Data Fig. 1e — baseline organ composition
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

# Gray background for non-highlighted cells
default_color = '#BAB0AC'

# ──────────────────────────────────────────────────────────────
# 1. Create mask for your new condition
# ──────────────────────────────────────────────────────────────
empty_detailed = (
    rna.obs['condition_detailed'].isna() |
    (rna.obs['condition_detailed'].astype(str).str.strip() == '')
)

highlight_mask = (
    (rna.obs['condition_detailed_simplified'].str.lower() == 'baseline') &
    empty_detailed
)

# ──────────────────────────────────────────────────────────────
# 2. Exactly like your original code: use real annotation or None
# ──────────────────────────────────────────────────────────────
rna.obs['highlight_color'] = np.where(
    highlight_mask,
    rna.obs['cluster_annotation'],   # colored by cell type
    None                            # will become gray via na_color
)

# ──────────────────────────────────────────────────────────────
# 3. Same dot size logic as your original
#    but we make highlighted ones just a tiny bit bigger for visibility
# ──────────────────────────────────────────────────────────────
sizes = np.where(rna.obs['highlight_color'].notnull(), 5, 1)
# (80 and 40 give the same visual emphasis as your original "1 vs 1" but actually visible)

# ──────────────────────────────────────────────────────────────
# 4. Plot — 100% identical style to your working version
# ──────────────────────────────────────────────────────────────
plt.figure(figsize=(6.5, 5))
ax = plt.gca()

sc.pl.embedding(
    rna,
    basis="MDE_INCREMENTAL",
    color="highlight_color",
    palette=custom_colors,
    na_in_legend=False,           # no "NaN" in legend (same as before)
    legend_loc="right margin",
    ax=ax,
    size=sizes,                   # variable size (highlighted = bigger)
    show=False,
)

plt.tight_layout()
plt.savefig("Extended_Data_Fig_1e.pdf", bbox_inches="tight")
plt.show()
plt.close()

print("Plot saved: Extended_Data_Fig_1e.pdf")


In [ ]:
## Extended Data Fig. 1h — spleen LCMV Armstrong P14 on MDE
# Conditions to highlight (D30 and D60 merged)
conditions = ['spleen_LCMVarm_D7', 'spleen_LCMVarm_D30_D60']
# Gray background for non-highlighted cells
default_color = '#BAB0AC' # Gray for cells not meeting the condition
# Two panels: D7 and merged D30/D60
fig, axes = plt.subplots(1, 2, figsize=(7, 4))  # 2 subplots horizontally, adjusted size

# Iterate over conditions and corresponding axes
for idx, (condition, ax) in enumerate(zip(conditions, axes)):
    # Create a new column for coloring based on conditions
    if condition == 'spleen_LCMVarm_D30_D60':
        # Merged D30 + D60 condition
        rna.obs['highlight_color'] = np.where(
            (rna.obs['condition_detailed_organ'].isin(['spleen_LCMVarm_D30', 'spleen_LCMVarm_D60'])) & 
            (rna.obs['Ag_spe_v2'].str.lower() == 'p14'),
            rna.obs['cluster_annotation'],   # Use cluster_annotation for highlighted cells
            None                            # None for cells that don't meet the condition
        )
        plot_title = 'spleen_LCMVarm_D30_D60'
    else:
        # Original D7 condition
        rna.obs['highlight_color'] = np.where(
            (rna.obs['condition_detailed_organ'] == condition) & 
            (rna.obs['Ag_spe_v2'].str.lower() == 'p14'),
            rna.obs['cluster_annotation'],   # Use cluster_annotation for highlighted cells
            None                            # None for cells that don't meet the condition
        )
        plot_title = condition

    # Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 100, 1)

    # Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None,      # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=plot_title      # Add condition as subplot title
    )

# Adjust layout to prevent overlap
plt.tight_layout()

# Save and show the combined plot
plt.savefig("Extended_Data_Fig_1h.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Combined plot saved: Extended_Data_Fig_1h.pdf")


In [ ]:
# Extended Data Fig. 1h — spleen LCMV Armstrong Endogenous on MDE
# Conditions to highlight (D30 and D60 merged)
conditions = ['spleen_LCMVarm_D7', 'spleen_LCMVarm_D30_D60']
# Gray background for non-highlighted cells
default_color = '#BAB0AC' # Gray for cells not meeting the condition
# Two panels: D7 and merged D30/D60
fig, axes = plt.subplots(1, 2, figsize=(7, 4))  # 2 subplots horizontally, adjusted size

# Iterate over conditions and corresponding axes
for idx, (condition, ax) in enumerate(zip(conditions, axes)):
    # Create a new column for coloring based on conditions
    if condition == 'spleen_LCMVarm_D30_D60':
        # Merged D30 + D60 condition
        rna.obs['highlight_color'] = np.where(
            (rna.obs['condition_detailed_organ'].isin(['spleen_LCMVarm_D30', 'spleen_LCMVarm_D60'])) & 
            (rna.obs['Ag_spe_v2'].str.lower() == 'endogenous'),
            rna.obs['cluster_annotation'],   # Use cluster_annotation for highlighted cells
            None                            # None for cells that don't meet the condition
        )
        plot_title = 'spleen_LCMVarm_D30_D60'
    else:
        # Original D7 condition
        rna.obs['highlight_color'] = np.where(
            (rna.obs['condition_detailed_organ'] == condition) & 
            (rna.obs['Ag_spe_v2'].str.lower() == 'endogenous'),
            rna.obs['cluster_annotation'],   # Use cluster_annotation for highlighted cells
            None                            # None for cells that don't meet the condition
        )
        plot_title = condition

    # Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 20, 1)

    # Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None,      # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=plot_title      # Add condition as subplot title
    )

# Adjust layout to prevent overlap
plt.tight_layout()

# Save and show the combined plot
plt.savefig("Extended_Data_Fig_1h_v2.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Combined plot saved: Extended_Data_Fig_1h_v2.pdf")


In [ ]:
# Extended Data Fig. 1h — small intestine LCMV Armstrong P14 on MDE
# Conditions to highlight (D30 and D60 merged)
conditions = ['smallintestineIEL_LCMVarm_D7', 'smallintestineIEL_LCMVarm_D30_D60']
# Gray background for non-highlighted cells
default_color = '#BAB0AC' # Gray for cells not meeting the condition
# Two panels: D7 and merged D30/D60
fig, axes = plt.subplots(1, 2, figsize=(7, 4))  # 2 subplots horizontally, adjusted size

# Iterate over conditions and corresponding axes
for idx, (condition, ax) in enumerate(zip(conditions, axes)):
    # Create a new column for coloring based on conditions
    if condition == 'smallintestineIEL_LCMVarm_D30_D60':
        # Merged D30 + D60 condition
        rna.obs['highlight_color'] = np.where(
            (rna.obs['condition_detailed_organ'].isin(['smallintestineIEL_LCMVarm_D30', 'smallintestineIEL_LCMVarm_D60'])) & 
            (rna.obs['Ag_spe_v2'].str.lower() == 'p14'),
            rna.obs['cluster_annotation'],   # Use cluster_annotation for highlighted cells
            None                            # None for cells that don't meet the condition
        )
        plot_title = 'smallintestineIEL_LCMVarm_D30_D60'
    else:
        # Original D7 condition
        rna.obs['highlight_color'] = np.where(
            (rna.obs['condition_detailed_organ'] == condition) & 
            (rna.obs['Ag_spe_v2'].str.lower() == 'p14'),
            rna.obs['cluster_annotation'],   # Use cluster_annotation for highlighted cells
            None                            # None for cells that don't meet the condition
        )
        plot_title = condition

    # Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 100, 1)

    # Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None,      # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=plot_title      # Add condition as subplot title
    )

# Adjust layout to prevent overlap
plt.tight_layout()

# Save and show the combined plot
plt.savefig("Extended_Data_Fig_1h_v3.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Combined plot saved: Extended_Data_Fig_1h_v3.pdf")


In [ ]:
# Extended Data Fig. 1h — small intestine LCMV Armstrong Endogenous on MDE
# Conditions to highlight (D30 and D60 merged)
conditions = ['smallintestineIEL_LCMVarm_D7', 'smallintestineIEL_LCMVarm_D30_D60']
# Gray background for non-highlighted cells
default_color = '#BAB0AC' # Gray for cells not meeting the condition
# Two panels: D7 and merged D30/D60
fig, axes = plt.subplots(1, 2, figsize=(7, 4))  # 2 subplots horizontally, adjusted size

# Iterate over conditions and corresponding axes
for idx, (condition, ax) in enumerate(zip(conditions, axes)):
    # Create a new column for coloring based on conditions
    if condition == 'smallintestineIEL_LCMVarm_D30_D60':
        # Merged D30 + D60 condition
        rna.obs['highlight_color'] = np.where(
            (rna.obs['condition_detailed_organ'].isin(['smallintestineIEL_LCMVarm_D30', 'smallintestineIEL_LCMVarm_D60'])) & 
            (rna.obs['Ag_spe_v2'].str.lower() == 'endogenous'),
            rna.obs['cluster_annotation'],   # Use cluster_annotation for highlighted cells
            None                            # None for cells that don't meet the condition
        )
        plot_title = 'smallintestineIEL_LCMVarm_D30_D60'
    else:
        # Original D7 condition
        rna.obs['highlight_color'] = np.where(
            (rna.obs['condition_detailed_organ'] == condition) & 
            (rna.obs['Ag_spe_v2'].str.lower() == 'endogenous'),
            rna.obs['cluster_annotation'],   # Use cluster_annotation for highlighted cells
            None                            # None for cells that don't meet the condition
        )
        plot_title = condition

    # Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 20, 1)

    # Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None,      # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=plot_title      # Add condition as subplot title
    )

# Adjust layout to prevent overlap
plt.tight_layout()

# Save and show the combined plot
plt.savefig("Extended_Data_Fig_1h_v4.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Combined plot saved: Extended_Data_Fig_1h_v4.pdf")


In [ ]:
# Extended Data Fig. 1h — spleen LCMV Clone 13 P14 on MDE
# Conditions to highlight
conditions = ['spleen_LCMVcl13_D8', 'spleen_LCMVcl13_D27', 'spleen_LCMVcl13_D60']

# Gray background for non-highlighted cells
default_color = '#BAB0AC'  # Gray for cells not meeting the condition

# Three-panel layout
fig, axes = plt.subplots(1, 3, figsize=(10.5, 4))  # 3 subplots horizontally, adjusted size

# Iterate over conditions and corresponding axes
for idx, (condition, ax) in enumerate(zip(conditions, axes)):
    # Create a new column for coloring based on conditions
    rna.obs['highlight_color'] = np.where(
        (rna.obs['condition_detailed_organ'] == condition) & (rna.obs['Ag_spe_v2'].str.lower() == 'p14'),
        rna.obs['cluster_annotation'],  # Use cluster_annotation for highlighted cells
        None  # None for cells that don't meet the condition
    )
    
    # Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 20, 1)
    
    # Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None,  # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=condition  # Add condition as subplot title
    )

# Adjust layout to prevent overlap
plt.tight_layout()

# Save and show the combined plot
plt.savefig("Extended_Data_Fig_1h_v5.pdf", bbox_inches="tight")
plt.show()
plt.close()

print("Combined plot saved: Extended_Data_Fig_1h_v5.pdf")


In [ ]:
# Extended Data Fig. 1h — spleen LCMV Clone 13 Endogenous on MDE
# Work on a view/copy of the RNA modality
rna = mdata.mod['RNA']

# Conditions to highlight
conditions = ['spleen_LCMVcl13_D8', 'spleen_LCMVcl13_D27', 'spleen_LCMVcl13_D60']

# Gray background for non-highlighted cells
default_color = '#BAB0AC'  # Gray for cells not meeting the condition

# Three-panel layout
fig, axes = plt.subplots(1, 3, figsize=(10.5, 4))  # 3 subplots horizontally, adjusted size

# Iterate over conditions and corresponding axes
for idx, (condition, ax) in enumerate(zip(conditions, axes)):
    # Create a new column for coloring based on conditions
    rna.obs['highlight_color'] = np.where(
        (rna.obs['condition_detailed_organ'] == condition) & (rna.obs['Ag_spe_v2'].str.lower() == 'endogenous'),
        rna.obs['cluster_annotation'],  # Use cluster_annotation for highlighted cells
        None  # None for cells that don't meet the condition
    )
    
    # Set dot sizes: larger for highlighted cells
    sizes = np.where(rna.obs['highlight_color'].notnull(), 20, 1)
    
    # Plot MDE
    sc.pl.embedding(
        rna,
        basis="MDE_INCREMENTAL",
        color="highlight_color",
        palette=custom_colors,
        legend_loc=None,  # No legend
        ax=ax,
        size=sizes,
        show=False,
        title=condition  # Add condition as subplot title
    )

# Adjust layout to prevent overlap
plt.tight_layout()

# Save and show the combined plot
plt.savefig("Extended_Data_Fig_1h_v6.pdf", bbox_inches="tight")
plt.show()
plt.close()

print("Combined plot saved: Extended_Data_Fig_1h_v6.pdf")


In [ ]:
# Extended Data Fig. 1i — Naive-associated signature enrichment (module scores)
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

# ── Assuming your MuData object is already loaded ──
adata = mdata['RNA']

# Your signature files
signatures = {
    'CD5hi': 'data/CD5hi_Fulton.txt',   # From Fulton et al., Nat Immunol 2014
    'CD5lo': 'data/CD5lo_Fulton.txt',    # From Fulton et al., Nat Immunol 2014
}

# ── Define CD8 subset ───────────────────────────────────────────────────
cluster_key = 'cluster_annotation'
cd8_clusters = ['CD8.A', 'CD8.B']

mask_cd8 = adata.obs[cluster_key].isin(cd8_clusters)

if mask_cd8.sum() == 0:
    raise ValueError("No cells found in the specified CD8 clusters!")

print(f"Computing signatures using {mask_cd8.sum()} cells (CD8.A + CD8.B)")

# Initialize score columns with NaN
for sig_name in signatures:
    if sig_name not in adata.obs.columns:
        adata.obs[sig_name] = np.nan

# ── Compute signature scores ONLY on CD8 cells ──────────────────────────
for sig_name, sig_path in signatures.items():
    with open(sig_path, 'r') as f:
        gene_list = [line.strip() for line in f if line.strip()]
    
    gene_list = [g for g in gene_list if g in adata.var_names]
    
    if len(gene_list) == 0:
        print(f"No genes found for signature '{sig_name}'. Skipping.")
        continue
    
    adata_cd8 = adata[mask_cd8]
    
    sc.tl.score_genes(
        adata_cd8,
        gene_list,
        score_name=sig_name,
        random_state=42,
        copy=False
    )
    
    adata.obs.loc[mask_cd8, sig_name] = adata_cd8.obs[sig_name].values
    
    print(f"Computed score for '{sig_name}' using {len(gene_list)} genes.")


In [ ]:
# Extended Data Fig. 1i — Naive-associated signature enrichment (module scores)
# ── MDE colored by naive-associated signature scores ──
basis = 'MDE_INCREMENTAL'
if basis not in adata.obsm:
    raise ValueError(f"Embedding '{basis}' not found in adata.obsm.")

# ── Point sizes for background vs highlighted cells ──
background_size = 0.1      # ← change this for gray (non-CD8) cells
cd8_size = 0.5             # ← change this for colored (CD8) cells
# higher value = bigger dots

fig, axs = plt.subplots(1, 2, figsize=(11, 5))
axs = axs.flatten()

for i, sig_name in enumerate(signatures.keys()):
    if sig_name not in adata.obs.columns:
        print(f"Warning: '{sig_name}' not found in adata.obs – skipping plot")
        continue
    
    # Color limits based only on CD8 cells
    scores_cd8 = adata.obs[sig_name][mask_cd8]
    if len(scores_cd8) < 5:
        vmin, vmax = None, None
    else:
        vmin = np.percentile(scores_cd8, 5)
        vmax = np.percentile(scores_cd8, 95)
    
    # Create colors array (object dtype)
    colors = np.full(len(adata), '#d3d3d3', dtype=object)  # light gray by default
    
    if vmin is not None and vmax is not None:
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        cmap = plt.cm.viridis
        rgba_colors = cmap(norm(scores_cd8.values))
        cd8_indices = np.where(mask_cd8)[0]
        colors[cd8_indices] = [tuple(rgba) for rgba in rgba_colors]

    # ── Plot base layer: background gray points ─────────────────────────
    sc.pl.embedding(
        adata,
        basis=basis,
        color=None,
        ax=axs[i],
        show=False,
        frameon=False,
        title=f"{sig_name}\n(calculated & scaled using CD8.A + CD8.B only)",
    )
    
    emb = adata.obsm[basis]
    
    # Background (gray) points
    bg_mask = ~mask_cd8
    axs[i].scatter(
        emb[bg_mask, 0],
        emb[bg_mask, 1],
        c='#d3d3d3',
        s=background_size,
        rasterized=True,
        edgecolor='none',
        linewidth=0,
        zorder=1
    )
    
    # ── Overlay CD8 cells, sorted by score (highest on top) ─────────────
    if vmin is not None and vmax is not None:
        # Sort CD8 cells by score (highest last → plotted on top)
        sort_idx = np.argsort(scores_cd8.values)
        cd8_sorted_indices = cd8_indices[sort_idx]
        
        axs[i].scatter(
            emb[cd8_sorted_indices, 0],
            emb[cd8_sorted_indices, 1],
            c=colors[cd8_sorted_indices],
            s=cd8_size,
            rasterized=True,
            edgecolor='none',
            linewidth=0,
            zorder=2
        )
        
        # Colorbar
        sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=vmin, vmax=vmax))
        sm.set_array([])
        plt.colorbar(sm, ax=axs[i], fraction=0.046, pad=0.04, label=sig_name)

plt.tight_layout()
plt.show()

fig.savefig('Extended_Data_Fig_1i.pdf', dpi=350, bbox_inches='tight')


# Extended Data Fig. 2

In [ ]:
# Extended Data Fig. 2h-i — Heatmaps of expression for genes driving top factors
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load matrices (adjust paths as needed)
F_df = pd.read_csv("data/gene_factor_matrix.txt", sep="\t", index_col=0)
L_df = pd.read_csv("data/cell_factor_matrix.txt", sep="\t", index_col=0)

# Convert to NumPy arrays for positional matrix multiplication
F_np = F_df.to_numpy()
L_np = L_df.to_numpy()

# Preserve column names for factor labels
factor_names = L_df.columns  # Use F1–F200 from L_df
gene_names = F_df.index

# Verify shapes
print("F_np shape:", F_np.shape)  # Expected: (19805, 200)
print("L_np shape:", L_np.shape)  # Expected: (682953, 200)

# Assuming mdata is your MuData object
# Replace 'mdata' with your actual MuData variable
mdata_obs = mdata['RNA'].obs

# Check if cellID matches IGT_cellID
if 'cellID' in mdata_obs.columns:
    print("Using 'cellID' from mdata_obs")
    cell_id_col = 'cellID'
else:
    print("Using 'IGT_cellID' from mdata_obs")
    cell_id_col = 'IGT_cellID'

# Get unique clusters
unique_clusters = mdata_obs['cluster_annotation'].unique()
print(f"Found {len(unique_clusters)} clusters in cluster_annotation")

# Sort clusters alphabetically
unique_clusters = sorted(unique_clusters)
print("Sorted clusters:", unique_clusters)

# Function to compute log2FC for a cluster vs others
def compute_factor_log2fc(L_np, group1, group2, factor_names):
    group1 = [g for g in group1 if g in L_df.index]
    group2 = [g for g in group2 if g in L_df.index]
    if not group1 or not group2:
        return None
    group1_idx = [L_df.index.get_loc(g) for g in group1]
    group2_idx = [L_df.index.get_loc(g) for g in group2]
    loadings_group1 = L_np[group1_idx].mean(axis=0)
    loadings_group2 = L_np[group2_idx].mean(axis=0)
    fc_loadings = loadings_group1 - loadings_group2
    return pd.DataFrame({
        'SYMBOL': factor_names,
        'log2FC': fc_loadings / np.log(2)
    })

# Identify top factor for CD8.I, CD8.J, CD8.K
target_clusters = ['CD8.I', 'CD8.J', 'CD8.K']
top_factors = []
n_top = 1  # Top 1 factor per cluster

for cluster in target_clusters:
    group1 = mdata_obs[
        mdata_obs['cluster_annotation'] == cluster
    ][cell_id_col].tolist()
    group2 = mdata_obs[
        mdata_obs['cluster_annotation'] != cluster
    ][cell_id_col].tolist()
    
    if group1 and group2:
        log2fc_df = compute_factor_log2fc(L_np, group1, group2, factor_names)
        if log2fc_df is not None:
            top = log2fc_df.sort_values(by='log2FC', ascending=False).head(n_top)
            top_factors.extend(top['SYMBOL'].tolist())
            print(f"Top factor for {cluster}: {top['SYMBOL'].tolist()}")
    else:
        print(f"Skipping {cluster}: group1 has {len(group1)} cells, group2 has {len(group2)} cells")

# Remove duplicates while preserving order
top_factors = list(dict.fromkeys(top_factors))
print(f"Selected factors: {top_factors}")

# Compute mean loadings for each cluster and factor
cluster_loadings = []
for cluster in unique_clusters:
    group = mdata_obs[
        mdata_obs['cluster_annotation'] == cluster
    ][cell_id_col].tolist()
    
    if group:
        group_idx = [L_df.index.get_loc(g) for g in group]
        loadings = L_np[group_idx].mean(axis=0)
        for i, factor in enumerate(factor_names):
            if factor in top_factors:
                cluster_loadings.append({
                    'cluster': cluster,
                    'factor': factor,
                    'mean_loading': loadings[i],
                    'num_cells': len(group)
                })
    else:
        print(f"Skipping {cluster}: no valid cells")

# Convert to DataFrame
loadings_df = pd.DataFrame(cluster_loadings)

# Pivot data for plotting
pivot_df = loadings_df.pivot(index='cluster', columns='factor', values='mean_loading').fillna(0)
# Ensure clusters are in the desired order
pivot_df = pivot_df.reindex(unique_clusters)


In [ ]:
# Extended Data Fig. 2h-i — Heatmaps of expression for genes driving top factors
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import scipy.sparse as sparse

# Ensure we use your preferred normalized data
mdata['RNA'].X = mdata['RNA'].layers['scaled']

# Recompute top factors (self-contained block)
def compute_factor_log2fc(L_np, group1, group2, factor_names):
    group1 = [g for g in group1 if g in L_df.index]
    group2 = [g for g in group2 if g in L_df.index]
    if not group1 or not group2:
        return None
    group1_idx = [L_df.index.get_loc(g) for g in group1]
    group2_idx = [L_df.index.get_loc(g) for g in group2]
    loadings_group1 = L_np[group1_idx].mean(axis=0)
    loadings_group2 = L_np[group2_idx].mean(axis=0)
    fc_loadings = loadings_group1 - loadings_group2
    return pd.DataFrame({
        'SYMBOL': factor_names,
        'log2FC': fc_loadings / np.log(2)
    })

# Identify top factor for CD8.I, CD8.J, CD8.K
target_clusters = ['CD8.I', 'CD8.J', 'CD8.K']
top_factors = []
n_top = 1  # Top 1 factor per cluster
for cluster in target_clusters:
    group1 = mdata_obs[
        mdata_obs['cluster_annotation'] == cluster
    ][cell_id_col].tolist()
    group2 = mdata_obs[
        mdata_obs['cluster_annotation'] != cluster
    ][cell_id_col].tolist()
    if group1 and group2:
        log2fc_df = compute_factor_log2fc(L_np, group1, group2, factor_names)
        if log2fc_df is not None:
            top = log2fc_df.sort_values(by='log2FC', ascending=False).head(n_top)
            top_factors.extend(top['SYMBOL'].tolist())
            print(f"Top factor for {cluster}: {top['SYMBOL'].tolist()}")
    else:
        print(f"Skipping {cluster}: group1 has {len(group1)} cells, group2 has {len(group2)} cells")

# Remove duplicates
top_factors = list(dict.fromkeys(top_factors))
print(f"Selected factors: {top_factors}")

# Extract top genes for each selected factor
n_top_genes = 10  # Number of top genes per factor
factor_genes = {factor: [] for factor in top_factors}  # Store genes per factor
for factor in top_factors:
    factor_idx = list(factor_names).index(factor)
    gene_loadings = pd.Series(F_np[:, factor_idx], index=gene_names)
    top = gene_loadings.abs().sort_values(ascending=False).head(n_top_genes)
    factor_genes[factor] = top.index.tolist()
    print(f"Top {n_top_genes} genes for {factor}: {top.index.tolist()}")

# Compute mean expression across all clusters that will be plotted (unique_clusters must be defined)
# Assuming unique_clusters is already defined earlier in your notebook as all clusters you want to show
# If not defined, fallback to all CD8 clusters
if 'unique_clusters' not in globals():
    unique_clusters = sorted([c for c in mdata_obs['cluster_annotation'].unique() if c.startswith('CD8_')],
                             key=lambda x: int(x.split('_cl')[-1]) if '_cl' in x else 0)

cluster_mask = mdata_obs['cluster_annotation'].isin(unique_clusters)
subset_obs = mdata_obs[cluster_mask]
subset_X = mdata['RNA'].X[cluster_mask]
cluster_labels = subset_obs['cluster_annotation'].values

unique_labels, inverse = np.unique(cluster_labels, return_inverse=True)
n_genes = subset_X.shape[1]

mean_expr_matrix = np.empty((len(unique_labels), n_genes))
for i, label in enumerate(unique_labels):
    mask_i = (inverse == i)
    if sparse.issparse(subset_X):
        mean_expr_matrix[i] = np.ravel(subset_X[mask_i].mean(axis=0))
    else:
        mean_expr_matrix[i] = subset_X[mask_i].mean(axis=0)

mean_expr_pivot = pd.DataFrame(
    mean_expr_matrix,
    index=unique_labels,
    columns=mdata['RNA'].var_names
).T  # genes x clusters
mean_expr_pivot = mean_expr_pivot[unique_clusters]  # Order correctly

# === Build expression DataFrame using expression ===
gene_expression = []
for cluster in unique_clusters:
    for factor in top_factors:
        for gene in factor_genes[factor]:
            if gene in mean_expr_pivot.index:
                gene_expression.append({
                    'cluster': cluster,
                    'gene': gene,
                    'mean_expr': mean_expr_pivot.loc[gene, cluster],
                    'factor': factor
                })

# Convert to DataFrame
expr_df = pd.DataFrame(gene_expression)

# Use a classic diverging red-blue colormap (negative: blue, zero: white, positive: red)
cmap = 'RdBu_r'  # Built-in matplotlib colormap: red-blue reversed (high=red, low=blue)

# Set z-score range for expression scale
zscore_min = -2.0
zscore_max = 2.0

# Create heatmaps for each factor
for i, factor in enumerate(top_factors):
    # Filter expression data for genes of this factor
    factor_expr_df = expr_df[expr_df['factor'] == factor]
    pivot_df = factor_expr_df.pivot(index='gene', columns='cluster', values='mean_expr').fillna(0)
    pivot_df = pivot_df[unique_clusters]  # Order clusters

    # Z-score normalization: (x - mean) / std
    pivot_df = pivot_df.apply(lambda x: (x - x.mean()) / x.std() if x.std() != 0 else x, axis=1)

    # Order genes by max absolute z-score
    pivot_df = pivot_df.loc[pivot_df.abs().max(axis=1).sort_values(ascending=False).index]

    # Use red-blue colormap for real expression
    cmap = 'RdBu_r'

    # Create heatmap
    fig, ax = plt.subplots(figsize=(8, 4.25))
    # Plot heatmap with fixed z-score range for colormap
    heatmap = ax.imshow(pivot_df.values, cmap=cmap, aspect='auto',
                        vmin=zscore_min, vmax=zscore_max, interpolation='nearest')
    # Reverse y-axis to match DataFrame order (highest z-score at top)
    ax.invert_yaxis()
    # Customize plot
    ax.set_xticks(np.arange(len(unique_clusters)))
    ax.set_yticks(np.arange(len(pivot_df.index)))
    ax.set_xticklabels(unique_clusters, rotation=45, ha='right')
    ax.set_yticklabels(pivot_df.index)
    ax.set_xlabel('Annotation Level 2 Clusters')
    ax.set_ylabel('Genes')
    ax.set_title(f'Differentially Expressed Genes for {factor}')
    ax.set_xticklabels([cluster.replace('CD8_', '') for cluster in unique_clusters], rotation=90, ha='center')
    ax.grid(False)
    # Add colorbar
    plt.colorbar(heatmap, label='Z-score Normalized Expression')
    plt.tight_layout()
    # Save as PDF
    plt.savefig(f'Extended_Data_Fig_2h-i_{factor}.pdf', format='pdf', bbox_inches='tight', dpi=300)
    plt.show()
    plt.close()
    print(f'Plot saved: Extended_Data_Fig_2h-i_{factor}.pdf')

# Print results
print("\nZ-score normalized gene expression for all genes:")
pivot_df = expr_df.groupby(['gene', 'cluster'])['mean_expr'].first().unstack().fillna(0)[unique_clusters]
pivot_df = pivot_df.apply(lambda x: (x - x.mean()) / x.std() if x.std() != 0 else x, axis=1)
print(pivot_df)


# Extended Data Fig. 3

In [ ]:
# Extended Data Fig. 3c - Heatmaps of gene-program gene expression (z-scored mean expression)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load matrices (adjust paths as needed)
F_df = pd.read_csv("data/gene_factor_matrix.txt", sep="\t", index_col=0)
L_df = pd.read_csv("data/cell_factor_matrix.txt", sep="\t", index_col=0)

# Convert to NumPy arrays for positional matrix multiplication
F_np = F_df.to_numpy()
L_np = L_df.to_numpy()

# Preserve column names for factor labels
factor_names = L_df.columns  # Use F1–F200 from L_df
gene_names = F_df.index

# Verify shapes
print("F_np shape:", F_np.shape)  # Expected: (19805, 200)
print("L_np shape:", L_np.shape)  # Expected: (682953, 200)

# Assuming mdata is your MuData object
# Replace 'mdata' with your actual MuData variable
mdata_obs = mdata['RNA'].obs

# Check if cellID matches IGT_cellID
if 'cellID' in mdata_obs.columns:
    print("Using 'cellID' from mdata_obs")
    cell_id_col = 'cellID'
else:
    print("Using 'IGT_cellID' from mdata_obs")
    cell_id_col = 'IGT_cellID'

# Get unique clusters
unique_clusters = mdata_obs['cluster_annotation'].unique()
print(f"Found {len(unique_clusters)} clusters in cluster_annotation")

# Function to compute log2FC for a cluster vs others
def compute_factor_log2fc(L_np, group1, group2, factor_names):
    group1 = [g for g in group1 if g in L_df.index]
    group2 = [g for g in group2 if g in L_df.index]
    if not group1 or not group2:
        return None
    group1_idx = [L_df.index.get_loc(g) for g in group1]
    group2_idx = [L_df.index.get_loc(g) for g in group2]
    loadings_group1 = L_np[group1_idx].mean(axis=0)
    loadings_group2 = L_np[group2_idx].mean(axis=0)
    fc_loadings = loadings_group1 - loadings_group2
    return pd.DataFrame({
        'SYMBOL': factor_names,
        'log2FC': fc_loadings / np.log(2)
    })

# Identify top factor for CD8.E, CD8.F, CD8.G, CD8.H
target_clusters = ['CD8.E', 'CD8.F', 'CD8.G', 'CD8.H']
top_factors = []
n_top = 1  # Top 1 factor per cluster

for cluster in target_clusters:
    group1 = mdata_obs[
        mdata_obs['cluster_annotation'] == cluster
    ][cell_id_col].tolist()
    group2 = mdata_obs[
        mdata_obs['cluster_annotation'] != cluster
    ][cell_id_col].tolist()
    
    if group1 and group2:
        log2fc_df = compute_factor_log2fc(L_np, group1, group2, factor_names)
        if log2fc_df is not None:
            top = log2fc_df.sort_values(by='log2FC', ascending=False).head(n_top)
            top_factors.extend(top['SYMBOL'].tolist())
            print(f"Top factor for {cluster}: {top['SYMBOL'].tolist()}")
    else:
        print(f"Skipping {cluster}: group1 has {len(group1)} cells, group2 has {len(group2)} cells")

# Remove duplicates while preserving order
top_factors = list(dict.fromkeys(top_factors))
print(f"Selected factors: {top_factors}")

# Compute mean loadings for each cluster and factor
cluster_loadings = []
for cluster in unique_clusters:
    group = mdata_obs[
        mdata_obs['cluster_annotation'] == cluster
    ][cell_id_col].tolist()
    
    if group:
        group_idx = [L_df.index.get_loc(g) for g in group]
        loadings = L_np[group_idx].mean(axis=0)
        for i, factor in enumerate(factor_names):
            if factor in top_factors:
                cluster_loadings.append({
                    'cluster': cluster,
                    'factor': factor,
                    'mean_loading': loadings[i],
                    'num_cells': len(group)
                })
    else:
        print(f"Skipping {cluster}: no valid cells")

# Convert to DataFrame
loadings_df = pd.DataFrame(cluster_loadings)

# Pivot data for plotting
pivot_df = loadings_df.pivot(index='cluster', columns='factor', values='mean_loading').fillna(0)
# Ensure clusters are in the desired order
pivot_df = pivot_df.reindex(unique_clusters)


In [ ]:
# Extended Data Fig. 3c - Heatmaps of gene-program gene expression (z-scored mean expression)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import scipy.sparse as sparse
# Ensure we use your preferred normalized data
mdata['RNA'].X = mdata['RNA'].layers['scaled']
# Recompute top factors (self-contained block)
def compute_factor_log2fc(L_np, group1, group2, factor_names):
    group1 = [g for g in group1 if g in L_df.index]
    group2 = [g for g in group2 if g in L_df.index]
    if not group1 or not group2:
        return None
    group1_idx = [L_df.index.get_loc(g) for g in group1]
    group2_idx = [L_df.index.get_loc(g) for g in group2]
    loadings_group1 = L_np[group1_idx].mean(axis=0)
    loadings_group2 = L_np[group2_idx].mean(axis=0)
    fc_loadings = loadings_group1 - loadings_group2
    return pd.DataFrame({
        'SYMBOL': factor_names,
        'log2FC': fc_loadings / np.log(2)
    })
# Identify top factor for CD8.E, CD8.F, CD8.H, CD8.G
target_clusters = ['CD8.E', 'CD8.F', 'CD8.H', 'CD8.G']
top_factors = []
n_top = 1 # Top 1 factor per cluster
for cluster in target_clusters:
    group1 = mdata_obs[
        mdata_obs['cluster_annotation'] == cluster
    ][cell_id_col].tolist()
    group2 = mdata_obs[
        mdata_obs['cluster_annotation'] != cluster
    ][cell_id_col].tolist()
    if group1 and group2:
        log2fc_df = compute_factor_log2fc(L_np, group1, group2, factor_names)
        if log2fc_df is not None:
            top = log2fc_df.sort_values(by='log2FC', ascending=False).head(n_top)
            top_factors.extend(top['SYMBOL'].tolist())
            print(f"Top factor for {cluster}: {top['SYMBOL'].tolist()}")
        else:
            print(f"Skipping {cluster}: group1 has {len(group1)} cells, group2 has {len(group2)} cells")
# Remove duplicates
top_factors = list(dict.fromkeys(top_factors))
print(f"Selected factors: {top_factors}")
# Extract top genes for each selected factor
n_top_genes = 10 # Number of top genes per factor
factor_genes = {factor: [] for factor in top_factors}
for factor in top_factors:
    factor_idx = list(factor_names).index(factor)
    gene_loadings = pd.Series(F_np[:, factor_idx], index=gene_names)
    top = gene_loadings.abs().sort_values(ascending=False).head(n_top_genes)
    factor_genes[factor] = top.index.tolist()
    print(f"Top {n_top_genes} genes for {factor}: {top.index.tolist()}")
# Automatically use all CD8 clusters if unique_clusters not defined
if 'unique_clusters' not in globals():
    unique_clusters = sorted(
        [c for c in mdata_obs['cluster_annotation'].unique() if c.startswith('CD8_')]
    )
cluster_mask = mdata_obs['cluster_annotation'].isin(unique_clusters)
subset_obs = mdata_obs[cluster_mask]
subset_X = mdata['RNA'].X[cluster_mask]
cluster_labels = subset_obs['cluster_annotation'].values
unique_labels, inverse = np.unique(cluster_labels, return_inverse=True)
n_genes = subset_X.shape[1]
mean_expr_matrix = np.empty((len(unique_labels), n_genes))
for i, label in enumerate(unique_labels):
    mask_i = (inverse == i)
    if sparse.issparse(subset_X):
        mean_expr_matrix[i] = np.ravel(subset_X[mask_i].mean(axis=0))
    else:
        mean_expr_matrix[i] = subset_X[mask_i].mean(axis=0)
mean_expr_pivot = pd.DataFrame(
    mean_expr_matrix,
    index=unique_labels,
    columns=mdata['RNA'].var_names
).T # genes x clusters
mean_expr_pivot = mean_expr_pivot[unique_clusters] # Order correctly
# === Build expression DataFrame using REAL expression ===
gene_expression = []
for cluster in unique_clusters:
    for factor in top_factors:
        for gene in factor_genes[factor]:
            if gene in mean_expr_pivot.index:
                gene_expression.append({
                    'cluster': cluster,
                    'gene': gene,
                    'mean_expr': mean_expr_pivot.loc[gene, cluster],
                    'factor': factor
                })
expr_df = pd.DataFrame(gene_expression)
# Use a classic diverging red-blue colormap (negative: blue, zero: white, positive: red)
cmap = 'RdBu_r' # Built-in matplotlib colormap: red-blue reversed (high=red, low=blue)
# Set z-score range for expression scale
zscore_min = -2.0
zscore_max = 2.0
# Create heatmaps for each factor
for i, factor in enumerate(top_factors):
    factor_expr_df = expr_df[expr_df['factor'] == factor]
    pivot_df = factor_expr_df.pivot(index='gene', columns='cluster', values='mean_expr').fillna(0)
    pivot_df = pivot_df[unique_clusters]
    # Z-score normalization per gene
    pivot_df = pivot_df.apply(lambda x: (x - x.mean()) / x.std() if x.std() != 0 else x, axis=1)
    # Order genes by max absolute z-score
    pivot_df = pivot_df.loc[pivot_df.abs().max(axis=1).sort_values(ascending=False).index]
    # Color map
    cmap = 'RdBu_r'
    # Create heatmap
    fig, ax = plt.subplots(figsize=(8, 4.25))
    heatmap = ax.imshow(pivot_df.values, cmap=cmap, aspect='auto',
    vmin=zscore_min, vmax=zscore_max, interpolation='nearest')
    ax.invert_yaxis()
    ax.set_xticks(np.arange(len(unique_clusters)))
    ax.set_yticks(np.arange(len(pivot_df.index)))
    ax.set_xticklabels(unique_clusters, rotation=45, ha='right')
    ax.set_yticklabels(pivot_df.index)
    ax.set_xlabel('Annotation Level 2 Clusters')
    ax.set_ylabel('Genes')
    ax.set_title(f'Differentially Expressed Genes for {factor}')
    ax.set_xticklabels([cluster.replace('CD8_', '') for cluster in unique_clusters], rotation=90, ha='center')
    ax.grid(False)
    plt.colorbar(heatmap, label='Z-score Normalized Expression')
    plt.tight_layout()
    plt.savefig(f'Extended_Data_Fig_3c_{factor}.pdf', format='pdf', bbox_inches='tight', dpi=300)
    plt.show()
    plt.close()
    print(f'Plot saved: Extended_Data_Fig_3c_{factor}.pdf')
# Print results
print("\nZ-score normalized gene expression for all genes:")
pivot_df = expr_df.groupby(['gene', 'cluster'])['mean_expr'].first().unstack().fillna(0)[unique_clusters]
pivot_df = pivot_df.apply(lambda x: (x - x.mean()) / x.std() if x.std() != 0 else x, axis=1)
print(pivot_df)

In [ ]:
# Extended Data Fig. 3d — Representative condition enriched for memory clusters
condition = 'skinback_VV_D80_bystander'
default_color = '#BAB0AC'  # Gray for cells not meeting the condition

fig, ax = plt.subplots(figsize=(3.75, 4))

# Create a new column for coloring based on the condition
rna.obs['highlight_color'] = np.where(
    rna.obs['condition_detailed_organ'] == condition,
    rna.obs['cluster_annotation'],  # Use cluster_annotation for highlighted cells
    None                            # None for cells that don't meet the condition
)

# Set dot sizes: larger for highlighted cells
sizes = np.where(rna.obs['highlight_color'].notnull(), 20, 1)

# Plot MDE
sc.pl.embedding(
    rna,
    basis="MDE_INCREMENTAL",
    color="highlight_color",
    palette=custom_colors,
    legend_loc=None,
    ax=ax,
    size=sizes,
    show=False,
    title=condition
)

plt.tight_layout()
plt.savefig("Extended_Data_Fig_3d.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Plot saved: Extended_Data_Fig_3d.pdf")

In [ ]:
# Extended Data Fig. 3e-h - Volcano plots and balloon plots (memory cluster DEGs)
import pandas as pd

# Set your MuData path here (replace with the actual path to your .h5mu file)
adata_all = mdata['RNA']  # Extract the RNA modality

conditions = ["healthy", "autoimmunity", "virus"]
clusters = ['CD8.E', 'CD8.F', 'CD8.H', 'CD8.G']

# Select best sample per condition for each cluster
selected_samples = {}
for cluster in clusters:
    selected_samples[cluster] = []
    for condition in conditions:
        # Use a boolean mask to filter obs without copying the AnnData
        subset_mask = adata_all.obs['condition_broad'] == condition
        
        # Work directly with the filtered obs DataFrame
        obs_subset = adata_all.obs[subset_mask][['sample_code', 'cluster_annotation']]
        
        # Compute cell counts per sample and cluster
        cell_counts = obs_subset.groupby(['sample_code', 'cluster_annotation']).size().reset_index(name='cell_count')
        
        if cluster not in obs_subset['cluster_annotation'].values:
            continue
        
        cluster_counts = cell_counts[cell_counts['cluster_annotation'] == cluster]
        if cluster_counts.empty:
            continue
        
        best_sample = cluster_counts.sort_values('cell_count', ascending=False).iloc[0]['sample_code']
        selected_samples[cluster].append((condition, best_sample))

# Print or return the selected samples
for cluster in clusters:
    if selected_samples[cluster]:
        print(f"Cluster {cluster}:")
        for condition, sample in selected_samples[cluster]:
            print(f"  Condition: {condition}, Selected Sample: {sample}")


In [ ]:
# Extended Data Fig. 3e-h - Volcano plots and balloon plots (memory cluster DEGs)
import pandas as pd
import numpy as np

def select_top_genes(signature_file, clusters, pval_threshold=0.05, log2fc_threshold=0.5, highlight_gene_column='SYMBOL'):
    """
    Select the top 10 up-regulated and top 10 down-regulated genes for each cluster from a signature file.
    Parameters:
    - signature_file (str): Path to the signature file with DE results.
    - clusters (list): List of cluster names (e.g., ['CD8.E', 'CD8.F', ...]).
    - pval_threshold (float): Adjusted p-value threshold for significance.
    - log2fc_threshold (float): |log2FC| threshold for significance.
    - highlight_gene_column (str): Column name for genes in signature_file.
    Returns:
    - dict: Dictionary with cluster names as keys and tuples of (top_10_up, top_10_down) gene lists as values.
    """
    # Load the signature DataFrame
    signature_df = pd.read_csv(signature_file, sep='\t', encoding='utf-8')
    
    top_genes = {}
    for cluster in clusters:
        # Prepare data for the cluster
        log2fc_col = f'log2FC_{cluster}_vs_All'
        pval_col = f'adj.P.Val_{cluster}_vs_All'
        plot_df = signature_df[[highlight_gene_column, log2fc_col, pval_col]].copy()
        plot_df['gene'] = plot_df[highlight_gene_column]
        plot_df['log2FC'] = plot_df[log2fc_col]
        plot_df['minus_log10_padj'] = -np.log10(plot_df[pval_col].clip(lower=1e-300))
        
        # Highlight significant genes
        significant = plot_df[
            (plot_df[pval_col] < pval_threshold) & 
            (abs(plot_df['log2FC']) > log2fc_threshold)
        ]
        
        # Select top 10 up-regulated (highest positive log2FC)
        top_10_up = significant[significant['log2FC'] > 0][['gene', 'log2FC']].sort_values(
            'log2FC', ascending=False
        ).head(10)['gene'].tolist()
        
        # Select top 10 down-regulated (lowest negative log2FC)
        top_10_down = significant[significant['log2FC'] < 0][['gene', 'log2FC']].sort_values(
            'log2FC', ascending=True
        ).head(10)['gene'].tolist()
        
        top_genes[cluster] = (top_10_up, top_10_down)
    
    return top_genes

# Define file path and clusters
main_signature_file = 'data/ttlist_OneVsAll.txt'
clusters = ['CD8.E', 'CD8.F', 'CD8.H', 'CD8.G']

# Select top genes
top_genes_dict = select_top_genes(
    signature_file=main_signature_file,
    clusters=clusters,
    pval_threshold=0.05,
    log2fc_threshold=0.5,
    highlight_gene_column='SYMBOL'
)

# Print results
for cluster in clusters:
    up_genes, down_genes = top_genes_dict[cluster]
    print(f"Cluster {cluster}:")
    print(f"  Top 10 Up-regulated Genes: {up_genes}")
    print(f"  Top 10 Down-regulated Genes: {down_genes}")


In [ ]:
# Extended Data Fig. 3e-h - Volcano plots and balloon plots (memory cluster DEGs)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from adjustText import adjust_text

# Define custom colors for clusters
custom_colors = {
    'CD8.E': '#6B8E23',  # darkolivegreen2 (olive drab)
    'CD8.F': '#9400D3',  # darkorchid2
    'CD8.H': '#7AC5CD',  # cadetblue2
    'CD8.G': '#CD3333',  # brown2
}

def plot_volcano(
    signature_file,
    cluster,
    pval_threshold=0.05,
    log2fc_threshold=0.5,
    non_highlight_size=1,
    highlight_size=5,
    label_top_n=10,
    figsize=(5, 5),
    save_path=None,
    highlight_gene_column='SYMBOL',
    xlim_min=-3,
    xlim_max=4
):
    """
    Generate a volcano plot from a signature DataFrame for a specific cluster with staggered gene labels.
    Labels the top 10 up-regulated and top 10 down-regulated significant genes.
    Parameters:
    - signature_file (str): Path to the signature file with DE results.
    - cluster (str): Cluster name (e.g., 'CD8.E', 'CD8.F', etc.).
    - pval_threshold (float): Adjusted p-value threshold for significance.
    - log2fc_threshold (float): |log2FC| threshold for significance.
    - non_highlight_size (float): Size of non-significant gene points.
    - highlight_size (float): Size of significant gene points.
    - label_top_n (int): Number of top up- and down-regulated genes to label.
    - figsize (tuple): Figure size (width, height).
    - save_path (str, optional): Path to save the plot. If None, plot is displayed.
    - highlight_gene_column (str): Column name for genes in signature_file.
    - xlim_min (float): Minimum x-axis limit (log2 fold change).
    - xlim_max (float): Maximum x-axis limit (log2 fold change).
    """
    # Load the signature DataFrame
    signature_df = pd.read_csv(signature_file, sep='\t', encoding='utf-8')
    # Prepare data for volcano plot
    log2fc_col = f'log2FC_{cluster}_vs_All'
    pval_col = f'adj.P.Val_{cluster}_vs_All'
    plot_df = signature_df[[highlight_gene_column, log2fc_col, pval_col]].copy()
    plot_df['gene'] = plot_df[highlight_gene_column]
    plot_df['log2FC'] = plot_df[log2fc_col]
    plot_df['minus_log10_padj'] = -np.log10(plot_df[pval_col].clip(lower=1e-300))
    # Highlight based on significance thresholds
    plot_df['highlight'] = (plot_df[pval_col] < pval_threshold) & (abs(plot_df['log2FC']) > log2fc_threshold)
    # Create the volcano plot
    fig, ax = plt.subplots(figsize=figsize)
    # Plot non-significant genes
    non_highlight = plot_df[~plot_df['highlight']]
    ax.scatter(
        non_highlight['log2FC'],
        non_highlight['minus_log10_padj'],
        color='grey',
        alpha=0.5,
        s=non_highlight_size,
        rasterized=True
    )
    # Plot significant genes with cluster-specific color
    highlight = plot_df[plot_df['highlight']]
    ax.scatter(
        highlight['log2FC'],
        highlight['minus_log10_padj'],
        color=custom_colors.get(cluster, '#FF0000'),  # Use cluster color, fallback to red
        alpha=1,
        s=highlight_size,
        rasterized=True
    )
    # Add labels for top 10 up-regulated and top 10 down-regulated significant genes
    significant = plot_df[plot_df['highlight']]
    top_up = significant[significant['log2FC'] > 0][['gene', 'log2FC', 'minus_log10_padj']].sort_values(
        'log2FC', ascending=False
    ).head(label_top_n)
    top_down = significant[significant['log2FC'] < 0][['gene', 'log2FC', 'minus_log10_padj']].sort_values(
        'log2FC', ascending=True
    ).head(label_top_n)
    top_highlight = pd.concat([top_up, top_down])
    texts = []
    for _, row in top_highlight.iterrows():
        text = ax.text(
            row['log2FC'],
            row['minus_log10_padj'],
            row['gene'],
            fontsize=6,
            ha='right' if row['log2FC'] < 0 else 'left'
        )
        texts.append(text)
    # Adjust text to avoid overlap
    adjust_text(
        texts,
        arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
        expand_points=(1.2, 1.2),
        force_points=0.5,
        force_text=0.5
    )
    # Axes and title
    ax.set_xlabel(f'Log2 Fold Change ({cluster} vs Others)')
    ax.set_ylabel('-Log10 (Adjusted P-value)')
    ax.set_title(f'Volcano Plot: {cluster} vs All Others (Signature Genes)')
    # Set x-axis limits
    ax.set_xlim(xlim_min, xlim_max)
    # Add significance lines
    ax.axvline(x=log2fc_threshold, color='black', linestyle='--', linewidth=0.5)
    ax.axvline(x=-log2fc_threshold, color='black', linestyle='--', linewidth=0.5)
    ax.axhline(y=-np.log10(pval_threshold), color='black', linestyle='--', linewidth=0.5)
    # Adjust layout
    plt.tight_layout()
    # Save or display
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()

# Define file path and clusters
main_signature_file = 'data/ttlist_OneVsAll.txt'
clusters = ['CD8.E', 'CD8.F', 'CD8.H', 'CD8.G']

# Generate volcano plots
for cluster in clusters:
    save_path = f'Extended_Data_Fig_3e-h_{cluster}.pdf'
    plot_volcano(
        signature_file=main_signature_file,
        cluster=cluster,
        pval_threshold=0.05,
        log2fc_threshold=0.5,
        non_highlight_size=1,
        highlight_size=5,
        label_top_n=10,
        figsize=(5, 5),
        save_path=save_path,
        highlight_gene_column='SYMBOL',
        xlim_min=-3,
        xlim_max=4
    )


In [ ]:
# Extended Data Fig. 3e-h - Volcano plots and balloon plots (memory cluster DEGs)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import scanpy as sc
from matplotlib.colors import LinearSegmentedColormap

# Define custom colors for clusters
custom_colors = {
    'CD8.E': '#6B8E23',  # darkolivegreen2 (olive drab)
    'CD8.F': '#9400D3',  # darkorchid2
    'CD8.H': '#7AC5CD',  # cadetblue2
    'CD8.G': '#CD3333',  # brown2
}

def select_top_genes(signature_file, clusters, pval_threshold=0.05, log2fc_threshold=0.5, highlight_gene_column='SYMBOL'):
    """
    Select the top 10 up-regulated and top 10 down-regulated genes for each cluster from a signature file.
    Parameters:
    - signature_file (str): Path to the signature file with DE results.
    - clusters (list): List of cluster names (e.g., ['CD8.E', 'CD8.F', ...]).
    - pval_threshold (float): Adjusted p-value threshold for significance.
    - log2fc_threshold (float): |log2FC| threshold for significance.
    - highlight_gene_column (str): Column name for genes in signature_file.
    Returns:
    - dict: Dictionary with cluster names as keys and tuples of (top_10_up, top_10_down) gene lists as values.
    """
    signature_df = pd.read_csv(signature_file, sep='\t', encoding='utf-8')
    top_genes = {}
    for cluster in clusters:
        log2fc_col = f'log2FC_{cluster}_vs_All'
        pval_col = f'adj.P.Val_{cluster}_vs_All'
        plot_df = signature_df[[highlight_gene_column, log2fc_col, pval_col]].copy()
        plot_df['gene'] = plot_df[highlight_gene_column]
        plot_df['log2FC'] = plot_df[log2fc_col]
        significant = plot_df[
            (plot_df[pval_col] < pval_threshold) & 
            (abs(plot_df['log2FC']) > log2fc_threshold)
        ]
        top_10_up = significant[significant['log2FC'] > 0][['gene', 'log2FC']].sort_values(
            'log2FC', ascending=False
        ).head(10)['gene'].tolist()
        top_10_down = significant[significant['log2FC'] < 0][['gene', 'log2FC']].sort_values(
            'log2FC', ascending=True
        ).head(10)['gene'].tolist()
        top_genes[cluster] = (top_10_up, top_10_down)
    return top_genes

# Define file path and clusters
main_signature_file = 'data/ttlist_OneVsAll.txt'
clusters = ['CD8.E', 'CD8.F', 'CD8.H', 'CD8.G']

# Select top genes
top_genes_dict = select_top_genes(
    signature_file=main_signature_file,
    clusters=clusters,
    pval_threshold=0.05,
    log2fc_threshold=0.5,
    highlight_gene_column='SYMBOL'
)

# Load MuData
adata_all = mdata['RNA']  # Extract the RNA modality

conditions = ["healthy", "autoimmunity", "virus"]
os.makedirs('figures', exist_ok=True)  # Create figures directory for scanpy outputs

# Select best samples and generate balloon plots
for cluster in clusters:
    # Get significant genes (combine up and down)
    up_genes, down_genes = top_genes_dict[cluster]
    sig_genes = up_genes + down_genes
    if not sig_genes:
        print(f"No significant genes for {cluster}, skipping balloon plot.")
        continue
    
    # Select best sample per condition for this cluster
    selected_samples = []
    for condition in conditions:
        subset_mask = adata_all.obs['condition_broad'] == condition
        obs_subset = adata_all.obs[subset_mask][['sample_code', 'cluster_annotation']]
        cell_counts = obs_subset.groupby(['sample_code', 'cluster_annotation']).size().reset_index(name='cell_count')
        
        if cluster not in obs_subset['cluster_annotation'].values:
            continue
        
        cluster_counts = cell_counts[cell_counts['cluster_annotation'] == cluster]
        if cluster_counts.empty:
            continue
        
        best_sample = cluster_counts.sort_values('cell_count', ascending=False).iloc[0]['sample_code']
        selected_samples.append((condition, best_sample))
    
    if not selected_samples:
        print(f"No selected samples for {cluster} across conditions, skipping balloon plot.")
        continue
    
    # Prepare subset AnnData for the cluster cells in selected samples
    sample_list = [samp for cond, samp in selected_samples]
    sample_to_condition = {samp: cond for cond, samp in selected_samples}  # For reference, not used in plot
    
    cluster_mask = adata_all.obs['cluster_annotation'] == cluster
    sample_mask = adata_all.obs['sample_code'].isin(sample_list)
    adata_cluster = adata_all[sample_mask & cluster_mask]
    
    if adata_cluster.n_obs == 0:
        print(f"No cells in selected samples for {cluster}, skipping balloon plot.")
        continue
    
    # Use normalized scaled data if available
    if 'scaled' in adata_cluster.layers:
        adata_cluster.X = adata_cluster.layers['scaled']
    
    # Filter to present significant genes
    present_genes = [g for g in sig_genes if g in adata_cluster.var_names]
    if not present_genes:
        print(f"No present genes for {cluster} in data, skipping balloon plot.")
        continue
    
    # Create custom colormap: white to cluster-specific color
    cluster_color = custom_colors.get(cluster, '#FF0000')  # Fallback to red
    custom_cmap = LinearSegmentedColormap.from_list(
        f'white_to_{cluster}', ['#FFFFFF', cluster_color], N=256
    )
    
    # Generate balloon plot: genes on x, sample_code on y
    sc.pl.dotplot(
        adata_cluster,
        var_names=present_genes,
        groupby='sample_code',  # Use sample_code directly
        cmap=custom_cmap,  # White to cluster-specific color
        standard_scale='var',  # Normalize expression per gene
        save=f'Extended_Data_Fig_3e-h_{cluster}.pdf'
    )


# Extended Data Fig. 4

In [ ]:
# Extended Data Fig. 4a - GSEA
import pandas as pd
import os
import gseapy as gp
import matplotlib.pyplot as plt
import glob
import numpy as np

# ------------------------------------------------------------------
# 1. Only things you change
# ------------------------------------------------------------------
desired_clusters = ["CD8.Q"]   # full names
source_file = "data/ttlist_OneVsAll.txt"

# ------------------------------------------------------------------
# 2. Load
# ------------------------------------------------------------------
df = pd.read_csv(source_file, sep=None, engine="python", index_col=0)

print("All columns in source file:")
print(list(df.columns))
print()

# ------------------------------------------------------------------
# 3. Pull log2FC + corresponding adj.P.Val and filter
# ------------------------------------------------------------------
selected = {}

for cl in desired_clusters:
    # Find log2FC column
    fc_matches = [
        col for col in df.columns
        if cl in col and ("log2fc" in col.lower() or "log2foldchange" in col.lower())
    ]
    
    # Find adjusted p-value column
    padj_matches = [
        col for col in df.columns
        if cl in col and (
            "adj.p.val" in col.lower() or 
            "adj.pval" in col.lower() or 
            "padj" in col.lower() or
            "fdr" in col.lower()
        )
    ]
    
    if not fc_matches:
        print(f"WARNING: No log2FC column found for {cl}")
        continue
    if not padj_matches:
        print(f"WARNING: No adj.P.Val column found for {cl} → keeping all values (no filter)")
        selected[cl] = df[fc_matches[0]]
        continue
    
    fc_col = fc_matches[0]
    padj_col = padj_matches[0]
    
    print(f"Found {cl}")
    print(f"  log2FC → {fc_col}")
    print(f"  padj   → {padj_col}")
    
    # Keep log2FC only when adj.P.Val < 0.05
    filtered = df[fc_col].where(df[padj_col] < 0.05)
    selected[cl] = filtered

# ------------------------------------------------------------------
# 4. Build final table
# ------------------------------------------------------------------
if not selected:
    raise ValueError("No matching columns were found.")

log_fold_changes = pd.DataFrame(selected)

# Optional: drop genes that are non-significant in ALL selected clusters
log_fold_changes = log_fold_changes.dropna(how="all")

print("\nFinal shape (after adj.P.Val < 0.05 filter):", log_fold_changes.shape)
print(log_fold_changes.head())

# Convert the index of log_fold_changes to all uppercase
log_fold_changes.index = log_fold_changes.index.str.upper()
log_fold_changes

columns = log_fold_changes.columns
try:
    os.mkdir('gsea_preranks')
except:
    print('Directory already exists')

for i in columns:
    lfc_current = log_fold_changes[[i]]
    #strip whitespace in the index of lfc_current
    lfc_current.index = lfc_current.index.str.strip()
    #split any values in the index on ; and keep the first gene in the resulting list
    lfc_current.index = lfc_current.index.str.split(';').str[0]
    #drop na rows
    lfc_current = lfc_current.dropna()
    #make index uppercase
    lfc_current.index = lfc_current.index.str.upper()
    #drop rows with duplicate index
    lfc_current = lfc_current[~lfc_current.index.duplicated(keep='first')]



    lfc_current.sort_values(i, inplace=True, ascending = False)
    lfc_current.to_csv('gsea_preranks/'+i+'.rnk', sep='\t', header=False)

# File paths
file_paths = {
    'MackayRES': 'data/TRM_Mackay.txt',  # From Mackay et al., Science 2016
    'CrowlRES': 'data/TRM_Crowl.txt',    # From Crowl et al., Nat Immunol 2022
    'MilnerRES': 'data/TRM_Milner.txt'   # From Milner et al., Nature 2017
}

# Function to read gene list from file
def read_gene_list(file_path, is_excel=True):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    try:
        if is_excel:
            # Read Excel, assuming genes are in the first column, no header
            df = pd.read_excel(file_path, header=None, index_col=0)
        else:
            # Read text file, assuming one gene per line
            df = pd.read_csv(file_path, header=None, index_col=0)
        # Extract genes, convert to list, remove NaN, and convert to uppercase
        genes = [str(gene).upper() for gene in df.index if pd.notna(gene)]
        if not genes:
            raise ValueError(f"No valid genes found in {file_path}")
        return genes
    except Exception as e:
        raise ValueError(f"Error reading {file_path}: {e}")

# Read gene lists and store in dictionary
signatures = {}
for name, path in file_paths.items():
    is_excel = path.endswith('.xlsx')
    signatures[name] = read_gene_list(path, is_excel=is_excel)

# Print the signatures dictionary to verify
print(signatures)


In [ ]:
# Extended Data Fig. 4a - GSEA
all_results = []

# ====================== FIXED SETTINGS ======================
desired_order = ['MackayRES', 'CrowlRES', 'MilnerRES']

signature_colors = {
    'MackayRES': '#1f77b4',   # blue
    'CrowlRES':  '#ff7f0e',   # orange
    'MilnerRES': '#2ca02c',   # green
}
# ===========================================================

for cluster in ['CD8.Q']:
    print(f"Running GSEA for {cluster}...")   # helpful feedback
    
    pre_res = gp.prerank(
        rnk=f"gsea_preranks/{cluster}.rnk",
        gene_sets=signatures,
        threads=4,
        min_size=5,
        max_size=10000,
        permutation_num=1000,
        outdir=None,
        seed=6,
        verbose=True,
    )
    
    # === Get results for saving ===
    res_df = pre_res.res2d.copy()
    
    # Add cluster info and save all results (this was missing)
    res_df['Cluster'] = cluster
    all_results.append(res_df)          # <-- important
    
    # === Fixed order + colors for plotting ===
    res_df['order'] = res_df['Term'].apply(
        lambda x: next((i for i, sig in enumerate(desired_order) if sig in x), 999)
    )
    
    res_df = res_df.sort_values(by=['order', 'NES'], ascending=[True, False])
    
    plot_terms = res_df['Term'].head(len(desired_order)).tolist()
    
    plot_colors = []
    for t in plot_terms:
        assigned = False
        for sig, col in signature_colors.items():
            if sig in t:
                plot_colors.append(col)
                assigned = True
                break
        if not assigned:
            plot_colors.append('#7f7f7f')
    
    # Plot
    axs = pre_res.plot(
        terms=plot_terms,
        colors=plot_colors,
        legend_kws={'loc': (1.25, 0.5)},
        show_ranking=True,
        figsize=(4, 4.5),
    )
    
    plt.title(f'{cluster} — GSEA (fixed order & colors)')
    
    output_dir = os.path.join('gsea_figures', cluster)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(os.path.join(output_dir, f'Extended_Data_Fig_4a_{cluster}.pdf'),
                dpi=300, bbox_inches='tight')
    # plt.show()   # uncomment if you want to see it live

# ====================== SAVE ALL RESULTS ======================
if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    results_df.to_csv('Extended_Data_Fig_4a_results.csv', index=False)
    print(results_df)
else:
    print("No results were collected!")
    

In [ ]:
# Extended Data Fig. 4f - Transcription factor expression
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Your genes
genes = ["Hic1", "Zfp683", "Runx3"]

# Check which genes exist
available_genes = [g for g in genes if g in adata.var_names]
missing_genes = set(genes) - set(adata.var_names)

if missing_genes:
    print("Missing genes:", missing_genes)

if len(available_genes) == 0:
    raise ValueError("None of the requested genes were found!")

# Subset AnnData to available genes
adata_subset = adata[:, available_genes].copy()

# Extract normalized expression (handle sparse matrix)
X = adata_subset.layers["log_norm"]
if hasattr(X, "toarray"):          # convert sparse → dense if needed
    X = X.toarray()

expr_df = pd.DataFrame(
    X,
    columns=available_genes,
    index=adata_subset.obs_names
)

# Add grouping column
expr_df["cell_type"] = adata_subset.obs["cluster_annotation"].values

print("Expression shape:", expr_df.shape)
print("Unique cell types:", expr_df["cell_type"].nunique())
print(expr_df.head())

In [ ]:
# Extended Data Fig. 4f - Transcription factor expression
# Compute mean expression per cell type
mean_expr = expr_df.groupby('cell_type')[genes].mean()

# Optional: Add standard error for error bars
std_expr = expr_df.groupby('cell_type')[genes].std()
n_cells = expr_df.groupby('cell_type')[genes].count()  # Number of cells per group
sem_expr = std_expr / np.sqrt(n_cells)  # Standard error of the mean

print(mean_expr.head())  # Preview


In [ ]:
# Extended Data Fig. 4f - Transcription factor expression
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Melt mean and SEM
mean_long = mean_expr.reset_index().melt(
    id_vars='cell_type', var_name='Gene', value_name='Expression'
)

sem_long = sem_expr.reset_index().melt(
    id_vars='cell_type', var_name='Gene', value_name='SEM'
)

# 2. Merge them
plot_df = mean_long.merge(sem_long, on=['cell_type', 'Gene'])

# 3. Plot
plt.figure(figsize=(10, 5.5))
ax = sns.barplot(
    data=plot_df,
    x='Gene',
    y='Expression',
    hue='cell_type',
    palette=custom_colors,
    errorbar=None,          # we will add error bars ourselves
    edgecolor='black',
    linewidth=0.6
)

# 4. Add pre-computed SEM error bars
# Get the grouped bars
n_genes = len(genes)
n_celltypes = len(plot_df['cell_type'].unique())
bar_width = 0.8 / n_celltypes

for i, gene in enumerate(genes):
    gene_data = plot_df[plot_df['Gene'] == gene]
    
    for j, (_, row) in enumerate(gene_data.iterrows()):
        # Approximate x position of each bar
        x = i + (j - n_celltypes/2) * bar_width + bar_width/2
        ax.errorbar(
            x=x,
            y=row['Expression'],
            yerr=row['SEM'],
            fmt='none',
            c='black',
            capsize=3,
            elinewidth=1
        )

# Styling
plt.title('Average Expression of TFs by Cell Type')
plt.ylabel('Average Expression (log-normalized)')
plt.xlabel('Transcription Factor')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.grid(axis='y', color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.savefig('Extended_Data_Fig_4f.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Extended Data Fig. 4g - CD8.Q, T, wV frequency across samples
import pandas as pd
import matplotlib.pyplot as plt

# Load the Excel file
file_path = 'data/immgenT-CD8.xlsx'
df = pd.read_excel(file_path)

print(f"Loaded Excel file with {df.shape[0]} rows and {df.shape[1]} columns.")

# Define columns
prop_cols = ['CD8.Q.prop', 'CD8.T.prop', 'CD8.wV.prop']
ncells_cols = ['CD8.Q.ncells', 'CD8.T.ncells', 'CD8.wV.ncells']

# === FILTERS (applied on original rows before averaging) ===
if 'target_cells' in df.columns:
    df = df[~df['target_cells'].str.contains('CD4\+ CD44\+', case=False, na=False)].copy()
    unwanted = ['Treg', 'CD4+ T', 'DN', 'DO11.10 KJ1-26', 'OT2', 'TCRBV-TCRGD', 'TFH',
                'Vd6b F4.22', 'Vg4 49.2', 'Vg6 1C10', 'Vg7 F2.67', 'thymocytes']
    mask = df['target_cells'].isin(unwanted) | df['target_cells'].str.contains('|'.join(unwanted), case=False, na=False)
    df = df[~mask].copy()
    print(f"After target_cells filters: {df.shape[0]} rows remain.")

# Exclude KO conditions
exclude_conditions = ['KbDbKO', 'KbDbQa1KO', 'KbDbQa1KO_MCMV']
exclude_mask = df['condition_detailed_organ'].str.contains('|'.join(exclude_conditions), case=False, na=False)
df = df[~exclude_mask].copy()
print(f"After excluding KbDb conditions: {df.shape[0]} rows remain.")

# Exclude baseline
if 'condition_detailed_simplified' in df.columns:
    df = df[df['condition_detailed_simplified'] != 'baseline'].copy()
    print(f"After excluding baseline: {df.shape[0]} rows remain.")

# === AGGREGATE REPLICATES (Group by condition_detailed_organ) ===
group_cols = ['condition_detailed_organ']

agg_dict = {col: 'mean' for col in prop_cols + ncells_cols}
if 'target_cells' in df.columns:
    agg_dict['target_cells'] = 'first'   # keep one value for reference

df_agg = df.groupby(group_cols, as_index=False).agg(agg_dict)

print(f"After aggregating replicates: {df_agg.shape[0]} unique conditions.")

# === FILTER on averaged ncells: ≥10 in at least one cluster ===
min_cells = 10
valid_mask = (df_agg['CD8.Q.ncells'] >= min_cells) | \
             (df_agg['CD8.T.ncells'] >= min_cells) | \
             (df_agg['CD8.wV.ncells'] >= min_cells)

df_filtered = df_agg[valid_mask].copy()
print(f"After ≥ {min_cells} cells (averaged) in at least one cluster: {df_filtered.shape[0]} conditions remain.")

# Calculate total proportion for ranking
df_filtered['total_prop'] = df_filtered[prop_cols].sum(axis=1)
df_filtered = df_filtered[df_filtered['total_prop'] > 0].copy()

# Select top 50 unique conditions
df_top50 = df_filtered.nlargest(50, 'total_prop')

print(f"Selected top 50 unique conditions by average combined proportion.")

# Prepare for plotting
freq_table = df_top50[prop_cols].copy()
freq_table.index = df_top50['condition_detailed_organ']
freq_table.columns = ['CD8.Q', 'CD8.T', 'CD8.wV']

# Plot using your pre-existing custom_colors
fig, ax = plt.subplots(figsize=(14, 8))
freq_table.plot(kind='bar', stacked=True, ax=ax, 
                color=[custom_colors[col] for col in freq_table.columns])

ax.set_xlabel('Condition (Detailed Organ)')
ax.set_ylabel('Percentage within each sample (%)')
ax.set_title('Top 50 Unique Conditions by CD8.Q + CD8.T + CD8.wV Proportion\n(Averaged across replicates • After all filters)')
ax.grid(False)
ax.legend(title='Cluster', bbox_to_anchor=(1, 1), loc='upper left')

plt.xticks(rotation=90, ha='center')
plt.tight_layout()

# Save
plt.savefig('Extended_Data_Fig_4g.pdf', bbox_inches='tight')
plt.show()
plt.close()

print("Stacked bar plot saved: Extended_Data_Fig_4g.pdf")

# Preview
print("\nTop 10 unique conditions:")
print(df_top50[['condition_detailed_organ', 'total_prop'] + ncells_cols + prop_cols].head(10))


In [ ]:
# Extended Data Fig. 4h - Dot plot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc  # Explicitly import scanpy for clarity

# Define selected genes
selected_genes = ["Tcf7", "Lef1", "Bach2", "Id3", "Zeb1", "Sell", "Il7r", "Tbx21", "Cd44", "Zeb2",
                  "Id2", "Klrg1", "Klrd1", "Itgax", "Cx3cr1", "Cd160", "Pecam1", "Gzma", "Gzmk",
                  "Gzmb", "Ifng", "Prf1", "Entpd1", "Nt5e", "Runx3", "Hic1", "Itgae", "Cd69",
                  "P2rx7", "Itga1", "Xcl1", "Cxcr6", "Ccr9", "Prdm1", "Tox", "Pdcd1", "Havcr2",
                  "Tnfrsf18", "Mki67"]

# Subset mdata['RNA'] to include only clusters CD8.Q, CD8.T, and CD8.wV
subset_adata = mdata['RNA'][mdata['RNA'].obs['cluster_annotation'].isin(['CD8.Q', 'CD8.T', 'CD8.wV']), :]

# Create dot plot with the subsetted data
sc.pl.dotplot(subset_adata, var_names=list(selected_genes), groupby='cluster_annotation',
              use_raw=False,  # Set to True if .X is raw and you have .raw
              cmap='Reds',    # Color map for expression
              title='Differentially Expressed Genes (RNA)',
              standard_scale='var',  # Scale by variable (gene)
              figsize=(11.5, 1),      # Try passing figsize directly
              show=False)
plt.savefig('Extended_Data_Fig_4h.pdf', format='pdf', bbox_inches='tight')
plt.show()
plt.close()
print('Plot saved: Extended_Data_Fig_4h.pdf')


# Extended Data Fig. 5

In [ ]:
# Extended Data Fig. 5c — MNVCR6 MDE plots
color_palette = {
    'MNVCR6_infant': "#1e8f0f",  # Blue
    'MNVCR6_adult': '#ff7f0e',    # Orange
    'MNVCR6': '#d3d3d3'          # Gray for other categories (e.g., MNVCR6)
    # Add other categories in condition_detailed if necessary
}

# Filter for IGT == IGT26, IGT79, IGT80 and condition_broad == 'virus'
subset_conditions = rna[
    (rna.obs['IGT'].isin(['IGT26', 'IGT79', 'IGT80'])) & 
    (rna.obs['condition_broad'] == 'virus') &
    (rna.obs['condition_detailed_organ'].str.contains('MNVCR6', na=False))
].copy()

# Create figure and axis
fig, ax = plt.subplots(figsize=(5.5, 4.5))

# Plot all cells in gray as background
sc.pl.embedding(
    rna,
    basis='MDE_INCREMENTAL',
    color='condition_broad',
    palette=['#d3d3d3'],  # Gray color for all cells
    size=1,  # Smaller point size for background
    show=False,
    ax=ax,
    legend_loc=None  # Remove legend
)

# Overlay cells from specified conditions
sc.pl.embedding(
    subset_conditions,
    basis='MDE_INCREMENTAL',
    color='condition_detailed',  # Color by condition_detailed
    groups=['MNVCR6_adult', 'MNVCR6_infant'],  # Ensure only specified conditions are colored
    palette=color_palette,  # Use defined color palette
    title='Highlighted Conditions (MNVCR6_infant, MNVCR6_adult) Colored by Condition_Detailed',
    legend_loc='right margin',
    size=20,  # Larger point size for foreground
    show=False,
    ax=ax
)

# Save and show the plot
plt.tight_layout()
plt.savefig("Extended_Data_Fig_5c.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Plot saved: Extended_Data_Fig_5c.pdf")


In [ ]:
# Extended Data Fig. 5c — MNVCR6 MDE plots
color_palette = {
    'MNVCR6_infant': "#1e8f0f",  # Green
    'MNVCR6_adult': '#ff7f0e',    # Orange
    'MNVCR6': '#d3d3d3'          # Gray for other categories (e.g., MNVCR6)
    # Add other categories in condition_detailed if necessary
}

# Filter for IGT == IGT26, IGT79, IGT80 and condition_broad == 'virus'
subset_conditions = rna[
    (rna.obs['IGT'].isin(['IGT26', 'IGT79', 'IGT80'])) & 
    (rna.obs['condition_broad'] == 'virus') &
    (rna.obs['condition_detailed_organ'].str.contains('MNVCR6', na=False)) &
    (rna.obs['Ag_spe_v1'] == 'VP1Tetp')
].copy()

# Create figure and axis
fig, ax = plt.subplots(figsize=(5.5, 4.5))

# Plot all cells in gray as background
sc.pl.embedding(
    rna,
    basis='MDE_INCREMENTAL',
    color='condition_broad',
    palette=['#d3d3d3'],  # Gray color for all cells
    size=1,  # Smaller point size for background
    show=False,
    ax=ax,
    legend_loc=None  # Remove legend
)

# Overlay cells from specified conditions
sc.pl.embedding(
    subset_conditions,
    basis='MDE_INCREMENTAL',
    color='condition_detailed',  # Color by condition_detailed
    groups=['MNVCR6_adult', 'MNVCR6_infant'],  # Ensure only specified conditions are colored
    palette=color_palette,  # Use defined color palette
    title='Highlighted Conditions (MNVCR6_infant, MNVCR6_adult) Colored by Condition_Detailed',
    legend_loc='right margin',
    size=20,  # Larger point size for foreground
    show=False,
    ax=ax
)

# Save and show the plot
plt.tight_layout()
plt.savefig("Extended_Data_Fig_5c_v2.pdf", bbox_inches="tight")
plt.show()
plt.close()
print("Plot saved: Extended_Data_Fig_5c_v2.pdf")


In [ ]:
# Extended Data Fig. 5d - Feature plots
import matplotlib.pyplot as plt
import numpy as np
import muon as mu

# Set up embedding plot
with plt.rc_context():
    # ← ONLY THIS LINE CHANGED (your 6 genes)
    genes = ["Lag3", "Tigit", "Pdcd1", "Tox"]
    
    # ← Updated to match the 6 genes (one entry per gene)
    legend_locs = ["right margin"] * len(genes)   # all colorbars on the right
    
    # ← Your preferred fixed scale (same as original Cd44/Sell example)
    vmin_values = [0] * len(genes)    # all start at 0
    vmax_values = [5] * len(genes)    # all capped at 5 (you liked this before)

    # Editable figure size parameters
    base_width = 3.5   # Base width per column
    base_height = 4    # Base height per row

    # Calculate subplot grid size dynamically
    n_plots = len(genes)
    n_cols = 3         # ← Changed to 3 columns → perfect 2×3 grid for 6 genes
    n_rows = (n_plots + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(base_width * n_cols, base_height * n_rows))
    axes = axes.flatten() if n_plots > 1 else [axes]

    # ← Only this loop was kept exactly as you wrote it
    for ax, gene, legend_loc, vmin, vmax in zip(axes, genes, legend_locs, vmin_values, vmax_values):
        mu.pl.embedding(
            mdata['RNA'],
            basis="MDE_INCREMENTAL",
            color=gene,
            cmap="viridis",         # kept your original cmap
            legend_loc=legend_loc,  # colorbar on right margin
            ax=ax,
            size=2,                 # kept your original small dot size
            vmin=vmin,
            vmax=vmax,
            show=False,
        )

    # Remove unused subplots (none in this case, but kept for safety)
    for ax in axes[n_plots:]:
        fig.delaxes(ax)

    plt.tight_layout(w_pad=0.55)
    plt.savefig("Extended_Data_Fig_5d.pdf", bbox_inches="tight")  # ← changed filename to match your figure
    plt.show()
    plt.close()
    print("Plot saved: Extended_Data_Fig_5d.pdf")


In [ ]:
# Extended Data Fig. 5e — exhaustion markers across gene programs
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Load the gene x factor matrix
# ------------------------------------------------------------------
F_df = pd.read_csv(
    "data/gene_factor_matrix.txt",
    sep="\t",
    index_col=0
)
F_df.index = F_df.index.astype(str).str.strip()

# ------------------------------------------------------------------
# Markers
# ------------------------------------------------------------------
marker_genes = ["Lag3", "Tigit", "Pdcd1", "Tox"]
available = [g for g in marker_genes if g in F_df.index]
data = F_df.loc[available]

# ------------------------------------------------------------------
# Optional: only keep factors with decent signal
# ------------------------------------------------------------------
min_abs_weight = 0.05
has_signal = (data.abs() > min_abs_weight).any(axis=0)
data = data.loc[:, has_signal]

# ------------------------------------------------------------------
# Sort by exhaustion strength
# ------------------------------------------------------------------
exhaustion_score = data.clip(lower=0).mean()
data = data[exhaustion_score.sort_values(ascending=False).index]

# THIS IS THE ONLY NEW LINE — replaces "V" with "GP" in column names
data.columns = data.columns.str.replace(r'^V', 'GP', regex=True)

# ------------------------------------------------------------------
# Plot!
# ------------------------------------------------------------------
plt.figure(figsize=(20, 3))
cmap = "coolwarm"

ax = sns.heatmap(
    data,
    cmap=cmap,
    center=0,
    annot=False,
    cbar_kws={"label": "Gene weight in factor", "shrink": 0.7},
    yticklabels=data.index,
    xticklabels=data.columns
)

ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=14)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=9)

plt.title("Exhaustion markers (Lag3, Tigit, Pdcd1, Tox) across all gene programs\n"
          "Red = upregulated when factor is active | Blue = downregulated",
          fontsize=14, pad=20)
plt.ylabel("Gene")
plt.xlabel("Factor")
ax.grid(False)

plt.tight_layout()
plt.savefig("Extended_Data_Fig_5e.pdf", bbox_inches='tight', dpi=300)
plt.show()

print("Saved heatmap with GP labels → Extended_Data_Fig_5e.pdf")


In [ ]:
# Extended Data Fig. 5f — exhaustion markers across gene programs
import pandas as pd

# ============================= USER SETTINGS =============================
marker_genes = ["Lag3", "Tigit", "Pdcd1", "Tox"]   # change/add genes here if you want
min_positive_weight = 0.08                         # threshold for "positive"
min_markers = 4                                    # ←←← CHANGE ONLY THIS NUMBER (1 to 4)
# ======================================================================

# Load matrix
F_df = pd.read_csv(
    "data/gene_factor_matrix.txt",
    sep="\t",
    index_col=0
)
F_df.index = F_df.index.astype(str).str.strip()

# Check which markers exist
available = [g for g in marker_genes if g in F_df.index]
missing   = [g for g in marker_genes if g not in F_df.index]

print("Available markers :", available)
if missing:
    print("Missing markers   :", missing)

if len(available) == 0:
    raise ValueError("None of the marker genes were found!")

# Boolean mask: True if weight > threshold
mask = F_df.loc[available] > min_positive_weight

# Count how many markers are positive per factor
n_positive_per_factor = mask.sum(axis=0)   # <-- key line

# Apply the user-defined minimum
selected = n_positive_per_factor >= min_markers
selected_factors = selected[selected].index.tolist()

print(f"\n→ Found {len(selected_factors)} factor(s) with "
      f"at least {min_markers}/{len(available)} markers > {min_positive_weight}")
print("Factors:", ", ".join(sorted(selected_factors)) if selected_factors else "None")

# Show detailed table
if selected_factors:
    detail_df = F_df.loc[available, selected_factors].round(4)
    
    # Sort by total positive score (nice order)
    detail_df = detail_df.loc[:, detail_df.sum().sort_values(ascending=False).index]
    
    print(f"\nPrograms with ≥{min_markers} positive exhaustion markers:")
    print(detail_df)
    
    # Save
    filename = f"Extended_Data_Fig_5f_exhaustion_min_{min_markers}_of_{len(available)}_markers_positive.tsv"
    detail_df.to_csv(filename, sep="\t")
    print(f"\nSaved → {filename}")
else:
    print(f"\nNo factor found with ≥{min_markers} markers above {min_positive_weight}.")
    print("Try lowering min_positive_weight or min_markers.")


In [ ]:
# Extended Data Fig. 5f — exhaustion GP across clusters
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re

# -------------------------- USER SETTINGS --------------------------
cluster_column = "cluster_annotation"
cell_id_column = None  # auto-detect below
clusters_to_show = ["CD8.Q", "CD8.R", "CD8.S"]
# -----------------------------------------------------------------

# Load cell × factor matrix
L_df = pd.read_csv(
    "data/cell_factor_matrix.txt",
    sep="\t",
    index_col=0
)

# Load annotations
obs = mdata['RNA'].obs

# Auto-detect cell ID column
if 'cellID' in obs.columns:
    cell_id_col = 'cellID'
elif 'IGT_cellID' in obs.columns:
    cell_id_col = 'IGT_cellID'
else:
    raise KeyError("No known cell ID column found!")

cell_to_cluster = obs.set_index(cell_id_col)[cluster_column]

# Align with L matrix
common_cells = L_df.index.intersection(cell_to_cluster.index)
L_df = L_df.loc[common_cells]
cell_to_cluster = cell_to_cluster.loc[common_cells]

# ------------------- Match selected factors robustly -------------------
def get_factor_number(name):
    match = re.search(r'\d+', str(name))
    return int(match.group()) if match else -1

selected_nums = [get_factor_number(f) for f in selected_factors]
L_cols_nums = [get_factor_number(c) for c in L_df.columns]

matched_factors = []
for num in selected_nums:
    cols = [c for c, n in zip(L_df.columns, L_cols_nums) if n == num]
    if cols:
        matched_factors.append(cols[0])
    else:
        print(f"Warning: Factor ~{num} not found in L matrix")

if not matched_factors:
    raise ValueError("No selected factors found in cell loading matrix!")

print(f"Plotting stacked activity for: {matched_factors}")

# ------------------- Compute mean loading per cluster -------------------
plot_data = []
for factor in matched_factors:
    for cluster in clusters_to_show:
        if cluster not in cell_to_cluster.values:
            continue
        cells_in_cluster = cell_to_cluster[cell_to_cluster == cluster].index
        mean_load = L_df.loc[cells_in_cluster, factor].mean()
        plot_data.append({
            "Cluster": cluster,
            "Factor": factor,
            "Mean Loading": mean_load
        })

df = pd.DataFrame(plot_data)

# Pivot for stacked plot
df_pivot = df.pivot(index="Cluster", columns="Factor", values="Mean Loading").fillna(0)
df_pivot = df_pivot.loc[clusters_to_show]  # enforce order

# ------------------- Stacked barplot -------------------
plt.figure(figsize=(3, 4))
colors = sns.color_palette("Pastel1", n_colors=len(matched_factors))  # try also: Pastel2, Set2, Set3, husl

df_pivot.plot(
    kind="bar",
    stacked=True,
    color=colors,
    edgecolor="black",
    linewidth=1.2,
    width=0.65,
    ax=plt.gca()
)

plt.title(f"Canonical Exhaustion Program Activity\n"
          f"in CD8.Q, R, S (n={len(matched_factors)} factor(s) stacked)",
          fontsize=14, pad=20)
plt.xlabel("")
plt.ylabel("Total mean factor loading (stacked)", fontsize=12)
plt.legend(title="Factor", bbox_to_anchor=(1.05, 1), loc="upper left")

# THIS LINE MAKES X LABELS VERTICAL
plt.xticks(rotation=90, fontsize=12)

plt.grid(False)
sns.despine(trim=True)

plt.tight_layout()
plt.savefig("Extended_Data_Fig_5f.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Extended Data Fig. 5g-i - Volcano plots
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from adjustText import adjust_text

# Define custom colors for clusters
custom_colors = {
    'CD8.R': '#FF69B4',   # pink2 (hotpink)
    'CD8.Q': '#76EE00',  # chartreuse2
    'CD8.S': '#CD950C',  # darkgoldenrod2
}

def plot_volcano(
    signature_file,
    cluster,
    pval_threshold=0.05,
    log2fc_threshold=0.5,
    non_highlight_size=1,
    highlight_size=5,
    label_top_n=10,
    figsize=(5, 5),
    save_path=None,
    highlight_gene_column='SYMBOL',
    xlim_min=-3,
    xlim_max=4,
    highlight_genes=None,
    highlight_label=None,
    extra_label_genes=None
):
    """
    Generate a volcano plot from a signature DataFrame for a specific cluster with staggered gene labels.
    Labels the top 10 up-regulated and top 10 down-regulated significant genes (based on pval and log2FC thresholds).
    Optionally highlights genes from an external list.
    Optionally labels user-specified genes in red (extra_label_genes).
    """
    if extra_label_genes is None:
        extra_label_genes = []

    # Load the signature DataFrame
    signature_df = pd.read_csv(signature_file, sep='\t', encoding='utf-8')

    # Prepare data for volcano plot
    log2fc_col = f'log2FC_{cluster}_vs_All'
    pval_col = f'adj.P.Val_{cluster}_vs_All'

    plot_df = signature_df[[highlight_gene_column, log2fc_col, pval_col]].copy()
    plot_df['gene'] = plot_df[highlight_gene_column]
    plot_df['log2FC'] = plot_df[log2fc_col]
    plot_df['minus_log10_padj'] = -np.log10(plot_df[pval_col].clip(lower=1e-300))

    # Highlight based on significance thresholds or external gene list
    if highlight_genes is None:
        plot_df['highlight'] = (plot_df[pval_col] < pval_threshold) & (abs(plot_df['log2FC']) > log2fc_threshold)
    else:
        plot_df['highlight'] = plot_df['gene'].isin(highlight_genes)

    # Create the volcano plot
    fig, ax = plt.subplots(figsize=figsize)

    # Plot non-significant genes
    non_highlight = plot_df[~plot_df['highlight']]
    ax.scatter(
        non_highlight['log2FC'],
        non_highlight['minus_log10_padj'],
        color='grey',
        alpha=0.5,
        s=non_highlight_size,
        rasterized=True
    )

    # Plot highlighted genes with cluster-specific color
    highlight = plot_df[plot_df['highlight']]
    ax.scatter(
        highlight['log2FC'],
        highlight['minus_log10_padj'],
        color=custom_colors.get(cluster, '#FF0000'),
        alpha=1,
        s=highlight_size,
        rasterized=True
    )

    # Determine significant genes for top labeling
    significant = plot_df[(plot_df[pval_col] < pval_threshold) & (abs(plot_df['log2FC']) > log2fc_threshold)]
    if highlight_genes is not None:
        significant = significant[significant['gene'].isin(highlight_genes)]

    # Top up/down genes
    top_up = significant[significant['log2FC'] > 0][['gene', 'log2FC', 'minus_log10_padj']].sort_values(
        'log2FC', ascending=False
    ).head(label_top_n)

    top_down = significant[significant['log2FC'] < 0][['gene', 'log2FC', 'minus_log10_padj']].sort_values(
        'log2FC', ascending=True
    ).head(label_top_n)

    top_highlight = pd.concat([top_up, top_down])

    # Add user-specified genes to label (in red)
    extra_df = plot_df[plot_df['gene'].isin(extra_label_genes)]
    extra_to_label = extra_df[~extra_df['gene'].isin(top_highlight['gene'])]

    texts = []

    # Label top genes (cluster color)
    for _, row in top_highlight.iterrows():
        text = ax.text(
            row['log2FC'],
            row['minus_log10_padj'],
            row['gene'],
            fontsize=6,
            color='black',
            ha='right' if row['log2FC'] < 0 else 'left'
        )
        texts.append(text)

    # Label extra user genes (in RED)
    for _, row in extra_to_label.iterrows():
        text = ax.text(
            row['log2FC'],
            row['minus_log10_padj'],
            row['gene'],
            fontsize=6,
            color='red',
            ha='right' if row['log2FC'] < 0 else 'left',
            weight='bold'
        )
        texts.append(text)

    # Adjust text to avoid overlap
    adjust_text(
        texts,
        arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
        expand_points=(1.2, 1.2),
        force_points=0.5,
        force_text=0.5
    )

    # Axes and title
    ax.set_xlabel(f'Log2 Fold Change ({cluster} vs Others)')
    ax.set_ylabel('-Log10 (Adjusted P-value)')
    title = f'Volcano Plot: {cluster} vs All Others'
    if highlight_label:
        title += f' ({highlight_label} Genes)'
    else:
        title += ' (Signature Genes)'
    ax.set_title(title)

    # Set x-axis limits
    ax.set_xlim(xlim_min, xlim_max)

    # Add significance lines
    ax.axvline(x=log2fc_threshold, color='black', linestyle='--', linewidth=0.5)
    ax.axvline(x=-log2fc_threshold, color='black', linestyle='--', linewidth=0.5)
    ax.axhline(y=-np.log10(pval_threshold), color='black', linestyle='--', linewidth=0.5)

    # Adjust layout
    plt.tight_layout()

    # Save or display
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()

# ================================================================== #
# Usage
# ================================================================== #
main_signature_file = 'data/ttlist_OneVsAll.txt'
clusters = ['CD8.R', 'CD8.Q', 'CD8.S']

for cluster in clusters:
    plot_volcano(
        signature_file=main_signature_file,
        cluster=cluster,
        pval_threshold=0.05,
        log2fc_threshold=0.5,
        non_highlight_size=1,
        highlight_size=5,
        label_top_n=10,
        figsize=(5, 5),
        save_path=f'Extended_Data_Fig_5g-i_{cluster}.pdf',
        highlight_gene_column='SYMBOL',
        xlim_min=-3,
        xlim_max=4
    )

In [ ]:
# Extended Data Fig. 5h,i - GSEA
import pandas as pd
import os
import gseapy as gp
import matplotlib.pyplot as plt
import glob
import numpy as np

# ------------------------------------------------------------------
# 1. Only things you change
# ------------------------------------------------------------------
desired_clusters = ["CD8.R", "CD8.S"]   # full names
source_file = "data/ttlist_OneVsAll.txt"

# ------------------------------------------------------------------
# 2. Load
# ------------------------------------------------------------------
df = pd.read_csv(source_file, sep=None, engine="python", index_col=0)

print("All columns in source file:")
print(list(df.columns))
print()

# ------------------------------------------------------------------
# 3. Pull log2FC + corresponding adj.P.Val and filter
# ------------------------------------------------------------------
selected = {}

for cl in desired_clusters:
    # Find log2FC column
    fc_matches = [
        col for col in df.columns
        if cl in col and ("log2fc" in col.lower() or "log2foldchange" in col.lower())
    ]
    
    # Find adjusted p-value column
    padj_matches = [
        col for col in df.columns
        if cl in col and (
            "adj.p.val" in col.lower() or 
            "adj.pval" in col.lower() or 
            "padj" in col.lower() or
            "fdr" in col.lower()
        )
    ]
    
    if not fc_matches:
        print(f"WARNING: No log2FC column found for {cl}")
        continue
    if not padj_matches:
        print(f"WARNING: No adj.P.Val column found for {cl} → keeping all values (no filter)")
        selected[cl] = df[fc_matches[0]]
        continue
    
    fc_col = fc_matches[0]
    padj_col = padj_matches[0]
    
    print(f"Found {cl}")
    print(f"  log2FC → {fc_col}")
    print(f"  padj   → {padj_col}")
    
    # Keep log2FC only when adj.P.Val < 0.05
    filtered = df[fc_col].where(df[padj_col] < 0.05)
    selected[cl] = filtered

# ------------------------------------------------------------------
# 4. Build final table
# ------------------------------------------------------------------
if not selected:
    raise ValueError("No matching columns were found.")

log_fold_changes = pd.DataFrame(selected)

# Optional: drop genes that are non-significant in ALL selected clusters
log_fold_changes = log_fold_changes.dropna(how="all")

print("\nFinal shape (after adj.P.Val < 0.05 filter):", log_fold_changes.shape)
print(log_fold_changes.head())

# Convert the index of log_fold_changes to all uppercase
log_fold_changes.index = log_fold_changes.index.str.upper()
log_fold_changes

columns = log_fold_changes.columns
try:
    os.mkdir('gsea_preranks')
except:
    print('Directory already exists')

for i in columns:
    lfc_current = log_fold_changes[[i]]
    #strip whitespace in the index of lfc_current
    lfc_current.index = lfc_current.index.str.strip()
    #split any values in the index on ; and keep the first gene in the resulting list
    lfc_current.index = lfc_current.index.str.split(';').str[0]
    #drop na rows
    lfc_current = lfc_current.dropna()
    #make index uppercase
    lfc_current.index = lfc_current.index.str.upper()
    #drop rows with duplicate index
    lfc_current = lfc_current[~lfc_current.index.duplicated(keep='first')]



    lfc_current.sort_values(i, inplace=True, ascending = False)
    lfc_current.to_csv('gsea_preranks/'+i+'.rnk', sep='\t', header=False)

# File paths
file_paths = {
    'MackayRES': 'data/TRM_Mackay.txt',  # From Mackay et al., Science 2016
    'CrowlRES': 'data/TRM_Crowl.txt',    # From Crowl et al., Nat Immunol 2022
    'MilnerRES': 'data/TRM_Milner.txt'   # From Milner et al., Nature 2017
}

# Function to read gene list from file
def read_gene_list(file_path, is_excel=True):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    try:
        if is_excel:
            # Read Excel, assuming genes are in the first column, no header
            df = pd.read_excel(file_path, header=None, index_col=0)
        else:
            # Read text file, assuming one gene per line
            df = pd.read_csv(file_path, header=None, index_col=0)
        # Extract genes, convert to list, remove NaN, and convert to uppercase
        genes = [str(gene).upper() for gene in df.index if pd.notna(gene)]
        if not genes:
            raise ValueError(f"No valid genes found in {file_path}")
        return genes
    except Exception as e:
        raise ValueError(f"Error reading {file_path}: {e}")

# Read gene lists and store in dictionary
signatures = {}
for name, path in file_paths.items():
    is_excel = path.endswith('.xlsx')
    signatures[name] = read_gene_list(path, is_excel=is_excel)

# Print the signatures dictionary to verify
print(signatures)


In [ ]:
# Extended Data Fig. 5h,i - GSEA
all_results = []

# ====================== FIXED SETTINGS ======================
desired_order = ['MackayRES', 'CrowlRES', 'MilnerRES']

signature_colors = {
    'MackayRES': '#1f77b4',   # blue
    'CrowlRES':  '#ff7f0e',   # orange
    'MilnerRES': '#2ca02c',   # green
}
# ===========================================================

for cluster in ['CD8.R', 'CD8.S']:
    print(f"Running GSEA for {cluster}...")   # helpful feedback
    
    pre_res = gp.prerank(
        rnk=f"gsea_preranks/{cluster}.rnk",
        gene_sets=signatures,
        threads=4,
        min_size=5,
        max_size=10000,
        permutation_num=1000,
        outdir=None,
        seed=6,
        verbose=True,
    )
    
    # === Get results for saving ===
    res_df = pre_res.res2d.copy()
    
    # Add cluster info and save all results (this was missing)
    res_df['Cluster'] = cluster
    all_results.append(res_df)          # <-- important
    
    # === Fixed order + colors for plotting ===
    res_df['order'] = res_df['Term'].apply(
        lambda x: next((i for i, sig in enumerate(desired_order) if sig in x), 999)
    )
    
    res_df = res_df.sort_values(by=['order', 'NES'], ascending=[True, False])
    
    plot_terms = res_df['Term'].head(len(desired_order)).tolist()
    
    plot_colors = []
    for t in plot_terms:
        assigned = False
        for sig, col in signature_colors.items():
            if sig in t:
                plot_colors.append(col)
                assigned = True
                break
        if not assigned:
            plot_colors.append('#7f7f7f')
    
    # Plot
    axs = pre_res.plot(
        terms=plot_terms,
        colors=plot_colors,
        legend_kws={'loc': (1.25, 0.5)},
        show_ranking=True,
        figsize=(4, 4.5),
    )
    
    plt.title(f'{cluster} — GSEA (fixed order & colors)')
    
    output_dir = os.path.join('gsea_figures', cluster)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(os.path.join(output_dir, f'Extended_Data_Fig_5h-i_{cluster}.pdf'),
                dpi=300, bbox_inches='tight')
    plt.show()   # uncomment if you want to see it live

# ====================== SAVE ALL RESULTS ======================
if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    results_df.to_csv('Extended_Data_Fig_5h-i_results.csv', index=False)
    print(results_df)
else:
    print("No results were collected!")

In [ ]:
# Extended Data Fig. 5j - MDE plot
import scanpy as sc
import matplotlib.pyplot as plt

# Colors for OT-I groups (lung KP vs lung flu comparison)
color_palette = {
    'OT1_lung_KP':   "#8f0f0f",  # Deep red
    'OT1_lung_flu':  "#65d8d8",  # Cyan
}

# Create a temporary column in the subset for plotting purposes
subset = rna[
    (rna.obs['Ag_spe_v1'] == 'OT1') & (rna.obs['cluster_annotation'] == 'CD8.Q') &
    (
        (rna.obs['condition_detailed_organ'] == 'lung_KP') |
        (rna.obs['condition_detailed_organ'] == 'lung_flu')
    )
].copy()

# Add a new column that distinguishes the two conditions
subset.obs['OT1_group'] = subset.obs['condition_detailed_organ'].map({
    'lung_KP':  'OT1_lung_KP',
    'lung_flu': 'OT1_lung_flu'
}).astype('category')

# Create figure
fig, ax = plt.subplots(figsize=(5.5, 4.5))

# 1. Plot all cells in gray as background
sc.pl.embedding(
    rna,
    basis='MDE_INCREMENTAL',
    color='condition_broad',
    palette=['#d3d3d3'],   # light gray
    size=1,
    show=False,
    ax=ax,
    legend_loc=None
)

# 2. Overlay the two OT-I populations with different colors
sc.pl.embedding(
    subset,
    basis='MDE_INCREMENTAL',
    color='OT1_group',
    palette=color_palette,
    groups=['OT1_lung_KP', 'OT1_lung_flu'],  # ensures order and legend
    size=15,                                 # make highlighted cells stand out
    title='OT-I cells in lung_KP and lung_flu',
    legend_loc='right margin',
    show=False,
    ax=ax
)

# Finalize and save
plt.tight_layout()
plt.savefig("Extended_Data_Fig_5j.pdf", bbox_inches="tight")
plt.show()
plt.close()

print("Plot saved: Extended_Data_Fig_5j.pdf")


In [ ]:
# Extended Data Fig. 5k - Volcano plot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text
import scanpy as sc

# === CONFIGURATION ===
color_kp_higher   = "#8f0f0f"   # Deep red
color_flu_higher  = "#65d8d8"   # Cyan
save_path = 'Extended_Data_Fig_5k.pdf'

# Thresholds
pval_threshold = 0.05
log2fc_threshold = 0.5
label_top_n = 5  # ← now fully respected

# Genes you always want labeled
extra_label_genes = []

# =====================
# Differential expression
subset_for_de = rna[
    (rna.obs['Ag_spe_v1'] == 'OT1') &
    (rna.obs['cluster_annotation'] == 'CD8.Q') &
    (rna.obs['condition_detailed_organ'].isin(['lung_KP', 'lung_flu']))
].copy()

subset_for_de.obs['de_group'] = subset_for_de.obs['condition_detailed_organ'].map({
    'lung_KP': 'KP',
    'lung_flu': 'Flu'
}).astype('category')

sc.tl.rank_genes_groups(
    subset_for_de,
    groupby='de_group',
    method='wilcoxon',
    key_added='de_KP_vs_Flu'
)

de_results = sc.get.rank_genes_groups_df(subset_for_de, group='KP', key='de_KP_vs_Flu')
de_results = de_results[['names', 'logfoldchanges', 'pvals_adj']].copy()
de_results = de_results.rename(columns={
    'names': 'gene',
    'logfoldchanges': 'log2FC',
    'pvals_adj': 'padj'
})

# Prepare plotting df
plot_df = de_results.copy()
plot_df['minus_log10_padj'] = -np.log10(plot_df['padj'].clip(lower=1e-300))
plot_df['significant'] = (plot_df['padj'] < pval_threshold) & (abs(plot_df['log2FC']) > log2fc_threshold)

# Top significant genes (strictly label_top_n per side)
sig = plot_df[plot_df['significant']]
kp_higher  = sig[sig['log2FC'] > 0].sort_values('log2FC', ascending=False).head(label_top_n)
flu_higher = sig[sig['log2FC'] < 0].sort_values('log2FC', ascending=True).head(label_top_n)
top_genes  = pd.concat([kp_higher, flu_higher])

# Extra genes (only if not already in top_genes)
extra_df = plot_df[plot_df['gene'].isin(extra_label_genes)]
extra_to_label = extra_df[~extra_df['gene'].isin(top_genes['gene'])]

# Genes that will actually get labeled
genes_to_label = pd.concat([top_genes, extra_to_label]).drop_duplicates('gene')

# === Plot ===
fig, ax = plt.subplots(figsize=(5, 5))

# Non-significant
ax.scatter(plot_df[~plot_df['significant']]['log2FC'],
           plot_df[~plot_df['significant']]['minus_log10_padj'],
           color='grey', s=1, alpha=0.6, rasterized=True)

# Significant points
ax.scatter(plot_df[(plot_df['significant']) & (plot_df['log2FC'] > 0)]['log2FC'],
           plot_df[(plot_df['significant']) & (plot_df['log2FC'] > 0)]['minus_log10_padj'],
           color=color_kp_higher, s=6, alpha=1, label='Higher in KP', rasterized=True)

ax.scatter(plot_df[(plot_df['significant']) & (plot_df['log2FC'] < 0)]['log2FC'],
           plot_df[(plot_df['significant']) & (plot_df['log2FC'] < 0)]['minus_log10_padj'],
           color=color_flu_higher, s=6, alpha=1, label='Higher in Flu', rasterized=True)

# === Formatting ===
ax.set_xlabel('Log2 Fold Change (OT-I KP vs OT-I Flu)')
ax.set_ylabel('-Log10 (Adjusted P-value)')
ax.set_title('OT-I CD8.Q: lung_KP vs lung_flu', pad=15)
ax.set_xlim(-10, 10)
ax.axvline(log2fc_threshold, color='black', linestyle='--', lw=0.5, alpha=0.7)
ax.axvline(-log2fc_threshold, color='black', linestyle='--', lw=0.5, alpha=0.7)
ax.axhline(-np.log10(pval_threshold), color='black', linestyle='--', lw=0.5, alpha=0.7)
ax.grid(False)
ax.legend(loc='upper left', fontsize=7, frameon=False)

plt.tight_layout()
plt.savefig(save_path, dpi=400, bbox_inches='tight')
plt.close()

print(f"Volcano plot saved: {save_path}")


# Extended Data Fig. 7

In [ ]:
# Extended Data Fig. 7c - RNA/ADT correlation analysis
# Keep only cells where cite_seq is True
keep = mdata['RNA'].obs['cite_seq'] == True

mdata = mdata[keep].copy()   # recommended to add .copy()

print(f"Remaining cells after filtering: {mdata.n_obs} (was 206160)")

rna = mdata['RNA']
adt = mdata['ADT']

print(f"RNA: {rna.n_obs} cells | ADT: {adt.n_obs} cells")

In [ ]:
# Extended Data Fig. 7c - RNA/ADT correlation analysis
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from adjustText import adjust_text
from matplotlib.lines import Line2D

marker_pairs = {
    'Sell': 'CD62L', 'Cd44': 'CD44', 'Il7r': 'IL7RA.CD127', 'Klrg1': 'KLRG1',
    'Itgax': 'ITAX.CD11C', 'Cd69': 'CD69', 'Itgae': 'CD103', 'Itga1': 'CD49A',
    'Entpd1': 'CD39', 'Slamf6': 'LY108', 'Pdcd1': 'PDCD1.PD1.CD279', 'Tnfrsf18': 'GITR.CD357',
}
group_key = 'cluster_annotation'

# ====================== MOVE LOG_NORM LAYERS TO .X ======================
rna.X = rna.layers['log_norm']
adt.X = adt.layers['log_norm']

# ====================== CLUSTER-LEVEL ANALYSIS ======================
results = []
plot_data = []
plots_dir = Path("cluster_level_correlations")
plots_dir.mkdir(exist_ok=True)

for rna_gene, adt_prot in marker_pairs.items():
    if rna_gene not in rna.var_names or adt_prot not in adt.var_names:
        continue

    rna_expr = rna[:, rna_gene].X
    if hasattr(rna_expr, 'toarray'):
        rna_expr = rna_expr.toarray().flatten()
    else:
        rna_expr = np.asarray(rna_expr).flatten()

    adt_expr = adt[:, adt_prot].X
    if hasattr(adt_expr, 'toarray'):
        adt_expr = adt_expr.toarray().flatten()
    else:
        adt_expr = np.asarray(adt_expr).flatten()

    cluster_rna = pd.DataFrame({
        'expr': rna_expr,
        'cluster': rna.obs[group_key].values
    }).groupby('cluster')['expr'].mean()

    cluster_adt = pd.DataFrame({
        'expr': adt_expr,
        'cluster': adt.obs[group_key].values
    }).groupby('cluster')['expr'].mean()

    df_cluster = pd.DataFrame({
        'rna_mean': cluster_rna,
        'adt_mean': cluster_adt
    }).dropna()

    if len(df_cluster) < 3:
        continue

    corr, pval = spearmanr(df_cluster['rna_mean'], df_cluster['adt_mean'])
    df_cluster['residual'] = df_cluster['rna_mean'] - df_cluster['adt_mean']
    df_cluster['abs_residual'] = df_cluster['residual'].abs()

    n_top = min(5, len(df_cluster))
    top_divergent = df_cluster.nlargest(n_top, 'abs_residual')

    results.append({
        'rna': rna_gene,
        'protein': adt_prot,
        'spearman': corr,
        'pval': pval,
        'n_clusters': len(df_cluster),
        'max_abs_residual': df_cluster['abs_residual'].max(),
        'mean_abs_residual': df_cluster['abs_residual'].mean(),
        'top_divergent_clusters': '; '.join(top_divergent.index.tolist()),
        'top_divergent_details': ' | '.join(
            [f"{clus}: RNA={row['rna_mean']:.3f}, ADT={row['adt_mean']:.3f}, res={row['residual']:.3f}"
             for clus, row in top_divergent.iterrows()]
        )
    })

    # Min-max scaling for plotting
    df_plot = df_cluster.copy()
    df_plot['rna_mean'] = (df_plot['rna_mean'] - df_plot['rna_mean'].min()) / (df_plot['rna_mean'].max() - df_plot['rna_mean'].min())
    df_plot['adt_mean'] = (df_plot['adt_mean'] - df_plot['adt_mean'].min()) / (df_plot['adt_mean'].max() - df_plot['adt_mean'].min())

    plot_data.append({
        'df_plot': df_plot,
        'df_cluster': df_cluster,
        'rna_gene': rna_gene,
        'adt_prot': adt_prot,
        'corr': corr,
        'pval': pval,
        'top_divergent': top_divergent,
        'notable_threshold': df_cluster['abs_residual'].quantile(0.8)
    })

    print(f"{rna_gene} vs {adt_prot} | Spearman r={corr:.3f} | Max |res|={df_cluster['abs_residual'].max():.3f}")

# ====================== COMBINED DETAILS CSV ======================
if plot_data:
    combined_details = []
    for pdata in plot_data:
        df = pdata['df_cluster'].copy().reset_index().rename(columns={'index': 'cluster'})
        df['rna'] = pdata['rna_gene']
        df['protein'] = pdata['adt_prot']
        cols = ['rna', 'protein', 'cluster', 'rna_mean', 'adt_mean', 'residual', 'abs_residual']
        combined_details.append(df[cols])
    all_details = pd.concat(combined_details, ignore_index=True)
    all_details_path = plots_dir / "Extended_Data_Fig_7c_all_cluster_details.csv"
    all_details.to_csv(all_details_path, index=False)
    print(f"\nSaved combined details CSV: {all_details_path}")

# ====================== COMBINED GRID PDF ======================
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42   # Cleaner PDF output

n_plots = len(plot_data)
if n_plots > 0:
    ncols = 4
    nrows = int(np.ceil(n_plots / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.51 * ncols, 5 * nrows), squeeze=False)
    axes = axes.flatten()
    fig.patch.set_facecolor('white')

    for i, pdata in enumerate(plot_data):
        ax = axes[i]
        ax.set_facecolor('white')

        # Grid FIRST + behind points
        ax.grid(True, alpha=0.25, color='gray', linestyle='-', linewidth=0.5)
        ax.set_axisbelow(True)

        df_plot = pdata['df_plot']
        df_cluster = pdata['df_cluster']

        # Use raw scatter instead of seaborn for cleaner PDF
        colors = [custom_colors.get(name, 'C0') for name in df_plot.index]
        scatter = ax.scatter(
            df_plot['rna_mean'], df_plot['adt_mean'],
            s=75, c=colors, edgecolors='black', linewidths=0.25,
            clip_on=False   # prevents extra clipping masks
        )

        ax.plot([0, 1], [0, 1], 'k--', linewidth=0.5, alpha=1)

        # === Label top 3 highest + bottom 3 lowest ===
        if len(df_plot) >= 3:
            combined = df_plot['rna_mean'] + df_plot['adt_mean']
            top3 = combined.nlargest(3).index.tolist()
            bottom3 = combined.nsmallest(3).index.tolist()
            to_label = list(set(top3 + bottom3))

            texts = []
            for clus in to_label:
                row = df_plot.loc[clus]
                texts.append(ax.text(
                    row['rna_mean'] * 1.015, row['adt_mean'], str(clus),
                    fontsize=8, alpha=1
                ))
            if texts:
                adjust_text(
                    texts, ax=ax,
                    arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
                )

        title = (f"{pdata['rna_gene']} vs {pdata['adt_prot']}\n"
                 f"r = {pdata['corr']:.3f} (p={pdata['pval']:.1e})")
        ax.set_title(title, fontsize=12, pad=4)
        ax.set_xlabel(f"Mean {pdata['rna_gene']} (min-max scaled)", fontsize=10)
        ax.set_ylabel(f"Mean {pdata['adt_prot']} (min-max scaled)", fontsize=10)
        ax.tick_params(axis='both', labelsize=8)

    # Hide unused axes + legend (unchanged)
    for j in range(n_plots, len(axes)):
        axes[j].set_visible(False)

    all_clusters = set()
    for pdata in plot_data:
        all_clusters.update(pdata['df_plot'].index.tolist())
    cluster_order = [c for c in custom_colors if c in all_clusters]
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label=c,
               markerfacecolor=custom_colors[c], markersize=7,
               markeredgecolor='black', markeredgewidth=0.4)
        for c in cluster_order
    ]
    fig.legend(
        handles=legend_elements, title='Cluster',
        bbox_to_anchor=(0.9, 0.5), loc='center left',
        fontsize=8, title_fontsize=12, frameon=True
    )

    plt.tight_layout(rect=[0, 0, 0.88, 1])
    grid_pdf_path = plots_dir / "Extended_Data_Fig_7c.pdf"
    fig.savefig(grid_pdf_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved combined grid PDF: {grid_pdf_path}")

# ====================== SUMMARY ======================
df = pd.DataFrame(results)
df = df.sort_values('spearman', ascending=False)
df.to_csv("cluster_level_correlations/Extended_Data_Fig_7c_cluster_level_correlations.csv", index=False)

print("\n=== CLUSTER-LEVEL RESULTS (sorted by Spearman r) ===")
print(df[['rna', 'protein', 'spearman', 'pval', 'n_clusters',
          'max_abs_residual', 'mean_abs_residual']].round(4).to_string())

print("\nOutputs ready:")
print(" • cluster_level_correlations/Extended_Data_Fig_7c_cluster_level_correlations.csv")
print(" • cluster_level_correlations/Extended_Data_Fig_7c_all_cluster_details.csv")
print(" • cluster_level_correlations/Extended_Data_Fig_7c_all_cluster_level_correlations_grid.pdf")
